# Cardiac MRI CAD — persistent, resumable Kaggle pipeline

This notebook keeps the scientific workflow and separates execution into **three restart-safe stages**. Expensive caches use semantic, mount-independent fingerprints: a Kaggle remount or an identical CSV rewrite does not trigger quality analysis, matching, EfficientNet, or evaluation again.

| Kaggle accelerator | Stage | Persistent result |
|---|---|---|
| None / CPU | Initial audit | dataset manifest, image-quality audit, manual-target audit |
| GPU | Attention U-Net | one checkpoint per patient fold, one OOF prediction part per fold, automatic masks |
| None / CPU | Final analysis | Sick/Normal matching, CPU/float32 EfficientNet feature bank, evaluation results |

The shared workspace is `/kaggle/working/cad_attention_unet_workspace` unless `WORKSPACE_ROOT` is changed. After a restart, execute the definitions cell again and keep every `force_*` flag `False`.


## 1. Definitions

Run this cell after every Kaggle kernel restart. Importing the definitions does **not** train a model or start feature extraction. The module-level names `run_kaggle_*` are compatibility aliases only; their implementations live in `Pipeline`.


In [26]:
"""Resumable cardiac MRI CAD research pipeline for Kaggle CPU/GPU sessions.

The scientific workflow is unchanged, but every expensive stage now writes atomic,
fingerprint-validated artifacts under ``/kaggle/working/cad_attention_unet_workspace``
(or a user-selected workspace). A disconnected kernel can resume from completed
quality audits, per-fold Attention U-Net checkpoints, per-fold OOF predictions,
matching, the EfficientNet feature bank, or the final patient-level evaluation.

The workspace cleaner keeps only artifacts addressed by this pipeline and can also
remove unrelated non-hidden files from ``/kaggle/working``. Cleanup is explicit and
never runs automatically while importing the notebook. Cache identities are semantic:
volatile Kaggle input mtimes and harmless CSV rewrites do not invalidate expensive work.

KAGGLE EXECUTION
1. CPU initial   -> dataset manifest, quality audit, manual-target audit
2. GPU           -> five resumable Attention U-Nets + resumable OOF prediction
3. CPU final     -> matching, 11-mode EfficientNet bank, evaluation, HTML review

Scientific safeguards remain unchanged: patient-level folds, out-of-fold masks,
same-slice controls, matched cohorts, and patient-level evaluation. EfficientNet
remains CPU/float32 with ImageNet V1 weights and batch/forward size 4; Attention
training and inference use CUDA AMP when the GPU stage is selected.
"""

import base64
import contextlib
import csv
import gc
import hashlib
import importlib
from importlib.metadata import version as package_version
import json
import math
import os
import random
import re
import sys
import time
import uuid
from collections import OrderedDict, defaultdict, namedtuple
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, Javascript, clear_output, display
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from tqdm.auto import tqdm

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# Persistent workspace shared by CPU, GPU, review, and evaluation sessions.
Workspace = namedtuple(
    "Workspace",
    "root manual_masks manual_annotations predicted_masks prediction_audit "
    "mask_overlays review_history checkpoints outputs dataset_manifest quality_audit "
    "quality_summary manual_audit manual_summary training_summary prediction_summary "
    "prediction_parts_dir invalid_predictions segmentation_metrics segmentation_summary "
    "matching_manifest matching_summary feature_bank feature_metadata results_csv "
    "comparisons_csv predictions_dir evaluation_metadata stage_state",
)

# Manual target labels are semantic values, not experiment settings.
HEART_PRESENT = "HEART_PRESENT"
NO_HEART_VISIBLE = "NO_HEART_VISIBLE"
UNUSABLE = "UNUSABLE"
VALID_TARGET_TYPES = {HEART_PRESENT, NO_HEART_VISIBLE, UNUSABLE}
ANNOTATION_FIELDS = (
    "timestamp_utc", "image_token", "patient_id", "series_id",
    "target_type", "source", "sample_weight", "note",
)

# =============================================================================
# STAGE 1 — DATASET, PREPROCESSING, QUALITY, AND MANUAL TARGETS
# =============================================================================
class DataStage:
    """Own dataset discovery, preprocessing, atomic files, and manual targets.

    Expensive quality measurements are fingerprint-validated and persisted. The same
    workspace is reused by CPU, GPU, review, feature, and evaluation stages.
    """

    image_cache = OrderedDict()  # in-memory only; discarded when the run ends

    # Convert CSV/string values safely before numeric comparisons.
    @staticmethod
    def _as_float(value, default=0.0):
        try:
            result=float(value)
        except (TypeError, ValueError):
            return float(default)
        return result if np.isfinite(result) else float(default)
    # Convert CSV/string values safely to an integer.
    @staticmethod
    def _as_int(value, default=0):
        return int(round(DataStage._as_float(value, float(default))))
    # Find the CAD dataset in Kaggle, an environment override, or the local fallback.
    @staticmethod
    def _default_dataset_path():
        env_value=os.environ.get("CAD_DATASET_PATH", "").strip()
        candidates=[]
        if env_value:
            candidates.append(Path(env_value))
        candidates.extend(
            [
                Path("/kaggle/input/cad-cardiac-mri-dataset"),
                Path("/kaggle/input/datasets/danialsharifrazi/cad-cardiac-mri-dataset"),
                Path(r"C:\F\_Develop\AI\Datasets\CAD Cardiac MRI Dataset"),
            ]
        )

        def is_dataset_root(path):
            return path.is_dir() and (path / "Normal").is_dir() and (path / "Sick").is_dir()

        for candidate in candidates:
            if is_dataset_root(candidate):
                return candidate

        kaggle_input=Path("/kaggle/input")
        if kaggle_input.is_dir():
            for current_root, directory_names, _ in os.walk(kaggle_input):
                current=Path(current_root)
                try:
                    depth=len(current.relative_to(kaggle_input).parts)
                except ValueError:
                    depth=99
                if {"Normal", "Sick"}.issubset(set(directory_names)):
                    return current
                if depth >= 3:
                    directory_names[:]=[]

        return candidates[0] if candidates else Path.cwd() / "CAD Cardiac MRI Dataset"
    # Locate the existing manual-mask workspace without changing its folder layout.
    @staticmethod
    def _default_workspace_path():
        if Path("/kaggle/working").exists():
            return Path("/kaggle/working/cad_attention_unet_workspace")
        return Path.cwd() / "cad_attention_unet_workspace"
    # Seed Python, NumPy and PyTorch so patient folds and training are reproducible.
    @staticmethod
    def seed_everything(seed=None, include_cuda=False):
        seed=int(42 if seed is None else seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
        # Restore the full pipeline's explicit setting instead of inheriting a
        # deterministic-algorithms switch from an unrelated notebook cell.
        torch.use_deterministic_algorithms(False, warn_only=True)
        if include_cuda:
            torch.cuda.manual_seed_all(seed)
            # Deterministic algorithms were disabled in the researched runs; cuDNN
            # may therefore choose its fastest kernel for the current GPU shapes.
            torch.backends.cudnn.deterministic=False
            torch.backends.cudnn.benchmark=True
    # Initialize only the CPU/GPU resources needed by the current research stage.
    @staticmethod
    def start_device_stage(requested, stage_name):
        # A stage chooses CPU or CUDA only when it starts. Importing this file never
        # allocates GPU memory, which is important when moving between Kaggle sessions.
        requested=(requested or "cpu").strip().lower()
        if requested == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but is unavailable. Enable a Kaggle GPU.")
        if requested not in {"cpu", "cuda", "auto"}:
            raise ValueError("device must be 'cpu', 'cuda', or 'auto'.")
        if requested == "auto":
            requested="cuda" if torch.cuda.is_available() else "cpu"
        device=torch.device(requested)
        DataStage.seed_everything(include_cuda=device.type == "cuda")
        if device.type == "cuda":
            DataStage.release_device(device)
            try:
                torch.cuda.reset_peak_memory_stats(device)
            except Exception:
                pass
        print(f"[DEVICE] {stage_name}: {device.type}")
        return device
    # Report peak GPU memory when relevant, then release stage resources.
    @staticmethod
    def finish_device_stage(device, stage_name):
        # Print peak GPU memory when CUDA supplied this stage.
        if device.type == "cuda":
            try:
                peak_gb=torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                print(f"[DEVICE] {stage_name}: peak GPU={peak_gb:.2f} GB")
            except Exception:
                pass
        # Release Python and CUDA memory regardless of the stage result.
        DataStage.release_device(device)
        print(f"[DEVICE] {stage_name}: resources released")
    # Move a neural model to the selected device and efficient CUDA memory layout.
    @staticmethod
    def prepare_model(model, device):
        model=model.to(device)
        if device.type == "cuda":
            model=model.to(memory_format=torch.channels_last)
        return model
    # Move a tensor to the active stage device without changing its values.
    @staticmethod
    def move_tensor(tensor, device):
        tensor=tensor.to(device, non_blocking=device.type == "cuda")
        if device.type == "cuda" and tensor.ndim == 4:
            tensor=tensor.contiguous(memory_format=torch.channels_last)
        return tensor
    # Use mixed precision only on CUDA; CPU execution remains ordinary float32.
    @staticmethod
    def autocast(device):
        enabled=device.type == "cuda"
        if not enabled:
            return contextlib.nullcontext()
        try:
            return torch.autocast(device_type="cuda", dtype=torch.float16)
        except (AttributeError, TypeError):
            return torch.cuda.amp.autocast(enabled=True)
    # Create the AMP gradient scaler used by GPU Attention U-Net training.
    @staticmethod
    def grad_scaler(device):
        enabled=device.type == "cuda"
        try:
            return torch.amp.GradScaler("cuda", enabled=enabled)
        except (AttributeError, TypeError):
            return torch.cuda.amp.GradScaler(enabled=enabled)
    # Free Python/CUDA caches between folds and between Kaggle stages.
    @staticmethod
    def release_device(device):
        gc.collect()
        if device.type == "cuda" and torch.cuda.is_available():
            try:
                torch.cuda.synchronize(device)
            except Exception:
                pass
            torch.cuda.empty_cache()
    # Format stage timing for concise console output.
    @staticmethod
    def format_seconds(seconds):
        seconds=max(0, int(round(seconds)))
        hours, seconds=divmod(seconds, 3600)
        minutes, seconds=divmod(seconds, 60)
        if hours:
            return f"{hours}h {minutes:02d}m {seconds:02d}s"
        if minutes:
            return f"{minutes}m {seconds:02d}s"
        return f"{seconds}s"
    # Reuse the full-pipeline folder names needed by the HTML review workflow.
    @staticmethod
    def create_workspace(workspace_root=None, manual_root=None):
        root=Path(workspace_root or manual_root or DataStage._default_workspace_path())
        outputs=root / "simple_pipeline_outputs"
        workspace=Workspace(
            root=root,
            manual_masks=root / "manual_masks",
            manual_annotations=root / "manual_annotation_labels.csv",
            predicted_masks=root / "predicted_attention_masks",
            prediction_audit=root / "simple_attention_prediction_audit.csv",
            mask_overlays=root / "mask_overlays",
            review_history=root / "simple_review_history.csv",
            checkpoints=root / "checkpoints",
            outputs=outputs,
            dataset_manifest=root / "simple_dataset_manifest.csv",
            quality_audit=root / "simple_image_quality_audit.csv",
            quality_summary=outputs / "image_quality_summary.json",
            manual_audit=root / "simple_manual_mask_audit.csv",
            manual_summary=outputs / "manual_mask_summary.json",
            training_summary=root / "simple_training_summary.json",
            prediction_summary=root / "simple_prediction_summary.json",
            prediction_parts_dir=outputs / "attention_prediction_parts",
            invalid_predictions=root / "attention_invalid_after_retrain.csv",
            segmentation_metrics=outputs / "attention_oof_segmentation_metrics.csv",
            segmentation_summary=outputs / "attention_oof_segmentation_summary.json",
            matching_manifest=outputs / "cross_class_matching_manifest.csv",
            matching_summary=outputs / "cross_class_matching_summary.json",
            feature_bank=outputs / "patient_feature_bank.npz",
            feature_metadata=outputs / "patient_feature_bank.json",
            results_csv=outputs / "evaluation_summary.csv",
            comparisons_csv=outputs / "paired_auc_comparisons.csv",
            predictions_dir=outputs / "oof_predictions",
            evaluation_metadata=outputs / "evaluation_cache.json",
            stage_state=outputs / "pipeline_stage_state.json",
        )
        for directory in (
            workspace.root, workspace.manual_masks, workspace.predicted_masks,
            workspace.mask_overlays, workspace.checkpoints, workspace.outputs,
            workspace.prediction_parts_dir, workspace.predictions_dir,
        ):
            directory.mkdir(parents=True, exist_ok=True)
        return workspace

    @staticmethod
    def workspace_contract(workspace):
        """Return the only files/directories this pipeline is allowed to persist.

        The contract deliberately includes restart caches, manual-review state, and
        final scientific outputs. Legacy files, temporary files, ad-hoc exports, and
        artifacts from older pipeline layouts are not part of the contract.
        """
        workspace=workspace if hasattr(workspace, "root") else DataStage.create_workspace(workspace)
        root_files={
            Path(workspace.manual_annotations),
            Path(workspace.prediction_audit),
            Path(workspace.review_history),
            Path(workspace.dataset_manifest),
            Path(workspace.quality_audit),
            Path(workspace.manual_audit),
            Path(workspace.training_summary),
            Path(workspace.prediction_summary),
            Path(workspace.invalid_predictions),
        }
        output_files={
            Path(workspace.quality_summary),
            Path(workspace.manual_summary),
            Path(workspace.segmentation_metrics),
            Path(workspace.segmentation_summary),
            Path(workspace.matching_manifest),
            Path(workspace.matching_summary),
            Path(workspace.feature_bank),
            Path(workspace.feature_metadata),
            Path(workspace.results_csv),
            Path(workspace.comparisons_csv),
            Path(workspace.evaluation_metadata),
            Path(workspace.stage_state),
        }
        directories={
            Path(workspace.manual_masks),
            Path(workspace.predicted_masks),
            Path(workspace.mask_overlays),
            Path(workspace.checkpoints),
            Path(workspace.outputs),
            Path(workspace.prediction_parts_dir),
            Path(workspace.predictions_dir),
        }
        return {
            "root_files": root_files,
            "output_files": output_files,
            "directories": directories,
            "checkpoint_names": {f"attention_unet_fold_{fold}.pt" for fold in range(5)},
            "prediction_part_names": {
                f"fold_{fold}.{suffix}"
                for fold in range(5)
                for suffix in ("csv", "json")
            },
        }

    @staticmethod
    def clean_kaggle_working(
        workspace_root=None,
        dry_run=True,
        remove_other_working_items=True,
        keep_backup_zips=False,
        working_root=None,
    ):
        """Keep only the persistent contract used by this pipeline.

        Safety rules:
        - never touches ``/kaggle/input``;
        - never deletes the workspace itself, manual masks, automatic masks, or
          compatible restart/evaluation artifacts;
        - hidden top-level Kaggle entries are left alone;
        - ``dry_run=True`` only prints the cleanup plan.

        ``working_root`` exists mainly for local tests; on Kaggle it resolves to
        ``/kaggle/working``.
        """
        import shutil

        workspace=DataStage.create_workspace(workspace_root)
        workspace_root_path=Path(workspace.root).resolve()
        if working_root is None:
            working_root_path=(
                Path("/kaggle/working").resolve()
                if Path("/kaggle/working").exists()
                else workspace_root_path.parent.resolve()
            )
        else:
            working_root_path=Path(working_root).resolve()

        try:
            workspace_root_path.relative_to(working_root_path)
        except ValueError as error:
            raise RuntimeError(
                f"Refusing cleanup: workspace {workspace_root_path} is not inside "
                f"working root {working_root_path}."
            ) from error
        if working_root_path == Path("/"):
            raise RuntimeError("Refusing cleanup with filesystem root as working_root.")

        contract=DataStage.workspace_contract(workspace)
        allowed_root_entries={path.resolve() for path in contract["root_files"]}
        allowed_root_entries.update({path.resolve() for path in contract["directories"]})
        allowed_output_entries={path.resolve() for path in contract["output_files"]}
        allowed_output_entries.update({
            Path(workspace.prediction_parts_dir).resolve(),
            Path(workspace.predictions_dir).resolve(),
        })

        removals=[]
        def queue(path, reason):
            path=Path(path)
            if path.exists() or path.is_symlink():
                removals.append((path, reason))

        # Remove old/foreign entries inside the current workspace root.
        if workspace_root_path.is_dir():
            for child in workspace_root_path.iterdir():
                if child.resolve() not in allowed_root_entries:
                    queue(child, "not in current workspace contract")

        outputs=Path(workspace.outputs)
        if outputs.is_dir():
            for child in outputs.iterdir():
                if child.resolve() not in allowed_output_entries:
                    queue(child, "not in current output contract")

        checkpoints=Path(workspace.checkpoints)
        if checkpoints.is_dir():
            for child in checkpoints.iterdir():
                if child.name not in contract["checkpoint_names"]:
                    queue(child, "obsolete/non-fold checkpoint")

        parts=Path(workspace.prediction_parts_dir)
        if parts.is_dir():
            for child in parts.iterdir():
                if child.name not in contract["prediction_part_names"]:
                    queue(child, "obsolete/incomplete prediction part")

        predictions=Path(workspace.predictions_dir)
        if predictions.is_dir():
            allowed_prediction_names={f"{mode}.csv" for mode in FeatureStage.MODES}
            for child in predictions.iterdir():
                if child.name not in allowed_prediction_names:
                    queue(child, "not a current evaluation prediction table")

        # Mask/review folders contain only their native PNG artifacts.
        for directory, label in (
            (Path(workspace.manual_masks), "manual mask"),
            (Path(workspace.predicted_masks), "automatic mask"),
            (Path(workspace.mask_overlays), "review overlay"),
        ):
            if directory.is_dir():
                for child in directory.iterdir():
                    if not child.is_file() or child.suffix.lower() != ".png":
                        queue(child, f"not a {label} PNG")

        # Remove unrelated visible top-level entries from /kaggle/working. Hidden
        # Kaggle/runtime entries are deliberately left untouched.
        if remove_other_working_items and working_root_path.is_dir():
            for child in working_root_path.iterdir():
                if child.resolve() == workspace_root_path:
                    continue
                if child.name.startswith(".") or child.name.startswith("__"):
                    continue
                if keep_backup_zips and child.is_file() and child.suffix.lower() == ".zip":
                    continue
                queue(child, "outside current pipeline workspace")

        # De-duplicate nested removals so a parent directory is removed once.
        unique=[]
        for path, reason in sorted(removals, key=lambda item: len(item[0].parts)):
            resolved=path.resolve() if path.exists() else path.absolute()
            if any(
                resolved == parent or parent in resolved.parents
                for parent, _ in unique
            ):
                continue
            unique.append((resolved, reason))

        def size_bytes(path):
            try:
                if path.is_file() or path.is_symlink():
                    return path.stat().st_size
                return sum(
                    item.stat().st_size
                    for item in path.rglob("*")
                    if item.is_file()
                )
            except OSError:
                return 0

        total_bytes=sum(size_bytes(path) for path, _ in unique)
        action="WOULD REMOVE" if dry_run else "REMOVE"
        print(f"[CLEANUP] workspace={workspace_root_path}")
        print(f"[CLEANUP] {action}: {len(unique)} entries, {total_bytes / 1024**2:.1f} MiB")
        for path, reason in unique:
            print(f"  - {path}  [{reason}]")

        removed=[]
        if not dry_run:
            for path, reason in unique:
                try:
                    if path.is_dir() and not path.is_symlink():
                        shutil.rmtree(path)
                    else:
                        path.unlink(missing_ok=True)
                    removed.append(str(path))
                except FileNotFoundError:
                    pass
            # Recreate exactly the directories expected by the active pipeline.
            DataStage.create_workspace(workspace_root_path)
            print(f"[CLEANUP] Removed {len(removed)} entries.")
        else:
            print("[CLEANUP] Dry run only. Re-run with dry_run=False to apply.")

        return {
            "workspace": str(workspace_root_path),
            "working_root": str(working_root_path),
            "dry_run": bool(dry_run),
            "planned_entries": len(unique),
            "planned_bytes": int(total_bytes),
            "removed": removed,
            "kept_contract": {
                "manual_masks": str(workspace.manual_masks),
                "predicted_masks": str(workspace.predicted_masks),
                "checkpoints": str(workspace.checkpoints),
                "prediction_parts": str(workspace.prediction_parts_dir),
                "feature_bank": str(workspace.feature_bank),
                "evaluation_results": str(workspace.results_csv),
            },
        }

    # Atomically save manual masks, automatic masks, and review overlays.
    @staticmethod
    def write_png(path, image):
        """Atomically write a PNG and preserve mtime when pixels are unchanged."""
        path=Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        array=np.asarray(image)
        ok, encoded=cv2.imencode(
            ".png", array, [cv2.IMWRITE_PNG_COMPRESSION, 9]
        )
        if not ok:
            raise RuntimeError(f"OpenCV could not encode PNG: {path}")
        encoded_bytes=encoded.tobytes()
        if path.is_file():
            try:
                if path.stat().st_size == len(encoded_bytes) and path.read_bytes() == encoded_bytes:
                    return False
            except OSError:
                pass
        temporary=path.with_name(
            f".{path.name}.{os.getpid()}.{time.time_ns()}.tmp"
        )
        try:
            temporary.write_bytes(encoded_bytes)
            os.replace(temporary, path)
            return True
        finally:
            temporary.unlink(missing_ok=True)

    # Atomically persist the small CSV tables required to reopen the editor.
    @staticmethod
    def write_csv(path, rows, fields):
        """Atomically write a CSV, leaving an identical persistent file untouched."""
        path=Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary=path.with_name(
            f".{path.name}.{os.getpid()}.{time.time_ns()}.tmp"
        )
        try:
            with temporary.open("w", newline="", encoding="utf-8") as handle:
                writer=csv.DictWriter(
                    handle, fieldnames=list(fields), extrasaction="ignore"
                )
                writer.writeheader()
                for row in rows:
                    writer.writerow({field: row.get(field, "") for field in fields})

            unchanged=False
            if path.is_file() and path.stat().st_size == temporary.stat().st_size:
                unchanged=True
                with path.open("rb") as existing, temporary.open("rb") as candidate:
                    while True:
                        old_chunk=existing.read(1024 * 1024)
                        new_chunk=candidate.read(1024 * 1024)
                        if old_chunk != new_chunk:
                            unchanged=False
                            break
                        if not old_chunk:
                            break
            if unchanged:
                temporary.unlink(missing_ok=True)
                return False
            os.replace(temporary, path)
            return True
        finally:
            temporary.unlink(missing_ok=True)

    # Read one persisted review table as ordinary dictionaries.
    @staticmethod
    def read_csv(path):
        path=Path(path)
        if not path.is_file():
            return []
        with path.open(newline="", encoding="utf-8-sig") as handle:
            return list(csv.DictReader(handle))

    @staticmethod
    def write_json(path, payload):
        """Atomically write canonical JSON and preserve mtime when content is unchanged."""
        path=Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        encoded=json.dumps(
            payload, indent=2, sort_keys=True, default=str
        ).encode("utf-8")
        if path.is_file():
            try:
                if path.read_bytes() == encoded:
                    return False
            except OSError:
                pass
        temporary=path.with_name(f".{path.name}.{os.getpid()}.{time.time_ns()}.tmp")
        try:
            temporary.write_bytes(encoded)
            os.replace(temporary, path)
            return True
        finally:
            temporary.unlink(missing_ok=True)

    @staticmethod
    def read_json(path, default=None):
        path=Path(path)
        if not path.is_file():
            return default
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return default

    @staticmethod
    def sha256_file(path):
        digest=hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    @staticmethod
    def write_torch(path, payload):
        path=Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary=path.with_name(f".{path.name}.{os.getpid()}.{time.time_ns()}.tmp")
        try:
            torch.save(payload, temporary)
            os.replace(temporary, path)
        finally:
            temporary.unlink(missing_ok=True)

    @staticmethod
    def write_npz(path, arrays):
        path=Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary=path.with_name(f".{path.name}.{os.getpid()}.{time.time_ns()}.tmp.npz")
        try:
            np.savez_compressed(temporary, **arrays)
            os.replace(temporary, path)
        finally:
            temporary.unlink(missing_ok=True)

    @staticmethod
    def csv_fields(rows, preferred=()):
        fields=[]
        seen=set()
        for field in preferred:
            if field not in seen:
                fields.append(field)
                seen.add(field)
        for row in rows:
            for field in row:
                if field not in seen and field != "attention_mask_bits":
                    fields.append(field)
                    seen.add(field)
        return fields
    @staticmethod
    def stable_path_key(path):
        """Return a mount-independent path beginning at Directory_* when possible."""
        path=Path(str(path))
        parts=path.parts
        patient_index=next(
            (index for index, part in enumerate(parts) if str(part).startswith("Directory_")),
            None,
        )
        if patient_index is not None:
            return "/".join(map(str, parts[patient_index:]))
        return path.name

    @staticmethod
    def stable_sequence_group(row):
        image_parent=Path(str(row.get("image_path", ""))).parent
        return f"{row.get('series_id', '')}::{DataStage.stable_path_key(image_parent)}"

    @staticmethod
    def is_kaggle_input_path(path):
        text=str(Path(str(path))).replace("\\", "/")
        return text == "/kaggle/input" or text.startswith("/kaggle/input/")

    @staticmethod
    def canonical_float(value, default="nan"):
        try:
            number=float(value)
        except (TypeError, ValueError):
            return str(default)
        if not np.isfinite(number):
            return str(default)
        return format(number, ".12g")

    @staticmethod
    def file_identity(path):
        """Stable source-image identity; Kaggle mount mtimes are intentionally ignored."""
        path=Path(str(path))
        if not path.is_file():
            return (DataStage.stable_path_key(path), -1, "missing")
        stat=path.stat()
        mutable_marker=(
            "readonly_kaggle_input"
            if DataStage.is_kaggle_input_path(path)
            else str(int(stat.st_mtime_ns))
        )
        return (
            DataStage.stable_path_key(path),
            int(stat.st_size),
            mutable_marker,
        )

    @staticmethod
    def dataset_fingerprint(rows):
        """Hash dataset semantics without using volatile /kaggle/input mtimes."""
        digest=hashlib.sha256(b"cad-dataset-stable-v2")
        for row in sorted(rows, key=lambda item: str(item.get("image_token", ""))):
            identity=DataStage.file_identity(row.get("image_path", ""))
            values=(
                str(row.get("image_token", "")),
                DataStage._as_int(row.get("label"), -1),
                str(row.get("patient_id", "")),
                str(row.get("series_id", "")),
                DataStage._as_int(row.get("segmentation_fold"), -1),
                DataStage.stable_sequence_group(row),
                DataStage._as_int(row.get("sequence_index"), 0),
                DataStage._as_int(row.get("sequence_length"), 1),
                DataStage.stable_path_key(row.get("previous_image_path") or row.get("image_path", "")),
                DataStage.stable_path_key(row.get("next_image_path") or row.get("image_path", "")),
                identity,
            )
            digest.update(json.dumps(values, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))
        return digest.hexdigest()

    @staticmethod
    def manual_state_fingerprint(rows, workspace):
        """Hash every training-relevant manual decision and mask by content."""
        by_token={str(row.get("image_token", "")): row for row in rows}
        annotations=DataStage.load_annotations(workspace)
        manual_files={path.stem: path for path in Path(workspace.manual_masks).glob("*.png")}
        digest=hashlib.sha256(b"manual-target-state-v2")
        for token in sorted(set(annotations) | set(manual_files)):
            row=by_token.get(token, {})
            annotation=annotations.get(token, {})
            mask_path=manual_files.get(token)
            mask_hash=(DataStage.sha256_file(mask_path) if mask_path and mask_path.is_file() else "")
            values=(
                token,
                str(row.get("patient_id", "")),
                str(row.get("series_id", "")),
                DataStage._as_int(row.get("segmentation_fold"), -1),
                DataStage._as_int(row.get("quality_valid"), 0),
                str(row.get("quality_reason", "")),
                str(annotation.get("target_type", "")).strip().upper(),
                str(annotation.get("source", "")).strip(),
                DataStage.canonical_float(annotation.get("sample_weight", ""), default=""),
                mask_hash,
            )
            digest.update(json.dumps(values, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))
        return digest.hexdigest()

    @staticmethod
    def accepted_from_manual_audit(rows, audit):
        """Reconstruct accepted training rows from a compatible persisted audit."""
        by_token={str(row.get("image_token", "")): dict(row) for row in rows}
        accepted=[]
        for record in audit:
            if str(record.get("status", "")).upper() != "ACCEPTED":
                continue
            token=str(record.get("image_token", ""))
            base=by_token.get(token)
            if base is None:
                continue
            base.update({
                "manual_mask_path": str(record.get("manual_mask_path", base.get("manual_mask_path", ""))),
                "manual_mask_area_ratio": DataStage._as_float(record.get("area_ratio"), 0.0),
                "quality_valid": 1,
                "segmentation_target_type": str(record.get("target_type", "")),
                "heart_present": DataStage._as_int(record.get("heart_present"), 1),
                "sample_weight": DataStage._as_float(record.get("sample_weight"), 1.0),
                "annotation_source": str(record.get("annotation_source", "")),
            })
            accepted.append(base)
        return accepted

    @staticmethod
    def newest_mtime_ns(paths):
        newest=0
        for path in paths:
            path=Path(path)
            try:
                if path.is_file():
                    newest=max(newest, int(path.stat().st_mtime_ns))
            except OSError:
                continue
        return newest
    # Create a stable image identifier from its patient-relative path.
    @staticmethod
    def image_token(image_path, patient_id, series_id):
        # Hash the path starting at Directory_* so tokens remain stable if the dataset
        # is mounted under a different Kaggle/local root.
        parts=image_path.parts
        patient_index=next(
            (i for i, part in enumerate(parts) if str(part).startswith("Directory_")),
            None,
        )
        if patient_index is None:
            raise ValueError(f"Path does not contain Directory_*: {image_path}")
        relative="/".join(map(str, parts[patient_index:]))
        digest=hashlib.sha256(relative.encode("utf-8")).hexdigest()[:20]
        safe_series=str(series_id).replace("/", "__").replace("\\", "__").replace(" ", "_")
        return f"{patient_id}__{safe_series}__{digest}"
    # Fold assignment is derived only from patient identity and class label.
    # No image-level random split is permitted because thousands of correlated
    # slices from one patient would otherwise leak into both train and validation.
    @staticmethod
    def patient_folds(
        patients,
    ):
        if isinstance(patients, dict):
            groups=defaultdict(list)
            for patient_id, label in patients.items():
                groups[int(label)].append(str(patient_id))
        else:
            groups={0: list(map(str, patients))}

        result={}
        for label, patient_ids in sorted(groups.items()):
            ordered=sorted(
                set(patient_ids),
                key=lambda patient_id: hashlib.sha256(
                    f"{42}|segmentation-fold|{label}|{patient_id}".encode("utf-8")
                ).hexdigest(),
            )
            for index, patient_id in enumerate(ordered):
                result[patient_id]=index % int(5)
        return result
    # Scan the dataset and add patient folds plus previous/current/next image context.
    @staticmethod
    def discover_dataset(dataset_path, workspace):
        dataset_path=Path(dataset_path)
        started=time.perf_counter()
        raw_rows=[]
        patient_to_label={}

        for class_name, label in (("Normal", 0), ("Sick", 1)):
            class_path=dataset_path / class_name
            if not class_path.is_dir():
                raise FileNotFoundError(f"Missing directory: {class_path}")

            for patient_path in sorted(class_path.iterdir()):
                if not patient_path.is_dir() or not patient_path.name.lower().startswith("directory_"):
                    continue
                patient_id=patient_path.name
                old_label=patient_to_label.get(patient_id)
                if old_label is not None and old_label != label:
                    raise RuntimeError(f"{patient_id} appears in both classes.")
                patient_to_label[patient_id]=label

                for image_path in sorted(patient_path.iterdir()):
                    if image_path.is_file() and image_path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
                        raw_rows.append((image_path, label, patient_id, f"{patient_id}/__ROOT__"))

                for series_path in sorted(path for path in patient_path.iterdir() if path.is_dir()):
                    series_id=f"{patient_id}/{series_path.name}"
                    for image_path in sorted(series_path.rglob("*")):
                        if image_path.is_file() and image_path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
                            raw_rows.append((image_path, label, patient_id, series_id))

        if not raw_rows:
            raise RuntimeError(f"No images were found in {dataset_path}")

        folds = DataStage.patient_folds(patient_to_label)
        rows = [
            {
                "manifest_index": index,
                "image_path": str(path),
                "label": int(label),
                "patient_id": patient_id,
                "series_id": series_id,
                "image_token": DataStage.image_token(path, patient_id, series_id),
                "segmentation_fold": int(folds[patient_id]),
            }
            for index, (path, label, patient_id, series_id) in enumerate(raw_rows)
        ]
        if len({row["image_token"] for row in rows}) != len(rows):
            raise RuntimeError("Duplicate image tokens were generated.")
        for row in rows:
            token=row["image_token"]
            row["manual_mask_path"]=str(workspace.manual_masks / f"{token}.png")
            row["predicted_attention_mask_path"]=str(
                workspace.predicted_masks / f"{token}.png"
            )

        # Neighbours stay inside the same sequence directory and patient.
        by_sequence_group=defaultdict(list)
        for index, row in enumerate(rows):

            parent=str(Path(row["image_path"]).parent)
            sequence_group=f"{row['series_id']}::{parent}"
            row["sequence_group_id"]=sequence_group
            by_sequence_group[sequence_group].append(index)
        for indices in by_sequence_group.values():
            ordered=sorted(
                indices,
                key=lambda index: DataStage.natural_path_key(rows[index]["image_path"]),
            )
            for position, row_index in enumerate(ordered):
                previous_index=ordered[max(0, position - 1)]
                next_index=ordered[min(len(ordered) - 1, position + 1)]
                rows[row_index]["sequence_index"]=int(position)
                rows[row_index]["sequence_length"]=int(len(ordered))
                rows[row_index]["previous_image_path"]=rows[previous_index]["image_path"]
                rows[row_index]["next_image_path"]=rows[next_index]["image_path"]

        print("[DATA] Patients:", len(patient_to_label))
        print("[DATA] Normal:", sum(label == 0 for label in patient_to_label.values()))
        print("[DATA] Sick:", sum(label == 1 for label in patient_to_label.values()))
        print("[DATA] Images:", len(rows))
        print("[DATA] Series proxies:", len({row["series_id"] for row in rows}))
        print("[DATA] Scan time:", DataStage.format_seconds(time.perf_counter() - started))
        return rows
    # Sort slice filenames numerically (img2 before img10) when building sequence context.
    @staticmethod
    def natural_path_key(path):
        return tuple(
            int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", str(path))
        )
    # Read gray.
    @staticmethod
    def read_gray(path):
        image=cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"OpenCV cannot read image: {path}")
        return image
    # Detect near-black, low-variance border lines that are safe to treat as scanner padding.
    @staticmethod
    def is_dark_uniform_line(line):
        values=np.asarray(line, dtype=np.float32).reshape(-1)
        return bool(
            values.size
            and float(values.mean()) <= 12.0
            and float(values.std()) <= 4.0
            and float(np.mean(values <= 20))
            >= 0.98
        )
    # Find conservative crop bounds without using CAD labels or model outputs.
    @staticmethod
    def detect_padding_bounds(image):
        if image.ndim != 2:
            raise ValueError("The image must be a 2D grayscale array.")
        height, width=image.shape
        # Padding removal is deliberately conservative: at least 60% of each
        # original dimension must remain, and at most 20% is removed from one side.
        min_height=max(8, int(np.ceil(height * 0.60)))
        min_width=max(8, int(np.ceil(width * 0.60)))
        max_vertical=int(np.floor(height * 0.20))
        max_horizontal=int(np.floor(width * 0.20))

        top=0
        while top < max_vertical and height - top - 1 >= min_height and DataStage.is_dark_uniform_line(image[top, :]):
            top +=1
        bottom_crop=0
        while bottom_crop < max_vertical and height - top - bottom_crop - 1 >= min_height and DataStage.is_dark_uniform_line(image[height - 1 - bottom_crop, :]):
            bottom_crop +=1
        left=0
        while left < max_horizontal and width - left - 1 >= min_width and DataStage.is_dark_uniform_line(image[:, left]):
            left +=1
        right_crop=0
        while right_crop < max_horizontal and width - left - right_crop - 1 >= min_width and DataStage.is_dark_uniform_line(image[:, width - 1 - right_crop]):
            right_crop +=1

        if top < 2:
            top=0
        if bottom_crop < 2:
            bottom_crop=0
        if left < 2:
            left=0
        if right_crop < 2:
            right_crop=0

        bottom=height - bottom_crop
        right=width - right_crop
        if bottom - top < min_height or right - left < min_width:
            return 0, height, 0, width
        return int(top), int(bottom), int(left), int(right)
    # Scale intensities with 1st/99th percentiles so isolated extremes do not dominate.
    @staticmethod
    def robust_scale(image):
        image=np.asarray(image, dtype=np.float32)
        lower, upper=np.percentile(
            image,
            [1.0, 99.0],
        )
        lower, upper=float(lower), float(upper)
        if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
            lower, upper=float(image.min()), float(image.max())
        if upper <= lower:
            return np.zeros_like(image, dtype=np.float32), lower, upper
        scaled=np.clip(image, lower, upper)
        return ((scaled - lower) / (upper - lower)).astype(np.float32), lower, upper
    # Compute the resize and centering geometry used by both images and masks.
    @staticmethod
    def canvas_geometry(height, width):
        size=256
        long_side=240
        scale=float(long_side / max(height, width))
        resized_height=max(1, int(round(height * scale)))
        resized_width=max(1, int(round(width * scale)))
        if height >= width:
            resized_height=long_side
        else:
            resized_width=long_side
        resized_height=min(resized_height, size)
        resized_width=min(resized_width, size)
        top=(size - resized_height) // 2
        left=(size - resized_width) // 2
        return resized_height, resized_width, top, left, scale
    # Resize one image while preserving aspect ratio and center it on the fixed canvas.
    @staticmethod
    def to_canvas(image):
        image=np.asarray(image, dtype=np.float32)
        height, width=image.shape
        resized_height, resized_width, top, left, scale=DataStage.canvas_geometry(height, width)
        if (resized_height, resized_width) != (height, width):
            interpolation=cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC
            resized=cv2.resize(image, (resized_width, resized_height), interpolation=interpolation)
        else:
            resized=image
        canvas=np.zeros(
            (256, 256),
            dtype=np.float32,
        )
        canvas[top : top + resized_height, left : left + resized_width]=np.clip(resized, 0.0, 1.0)
        return canvas
    # Return only the robust/raw/content views actually consumed by the pipeline.
    @staticmethod
    def standardize_image(image):
        top, bottom, left, right=DataStage.detect_padding_bounds(image)
        cropped=image[top:bottom, left:right]
        if cropped.size == 0:
            raise RuntimeError("Automatic cropping produced an empty image.")
        robust, _, _=DataStage.robust_scale(cropped)
        cropped_float=cropped.astype(np.float32)
        raw=cropped_float / 255.0
        robust_canvas=DataStage.to_canvas(robust)
        raw_canvas=DataStage.to_canvas(raw)
        resized_height, resized_width, top_pad, left_pad, _=DataStage.canvas_geometry(*cropped.shape)
        content=np.zeros((256, 256), dtype=np.float32)
        content[top_pad:top_pad + resized_height, left_pad:left_pad + resized_width]=1.0
        return robust_canvas, raw_canvas, content
    # Return the canonical 256×256 robust-scaled image used by Attention U-Net.
    @staticmethod
    def standardized_uint8(path):
        image=DataStage.read_gray(path)
        robust, _, _=DataStage.standardize_image(image)
        return np.clip(np.round(robust * 255.0), 0, 255).astype(np.uint8)
    # Create the robust/raw/content views later combined with ROI or complement masks.
    @staticmethod
    def classifier_views(path):
        image=DataStage.read_gray(path)
        robust, raw, content=DataStage.standardize_image(image)
        size=224
        robust_224=cv2.resize(robust, (size, size), interpolation=cv2.INTER_AREA)
        raw_224=cv2.resize(raw, (size, size), interpolation=cv2.INTER_AREA)
        content_224=cv2.resize(content, (size, size), interpolation=cv2.INTER_NEAREST)
        return (
            robust_224.astype(np.float32),
            raw_224.astype(np.float32),
            (content_224 > 0.5).astype(np.float32),
        )
    # Build a compact appearance hash used for duplicate-aware review and matching.
    @staticmethod
    def perceptual_hash(image):
        resized=cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA)
        values=cv2.dct(resized.astype(np.float32))[:8, :8].reshape(-1)
        median=float(np.median(values[1:]))
        bits=values > median
        number=0
        for bit in bits:
            number=(number << 1) | int(bool(bit))
        return f"{number:016x}"
    # Reuse recently standardized slices to avoid repeating OpenCV preprocessing.
    @staticmethod
    def get_cached_image(path):
        if path in DataStage.image_cache:
            value=DataStage.image_cache.pop(path)
            DataStage.image_cache[path]=value
            return value.copy()
        value=DataStage.standardized_uint8(path)
        DataStage.image_cache[path]=value
        while len(DataStage.image_cache) > 2048:
            DataStage.image_cache.popitem(last=False)
        return value.copy()
    # Release standardized-image RAM between expensive stages.
    @staticmethod
    def clear_image_cache():
        DataStage.image_cache.clear()
    # Measure sharpness, noise, dynamic range, and perceptual hash for one slice.
    @staticmethod
    def image_quality_metrics(row):
        image=DataStage.read_gray(row["image_path"])
        robust, raw, content=DataStage.standardize_image(image)
        coordinates=np.argwhere(content > 0.5)
        if not len(coordinates):
            return {
                "perceptual_hash": "",
                "sharpness": np.nan,
                "noise_ratio": np.nan,
                "dynamic_range": np.nan,
            }
        top, left=coordinates.min(axis=0)
        bottom, right=coordinates.max(axis=0) + 1
        size=128
        robust_thumb=cv2.resize(
            robust[top:bottom, left:right], (size, size), interpolation=cv2.INTER_AREA
        )
        raw_thumb=cv2.resize(
            raw[top:bottom, left:right], (size, size), interpolation=cv2.INTER_AREA
        )
        robust_u8=np.clip(np.round(robust_thumb * 255.0), 0, 255).astype(np.uint8)
        raw_255=np.clip(raw_thumb * 255.0, 0.0, 255.0).astype(np.float32)
        sharpness=float(cv2.Laplacian(robust_u8, cv2.CV_32F).var())
        residual=raw_255 - cv2.GaussianBlur(raw_255, (3, 3), 0)
        residual_median=float(np.median(residual))
        noise_sigma=float(np.median(np.abs(residual - residual_median)) / 0.6744897501960817)
        lower, upper=np.percentile(raw_255, [5.0, 95.0])
        dynamic_range=float(max(upper - lower, 0.0))
        return {
            "perceptual_hash": DataStage.perceptual_hash(robust_u8),
            "sharpness": sharpness,
            "noise_ratio": float(noise_sigma / max(dynamic_range, 1.0)),
            "dynamic_range": dynamic_range,
        }
    # Recompute image quality in RAM; thresholds and measurements are unchanged.
    @staticmethod
    def _compute_quality_audit(rows):
        records = []
        started = time.perf_counter()
        for index, row in enumerate(rows, start=1):
            records.append({**row, **DataStage.image_quality_metrics(row)})
            if index == 1 or index % 5000 == 0 or index == len(rows):
                print(f"[QUALITY] {index}/{len(rows)} | "
                      f"{DataStage.format_seconds(time.perf_counter() - started)}")

        by_patient=defaultdict(list)
        for record in records:
            by_patient[str(record["patient_id"])].append(record)

        for patient_rows in by_patient.values():
            sharp=np.asarray([DataStage._as_float(row.get("sharpness"), np.nan) for row in patient_rows])
            noise=np.asarray([DataStage._as_float(row.get("noise_ratio"), np.nan) for row in patient_rows])
            finite_sharp=sharp[np.isfinite(sharp)]
            finite_noise=noise[np.isfinite(noise)]
            blur_threshold=float(4.0)
            noise_threshold=float(0.30)
            if len(finite_sharp) >= 5:
                blur_threshold=max(
                    blur_threshold,
                    float(np.quantile(finite_sharp, 0.03)),
                )
            if len(finite_noise) >= 5:
                noise_threshold=min(
                    noise_threshold,
                    float(np.quantile(finite_noise, 0.97)),
                )
            for record in patient_rows:
                sharpness=float(record.get("sharpness", np.nan))
                noise_ratio=float(record.get("noise_ratio", np.nan))
                dynamic_range=float(record.get("dynamic_range", np.nan))
                reasons=[]
                if not np.isfinite(dynamic_range) or dynamic_range < 12.0:
                    reasons.append("low_dynamic_range")
                if not np.isfinite(sharpness) or sharpness < blur_threshold:
                    reasons.append("blurred")
                if not np.isfinite(noise_ratio) or noise_ratio > noise_threshold:
                    reasons.append("noisy")
                record["patient_blur_threshold"]=blur_threshold
                record["patient_noise_threshold"]=noise_threshold
                record["quality_valid"]=int(not reasons)
                record["quality_reason"]=";".join(reasons)

        valid = sum(int(row["quality_valid"]) for row in records)
        print(f"[QUALITY] valid={valid}/{len(records)}; measurements retained in RAM")
        return records

    @staticmethod
    def build_quality_audit(rows, workspace=None, refresh=False):
        """Reuse a complete quality audit across Kaggle remounts when images are unchanged."""
        workspace=workspace if hasattr(workspace, "quality_audit") else None
        current_dataset_fingerprint=DataStage.dataset_fingerprint(rows)
        required={
            "image_size_bytes", "perceptual_hash", "sharpness", "noise_ratio",
            "dynamic_range", "patient_blur_threshold", "patient_noise_threshold",
            "quality_valid", "quality_reason",
        }
        candidates=[]
        if workspace is not None:
            candidates.extend([
                workspace.quality_audit,
                workspace.root / "attention_image_quality_audit.csv",
            ])

        invalid_reasons=[]
        if not refresh:
            summary=(DataStage.read_json(workspace.quality_summary, {}) or {}) if workspace else {}
            for path in candidates:
                cached=DataStage.read_csv(path)
                by_token={
                    str(record.get("image_token", "")): record
                    for record in cached if str(record.get("image_token", ""))
                }
                expected_tokens={str(row["image_token"]) for row in rows}
                token_match=(
                    bool(cached)
                    and len(by_token) == len(rows)
                    and set(by_token) == expected_tokens
                    and required.issubset(set(cached[0]))
                )
                if not token_match:
                    invalid_reasons.append(f"{Path(path).name}: token/schema mismatch")
                    continue

                # A v2 summary proves the full stable dataset identity. Older audits are
                # adopted by checking size and, only for mutable non-Kaggle sources, mtime.
                files_match=(
                    path == getattr(workspace, "quality_audit", None)
                    and summary.get("dataset_fingerprint") == current_dataset_fingerprint
                )
                if not files_match:
                    files_match=True
                    for base in rows:
                        image_path=Path(base["image_path"])
                        if not image_path.is_file():
                            files_match=False
                            invalid_reasons.append(f"{Path(path).name}: missing source image")
                            break
                        stat=image_path.stat()
                        old=by_token[str(base["image_token"])]
                        if DataStage._as_int(old.get("image_size_bytes"), -1) != int(stat.st_size):
                            files_match=False
                            invalid_reasons.append(f"{Path(path).name}: image size changed")
                            break
                        if not DataStage.is_kaggle_input_path(image_path):
                            try:
                                old_mtime=int(str(old.get("image_mtime_ns", "-1")).strip())
                            except (TypeError, ValueError):
                                old_mtime=-1
                            if old_mtime >= 0 and old_mtime != int(stat.st_mtime_ns):
                                files_match=False
                                invalid_reasons.append(f"{Path(path).name}: mutable image mtime changed")
                                break
                if not files_match:
                    continue

                merged=[]
                for base in rows:
                    old=by_token[str(base["image_token"])]
                    record=dict(base)
                    for field in required | {"image_mtime_ns"}:
                        record[field]=old.get(field, "")
                    image_path=Path(base["image_path"])
                    stat=image_path.stat()
                    record["image_size_bytes"]=int(stat.st_size)
                    record["image_mtime_ns"]=int(stat.st_mtime_ns)
                    merged.append(record)

                if workspace is not None:
                    if path != workspace.quality_audit:
                        DataStage.write_csv(
                            workspace.quality_audit,
                            merged,
                            DataStage.csv_fields(merged, rows[0].keys() if rows else ()),
                        )
                    DataStage.write_json(
                        workspace.quality_summary,
                        {
                            "schema": "simple-image-quality-cache-v2-stable-dataset",
                            "dataset_images": len(rows),
                            "dataset_fingerprint": current_dataset_fingerprint,
                            "valid_images": sum(DataStage._as_int(row.get("quality_valid"), 0) for row in merged),
                            "quality_audit": str(workspace.quality_audit),
                        },
                    )
                print(f"[QUALITY] Compatible persisted audit reused: {len(merged)} images.")
                return merged

        if invalid_reasons and not refresh:
            print("[QUALITY][CACHE] " + invalid_reasons[-1] + "; rebuilding audit.")
        records=DataStage._compute_quality_audit(rows)
        for record in records:
            image_path=Path(record["image_path"])
            stat=image_path.stat() if image_path.is_file() else None
            record["image_size_bytes"]=int(stat.st_size) if stat is not None else -1
            record["image_mtime_ns"]=int(stat.st_mtime_ns) if stat is not None else -1
        if workspace is not None:
            DataStage.write_csv(
                workspace.quality_audit,
                records,
                DataStage.csv_fields(records, rows[0].keys() if rows else ()),
            )
            DataStage.write_json(
                workspace.quality_summary,
                {
                    "schema": "simple-image-quality-cache-v2-stable-dataset",
                    "dataset_images": len(rows),
                    "dataset_fingerprint": current_dataset_fingerprint,
                    "valid_images": sum(DataStage._as_int(row.get("quality_valid"), 0) for row in records),
                    "quality_audit": str(workspace.quality_audit),
                },
            )
            print(f"[QUALITY] Audit persisted: {workspace.quality_audit}")
        return records
    # Read binary mask.
    @staticmethod
    def read_binary_mask(path, size=None):
        mask=cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Mask cannot be read: {path}")
        size=int(size or 256)
        if mask.shape != (size, size):
            mask=cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)
        return (mask > 127).astype(np.uint8)
    # Check whether a saved manual mask is readable and has a plausible non-trivial area.
    @staticmethod
    def manual_mask_qc(path):
        path=Path(path)
        if not path.is_file():
            return {"exists": 0, "usable": 0, "area_ratio": np.nan, "reason": "missing"}
        try:
            mask=DataStage.read_binary_mask(path)
        except Exception:
            return {"exists": 1, "usable": 0, "area_ratio": np.nan, "reason": "unreadable"}
        area=float(mask.mean())
        if area < 0.0005:
            reason="empty_or_nearly_empty"
        elif area > 0.98:
            reason="nearly_full"
        else:
            reason=""
        return {"exists": 1, "usable": int(not reason), "area_ratio": area, "reason": reason}
    # Read only the persistent human decisions. Empty/unavailable labels mean no decisions.
    @staticmethod
    def load_annotations(workspace):
        if not workspace.manual_annotations.is_file():
            return {}
        with workspace.manual_annotations.open(newline="", encoding="utf-8") as handle:
            return {
                str(row["image_token"]).strip(): dict(row)
                for row in csv.DictReader(handle)
                if str(row.get("image_token", "")).strip()
            }

    # Replace the label CSV only after the complete new table has been written.
    @staticmethod
    def save_annotations(workspace, annotations):
        path = workspace.manual_annotations
        pending = path.with_suffix(".csv.tmp")
        try:
            with pending.open("w", newline="", encoding="utf-8") as handle:
                writer = csv.DictWriter(handle, fieldnames=ANNOTATION_FIELDS, extrasaction="ignore")
                writer.writeheader()
                writer.writerows(annotations[token] for token in sorted(annotations))
            pending.replace(path)
        finally:
            pending.unlink(missing_ok=True)
    # Use lower weight for confirmed-auto masks while preserving explicit negatives.
    @staticmethod
    def default_target_weight(target_type, source):
        target_type=str(target_type).upper()
        source=str(source).lower()
        if target_type == NO_HEART_VISIBLE:
            return float(0.90)
        if "auto" in source:
            return float(0.72)
        return float(1.0)
    # Persist manual decisions only; no audit/history output file is needed.
    @staticmethod
    def set_annotation(
        workspace,
        row,
        target_type,
        source,
        sample_weight=None,
        note="",
    ):
        target_type=str(target_type).strip().upper()
        if target_type not in VALID_TARGET_TYPES:
            raise ValueError(f"Unknown annotation type: {target_type}")
        records=DataStage.load_annotations(workspace)
        weight=(
            DataStage.default_target_weight(target_type, source)
            if sample_weight is None
            else float(sample_weight)
        )
        records[str(row["image_token"])]={
            "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            "image_token": str(row["image_token"]),
            "patient_id": str(row.get("patient_id", "")),
            "series_id": str(row.get("series_id", "")),
            "target_type": target_type,
            "source": str(source),
            "sample_weight": float(weight),
            "note": str(note),
        }
        DataStage.save_annotations(workspace, records)
    # Persist manual decisions only; no audit/history output file is needed.
    @staticmethod
    def remove_annotation(workspace, image_token):
        records=DataStage.load_annotations(workspace)
        records.pop(str(image_token), None)
        DataStage.save_annotations(workspace, records)
    # Persist manual decisions only; no audit/history output file is needed.
    @staticmethod
    def register_existing_manual_masks(
        rows,
        workspace,
    ):
        by_token={
            str(row.get("image_token", "")).strip(): dict(row)
            for row in rows
            if str(row.get("image_token", "")).strip()
        }
        annotations=DataStage.load_annotations(workspace)
        manual_files=sorted(workspace.manual_masks.glob("*.png"))
        timestamp=pd.Timestamp.utcnow().isoformat()

        summary={
            "timestamp_utc": timestamp,
            "manual_png_files": len(manual_files),
            "registered_heart_present": 0,
            "already_explicitly_labeled": 0,
            "skipped_empty_or_invalid": 0,
            "skipped_token_absent_from_dataset": 0,
            "reason_counts": {},
            "annotations_path": str(workspace.manual_annotations),
        }
        reason_counts=defaultdict(int)

        for mask_path in manual_files:
            token=str(mask_path.stem)
            row=by_token.get(token)
            if row is None:
                summary["skipped_token_absent_from_dataset"] +=1
                reason_counts["token_absent_from_dataset"] +=1
                continue

            existing=annotations.get(token, {})
            existing_target=str(existing.get("target_type", "")).strip().upper()
            if existing_target in VALID_TARGET_TYPES:
                summary["already_explicitly_labeled"] +=1
                reason_counts[f"already_{existing_target.lower()}"] +=1
                continue

            mask_qc=DataStage.manual_mask_qc(mask_path)
            if not bool(mask_qc.get("usable", 0)):
                summary["skipped_empty_or_invalid"] +=1
                reason=str(mask_qc.get("reason", "") or "invalid_manual_mask")
                reason_counts[reason] +=1
                continue

            annotations[token]={
                "timestamp_utc": timestamp,
                "image_token": token,
                "patient_id": str(row.get("patient_id", "")),
                "series_id": str(row.get("series_id", "")),
                "target_type": HEART_PRESENT,
                "source": "existing_manual_on_review",
                "sample_weight": float(1.0),
                "note": (
                    "Existing non-empty manual PNG registered automatically "
                    "when review was opened; the mask file itself was not rewritten."
                ),
            }
            summary["registered_heart_present"] +=1
            reason_counts["registered_heart_present"] +=1

        if summary["registered_heart_present"] > 0 or not workspace.manual_annotations.is_file():
            DataStage.save_annotations(workspace, annotations)

        summary["reason_counts"]=dict(sorted(reason_counts.items()))
        print(
            "[REVIEW][EXISTING MASKS] "
            f"registered HEART_PRESENT={summary['registered_heart_present']}, "
            f"already labeled={summary['already_explicitly_labeled']}, "
            f"empty/invalid={summary['skipped_empty_or_invalid']}, "
            f"token missing={summary['skipped_token_absent_from_dataset']}"
        )
        return summary
    # Validate current manual targets and return the audit in memory, never as a CSV.
    @staticmethod
    def _compute_manual_masks(
        rows,
        workspace,
        minimum_masks=None,
    ):
        # The research pipeline was trained only after at least 800 positive manual masks
        # survived quality/mask QC. The caller may override this only for diagnostics.
        minimum_masks=800 if minimum_masks is None else int(minimum_masks)
        by_token={str(row["image_token"]): dict(row) for row in rows}
        manual_files={path.stem: path for path in workspace.manual_masks.glob("*.png")}
        annotations=DataStage.load_annotations(workspace)
        candidate_tokens=sorted(set(manual_files) | set(annotations))

        accepted=[]
        audit=[]
        patient_counts=defaultdict(int)
        positive_patient_counts=defaultdict(int)
        fold_positive_counts=defaultdict(int)
        fold_negative_counts=defaultdict(int)
        positive_count=0
        negative_count=0
        unusable_count=0

        for token in candidate_tokens:
            row=by_token.get(token)
            path=manual_files.get(token, workspace.manual_masks / f"{token}.png")
            annotation=annotations.get(token, {})
            if row is None:
                audit.append(
                    {
                        "image_token": token,
                        "manual_mask_path": str(path),
                        "status": "REJECTED",
                        "target_type": annotation.get("target_type", ""),
                        "reason": "token_absent_from_dataset",
                    }
                )
                continue

            mask_qc=DataStage.manual_mask_qc(path)
            target_type=str(annotation.get("target_type", "")).strip().upper()
            source=str(annotation.get("source", "")).strip()
            if not target_type:
                if mask_qc["usable"]:
                    target_type=HEART_PRESENT
                    source="legacy_nonempty_manual"
                else:
                    target_type="UNLABELED_EMPTY"
                    source="legacy_unlabeled"

            quality_row=row
            quality_valid=DataStage._as_int(quality_row.get("quality_valid"), 0) == 1
            reasons=[]
            heart_present=""

            if target_type == HEART_PRESENT:
                heart_present=1
                if not mask_qc["usable"]:
                    reasons.append(mask_qc["reason"] or "missing_positive_mask")
            elif target_type == NO_HEART_VISIBLE:
                heart_present=0
                if not path.is_file():
                    DataStage.write_png(
                        path,
                        np.zeros(
                            (256, 256),
                            dtype=np.uint8,
                        ),
                    )
                    mask_qc=DataStage.manual_mask_qc(path)
                if not np.isfinite(DataStage._as_float(mask_qc.get("area_ratio"), np.nan)):
                    reasons.append("unreadable_negative_mask")
                elif DataStage._as_float(mask_qc.get("area_ratio"), 1.0) >= 0.0005:
                    reasons.append("no_heart_annotation_has_nonempty_mask")
            elif target_type == UNUSABLE:
                unusable_count +=1
                reasons.append("explicitly_unusable")
            else:
                reasons.append("empty_mask_without_explicit_no_heart_label")

            if target_type != UNUSABLE and not quality_valid:
                reasons.append(str(quality_row.get("quality_reason", "quality_invalid")))

            if target_type == UNUSABLE:
                status="EXCLUDED"
            else:
                status="ACCEPTED" if not reasons else "REJECTED"

            sample_weight=DataStage._as_float(
                annotation.get("sample_weight"),
                DataStage.default_target_weight(target_type, source),
            )
            audit.append(
                {
                    "image_token": token,
                    "patient_id": row["patient_id"],
                    "series_id": row["series_id"],
                    "manual_mask_path": str(path),
                    "status": status,
                    "target_type": target_type,
                    "heart_present": heart_present,
                    "annotation_source": source,
                    "sample_weight": sample_weight,
                    "area_ratio": mask_qc.get("area_ratio", np.nan),
                    "quality_valid": quality_row.get("quality_valid", 0),
                    "quality_reason": quality_row.get("quality_reason", ""),
                    "reason": ";".join(filter(None, reasons)),
                }
            )

            if status != "ACCEPTED":
                continue
            row.update(
                {
                    "manual_mask_path": str(path),
                    "manual_mask_area_ratio": float(DataStage._as_float(mask_qc.get("area_ratio"), 0.0)),
                    "quality_valid": 1,
                    "perceptual_hash": quality_row.get("perceptual_hash", ""),
                    "segmentation_target_type": target_type,
                    "heart_present": int(heart_present),
                    "sample_weight": float(sample_weight),
                    "annotation_source": source,
                }
            )
            accepted.append(row)
            patient_counts[str(row["patient_id"])] +=1
            fold=int(row["segmentation_fold"])
            if int(heart_present) == 1:
                positive_count +=1
                positive_patient_counts[str(row["patient_id"])] +=1
                fold_positive_counts[fold] +=1
            else:
                negative_count +=1
                fold_negative_counts[fold] +=1

        rejected=sum(row.get("status") == "REJECTED" for row in audit)
        excluded=sum(row.get("status") == "EXCLUDED" for row in audit)
        unlabeled_empty=sum(
            "without_explicit_no_heart" in str(row.get("reason", "")) for row in audit
        )
        blur_or_noise=sum(
            row.get("status") == "REJECTED" and bool(row.get("quality_reason"))
            for row in audit
        )
        summary={
            "manual_png_files": len(manual_files),
            "explicit_annotations": len(annotations),
            "accepted_training_targets": len(accepted),
            "accepted_heart_present": positive_count,
            "accepted_no_heart_visible": negative_count,
            "explicit_unusable": unusable_count,
            "rejected_targets": rejected,
            "excluded_targets": excluded,
            "rejected_unlabeled_empty_masks": unlabeled_empty,
            "rejected_blur_or_noise": blur_or_noise,
            "patients_with_training_targets": len(patient_counts),
            "patients_with_positive_masks": len(positive_patient_counts),
            "targets_per_patient": dict(sorted(patient_counts.items())),
            "positive_masks_per_fold": {
                str(fold): int(fold_positive_counts.get(fold, 0))
                for fold in range(5)
            },
            "negative_masks_per_fold": {
                str(fold): int(fold_negative_counts.get(fold, 0))
                for fold in range(5)
            },
            "annotations": str(workspace.manual_annotations),
        }
        print(
            "[MANUAL MASKS] "
            f"heart_present={positive_count}, no_heart={negative_count}, "
            f"rejected={rejected}, unusable={unusable_count}, "
            f"patients={len(patient_counts)}"
        )

        reason_counts = defaultdict(int)
        for record in audit:
            if record.get("status") != "ACCEPTED":
                reason_counts[record.get("reason", "unknown")] += 1
        summary["rejection_reasons"] = dict(sorted(reason_counts.items()))
        if reason_counts:
            print("[MANUAL MASKS] rejected/excluded reasons:", summary["rejection_reasons"])

        if positive_count < minimum_masks:
            raise RuntimeError(
                f"Only {positive_count} valid HEART_PRESENT masks remain, below the minimum of "
                f"{minimum_masks}. Inspect the manual-target summary above and the HTML editor."
            )
        missing_folds=[
            fold
            for fold in range(5)
            if fold_positive_counts.get(fold, 0) == 0
        ]
        if missing_folds:
            raise RuntimeError(
                f"No valid HEART_PRESENT masks exist in folds {missing_folds}."
            )
        if len(positive_patient_counts) < 5 + 1:
            raise RuntimeError("Too few patients have positive masks for cross-fitting.")
        return accepted, audit, summary
    @staticmethod
    def audit_manual_masks(rows, workspace, minimum_masks=None):
        """Reuse the persisted manual audit when labels, masks, and quality are unchanged."""
        minimum_masks=800 if minimum_masks is None else int(minimum_masks)
        fingerprint=DataStage.manual_state_fingerprint(rows, workspace)
        summary=DataStage.read_json(workspace.manual_summary, {}) or {}
        cached=DataStage.read_csv(workspace.manual_audit)
        required={
            "image_token", "manual_mask_path", "status", "target_type",
            "heart_present", "annotation_source", "sample_weight", "area_ratio",
        }
        if (
            cached
            and required.issubset(set(cached[0]))
            and summary.get("input_fingerprint") == fingerprint
        ):
            accepted=DataStage.accepted_from_manual_audit(rows, cached)
            positive=[row for row in accepted if DataStage._as_int(row.get("heart_present"), 1) == 1]
            if len(positive) < minimum_masks:
                raise RuntimeError(
                    f"Only {len(positive)} valid HEART_PRESENT masks remain, below the minimum of {minimum_masks}."
                )
            missing_folds=[
                fold for fold in range(5)
                if not any(DataStage._as_int(row.get("segmentation_fold"), -1) == fold for row in positive)
            ]
            if missing_folds:
                raise RuntimeError(f"No valid HEART_PRESENT masks exist in folds {missing_folds}.")
            if len({str(row.get("patient_id", "")) for row in positive}) < 6:
                raise RuntimeError("Too few patients have positive masks for cross-fitting.")
            print(
                "[MANUAL MASKS] Compatible persisted audit reused: "
                f"accepted={len(accepted)}, positive={len(positive)}."
            )
            return accepted, cached, summary

        accepted, audit, summary=DataStage._compute_manual_masks(
            rows, workspace, minimum_masks=minimum_masks
        )
        # NO_HEART targets may create a zero mask during computation, so fingerprint after it.
        summary={
            **summary,
            "schema": "simple-manual-mask-cache-v2",
            "input_fingerprint": DataStage.manual_state_fingerprint(rows, workspace),
        }
        return accepted, audit, summary

# =============================================================================
# STAGE 2 — ATTENTION U-NET: MODEL + OOF SEGMENTATION
# =============================================================================
# These small classes are required by PyTorch. They are implementation details of
# AttentionStage, not separate pipeline-management layers.
class AttentionConvBlock(nn.Module):
    """Apply two normalized convolution layers inside the Attention U-Net."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        groups=next(group for group in (8, 4, 2, 1) if out_channels % group == 0)
        self.block=nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)
class AttentionGate(nn.Module):
    """Weight an encoder skip connection using decoder context."""

    def __init__(self, gating_channels, skip_channels, inter_channels):
        super().__init__()
        groups=next(group for group in (8, 4, 2, 1) if inter_channels % group == 0)
        self.gating=nn.Sequential(
            nn.Conv2d(gating_channels, inter_channels, 1, bias=False),
            nn.GroupNorm(groups, inter_channels),
        )
        self.skip=nn.Sequential(
            nn.Conv2d(skip_channels, inter_channels, 1, bias=False),
            nn.GroupNorm(groups, inter_channels),
        )
        self.weight=nn.Sequential(
            nn.SiLU(inplace=True),
            nn.Conv2d(inter_channels, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, gating, skip):
        gating=F.interpolate(gating, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return skip * self.weight(self.gating(gating) + self.skip(skip))
class AttentionUpBlock(nn.Module):
    """Upsample decoder features, gate the skip connection, and refine the result."""

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up=nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.gate=AttentionGate(out_channels, skip_channels, max(1, out_channels // 2))
        self.refine=AttentionConvBlock(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x=F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x=self.up(x)
        return self.refine(torch.cat([x, self.gate(x, skip)], dim=1))
class AttentionUNet(nn.Module):
    """2.5D Attention U-Net with segmentation and heart-presence heads.

    Neighboring frames provide local sequence context, while the auxiliary head
    suppresses false positive masks on localizers or frames without visible heart.
    """

    def __init__(
        self,
        base_channels=None,
        input_channels=None,):
        super().__init__()
        base=int(base_channels or 24)  # 24 channels in the first U-Net block
        input_channels=int(input_channels or 3)  # previous/current/next = 2.5D input
        self.input_channels=input_channels
        self.encoder1=AttentionConvBlock(input_channels, base)
        self.encoder2=AttentionConvBlock(base, base * 2)
        self.encoder3=AttentionConvBlock(base * 2, base * 4)
        self.encoder4=AttentionConvBlock(base * 4, base * 8)
        self.bottleneck=AttentionConvBlock(base * 8, base * 16)
        self.pool=nn.MaxPool2d(2)
        self.decoder4=AttentionUpBlock(base * 16, base * 8, base * 8)
        self.decoder3=AttentionUpBlock(base * 8, base * 4, base * 4)
        self.decoder2=AttentionUpBlock(base * 4, base * 2, base * 2)
        self.decoder1=AttentionUpBlock(base * 2, base, base)
        self.output=nn.Conv2d(base, 1, 1)
        self.presence_head=nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base * 16, base * 4),
            nn.SiLU(inplace=True),
            nn.Dropout(p=0.20),
            nn.Linear(base * 4, 1),
        )

    def forward(self, x):
        e1=self.encoder1(x)
        e2=self.encoder2(self.pool(e1))
        e3=self.encoder3(self.pool(e2))
        e4=self.encoder4(self.pool(e3))
        bottleneck=self.bottleneck(self.pool(e4))
        presence_logits=self.presence_head(bottleneck).squeeze(1)
        d4=self.decoder4(bottleneck, e4)
        d3=self.decoder3(d4, e3)
        d2=self.decoder2(d3, e2)
        d1=self.decoder1(d2, e1)
        return self.output(d1), presence_logits
class SegmentationDataset(Dataset):
    """Load manual segmentation targets and optional training augmentation."""

    def __init__(self, rows, augment, seed):
        self.rows=list(rows)
        self.augment=bool(augment)
        self.seed=int(seed)
        self.epoch=0

    def set_epoch(self, epoch):
        self.epoch=int(epoch)

    def __len__(self):
        return len(self.rows)

    @staticmethod
    def _image_stack(row):
        current=str(row["image_path"])
        # 2.5D input: previous, current, and next slices are always used.
        paths=(
            str(row.get("previous_image_path") or current),
            current,
            str(row.get("next_image_path") or current),
        )
        return np.stack(
            [DataStage.get_cached_image(path).astype(np.float32) / 255.0 for path in paths],
            axis=0,
        )

    def __getitem__(self, index):
        row=self.rows[index]
        image=self._image_stack(row)
        mask=DataStage.read_binary_mask(row["manual_mask_path"]).astype(np.float32)
        heart_present=float(DataStage._as_int(row.get("heart_present"), int(mask.any())))
        sample_weight=float(DataStage._as_float(row.get("sample_weight"), 1.0))

        if self.augment:
            rng=np.random.default_rng(self.seed + self.epoch * 1_000_003 + index)
            size=256
            angle=float(
                rng.uniform(
                    -7.0,
                    7.0,
                )
            )
            scale=float(
                rng.uniform(
                    0.92,
                    1.08,
                )
            )
            shift=0.06 * size
            tx, ty=map(float, rng.uniform(-shift, shift, size=2))
            matrix=cv2.getRotationMatrix2D((size / 2, size / 2), angle, scale)
            matrix[:, 2] +=(tx, ty)
            image=np.stack(
                [
                    cv2.warpAffine(
                        channel,
                        matrix,
                        (size, size),
                        flags=cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_CONSTANT,
                        borderValue=0,
                    )
                    for channel in image
                ],
                axis=0,
            )
            mask=cv2.warpAffine(
                mask,
                matrix,
                (size, size),
                flags=cv2.INTER_NEAREST,
                borderMode=cv2.BORDER_CONSTANT,
                borderValue=0,
            )

            gamma=float(
                rng.uniform(
                    0.85,
                    1.15,
                )
            )
            contrast=float(rng.uniform(0.90, 1.10))
            brightness=float(rng.uniform(-0.04, 0.04))
            image=np.power(np.clip(image, 0.0, 1.0), gamma)
            image=np.clip(image * contrast + brightness, 0.0, 1.0)
            if rng.random() < 0.15:
                image=np.stack(
                    [cv2.GaussianBlur(channel, (3, 3), 0) for channel in image],
                    axis=0,
                )
            noise_std=float(rng.uniform(0.0, 0.025))
            if noise_std > 0:
                image=np.clip(
                    image + rng.normal(0.0, noise_std, size=image.shape).astype(np.float32),
                    0.0,
                    1.0,
                )

        return (
            torch.from_numpy(np.ascontiguousarray(image)).float(),
            torch.from_numpy(np.ascontiguousarray(mask > 0.5)).unsqueeze(0).float(),
            torch.tensor(heart_present, dtype=torch.float32),
            torch.tensor(sample_weight, dtype=torch.float32),
            str(row["patient_id"]),
        )
class SegmentationInferenceDataset(Dataset):
    """Load image stacks for deterministic segmentation inference."""

    def __init__(self, rows):
        self.rows=list(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        image=SegmentationDataset._image_stack(self.rows[index])
        return torch.from_numpy(np.ascontiguousarray(image)).float(), int(index)

class AttentionStage:
    """Train five patient-level folds from scratch and predict every image OOF."""

    PREDICTION_FIELDS=(
        "image_token", "image_path", "patient_id", "series_id",
        "segmentation_fold", "predicted_attention_mask_path",
        "attention_valid_initial", "attention_valid_final",
        "attention_invalid_reason_initial", "attention_invalid_reason_final",
        "attention_repair_method", "attention_threshold_used",
        "attention_area_ratio", "attention_peak_probability",
        "attention_boundary_touch_fraction", "attention_heart_present",
        "attention_presence_probability", "attention_presence_threshold",
        "attention_tta_disagreement", "attention_mean_entropy",
        "attention_uncertainty_score", "attention_sequence_inconsistency",
        "attention_centroid_x", "attention_centroid_y",
        "attention_prior_deviation", "checkpoint_fingerprint",
    )

    @staticmethod
    def checkpoint_path(workspace, fold):
        return Path(workspace.checkpoints) / f"attention_unet_fold_{int(fold)}.pt"

    @staticmethod
    def torch_load(path):
        try:
            return torch.load(str(path), map_location="cpu", weights_only=False)
        except TypeError:
            return torch.load(str(path), map_location="cpu")

    @staticmethod
    def manual_fingerprint(rows, fold):
        """Stable training fingerprint independent of volatile Kaggle input mtimes."""
        payload={
            "schema": "simple-attention-input-stable-v4",
            "model_schema": "simple-attention-2p5d-presence-v3",
            "fold": int(fold), "size": 256, "input_channels": 3,
            "use_2_5d": True, "base_channels": 24, "epochs": 36,
            "learning_rate": 8e-4, "weight_decay": 1e-4,
            "loss": {"focal": 0.40, "tversky": 0.60, "presence": 0.30,
                     "alpha_fp": 0.65, "beta_fn": 0.35},
            "seed": 42,
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        for row in sorted(rows, key=lambda item: str(item["image_token"])):
            mask_path=Path(row["manual_mask_path"])
            values=(
                str(row["image_token"]),
                str(row.get("segmentation_target_type", "")),
                str(row.get("annotation_source", "")),
                DataStage.canonical_float(row.get("sample_weight", 1.0)),
                DataStage._as_int(row.get("heart_present"), 1),
                DataStage.file_identity(row["image_path"]),
                DataStage.stable_path_key(row.get("previous_image_path") or row["image_path"]),
                DataStage.stable_path_key(row.get("next_image_path") or row["image_path"]),
                DataStage.sha256_file(mask_path),
            )
            digest.update(json.dumps(values, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))
        return digest.hexdigest()

    @staticmethod
    def legacy_manual_fingerprint(rows, fold):
        """Original v3 fingerprint retained only to reuse historical checkpoints."""
        payload={
            "schema": "simple-attention-2p5d-presence-v3", "fold": int(fold),
            "size": 256, "input_channels": 3, "use_2_5d": True,
            "base_channels": 24, "epochs": 36, "learning_rate": 8e-4,
            "weight_decay": 1e-4,
            "loss": {"focal": 0.40, "tversky": 0.60, "presence": 0.30,
                     "alpha_fp": 0.65, "beta_fn": 0.35}, "seed": 42,
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        for row in sorted(rows, key=lambda item: str(item["image_token"])):
            mask_path=Path(row["manual_mask_path"])
            image_path=Path(row["image_path"])
            image_stat=image_path.stat()
            digest.update(str(row["image_token"]).encode("utf-8"))
            digest.update(str(row.get("segmentation_target_type", "")).encode("utf-8"))
            digest.update(str(row.get("annotation_source", "")).encode("utf-8"))
            digest.update(str(row.get("sample_weight", 1.0)).encode("ascii"))
            digest.update(str(image_stat.st_size).encode("ascii"))
            digest.update(str(image_stat.st_mtime_ns).encode("ascii"))
            digest.update(DataStage.sha256_file(mask_path).encode("ascii"))
        return digest.hexdigest()

    @staticmethod
    def checkpoint_reuse_status(
        checkpoint, checkpoint_path, summary_entry, stable_fingerprint,
        legacy_fingerprint, train_rows, validation_patients, accepted_rows,
        fold, workspace,
    ):
        if checkpoint.get("schema") != "simple-attention-2p5d-presence-v3":
            return False, "checkpoint schema changed"
        if checkpoint.get("stable_fingerprint") == stable_fingerprint:
            return True, "stable fingerprint"
        if checkpoint.get("fingerprint") in {stable_fingerprint, legacy_fingerprint}:
            return True, "exact stored fingerprint"
        checkpoint_sha=DataStage.sha256_file(checkpoint_path)
        if (
            summary_entry.get("stable_fingerprint") == stable_fingerprint
            and summary_entry.get("checkpoint_sha256") == checkpoint_sha
        ):
            return True, "stable summary sidecar"

        # One-time safe adoption of the old mtime-sensitive checkpoint. The model,
        # split membership, counts, summary fingerprint, and manual-file freshness
        # must all agree; otherwise the fold is retrained.
        if not summary_entry or checkpoint.get("fingerprint") != summary_entry.get("fingerprint"):
            return False, "legacy checkpoint has no matching training summary"
        expected_train=sorted({str(row["patient_id"]) for row in train_rows})
        expected_validation=sorted(map(str, validation_patients))
        expected_excluded=sorted({
            str(row["patient_id"]) for row in accepted_rows
            if DataStage._as_int(row.get("segmentation_fold"), -1) == int(fold)
        })
        if checkpoint.get("train_patients", []) != expected_train:
            return False, "training-patient split changed"
        if checkpoint.get("validation_patients", []) != expected_validation:
            return False, "validation-patient split changed"
        if checkpoint.get("excluded_test_patients", []) != expected_excluded:
            return False, "held-out-patient split changed"
        train_positive=sum(DataStage._as_int(row.get("heart_present"), 1) == 1 for row in train_rows)
        train_negative=len(train_rows) - train_positive
        if DataStage._as_int(checkpoint.get("train_positive_targets"), -1) != train_positive:
            return False, "positive target count changed"
        if DataStage._as_int(checkpoint.get("train_negative_targets"), -1) != train_negative:
            return False, "negative target count changed"
        manual_paths=[row["manual_mask_path"] for row in train_rows]
        manual_paths.append(workspace.manual_annotations)
        if DataStage.newest_mtime_ns(manual_paths) > int(checkpoint_path.stat().st_mtime_ns):
            return False, "manual target changed after checkpoint"
        return True, "legacy checkpoint safely adopted"

    @staticmethod
    def load_checkpoint_map(workspace):
        result={}
        for fold in range(5):
            path=AttentionStage.checkpoint_path(workspace, fold)
            if not path.is_file():
                raise FileNotFoundError(
                    f"Missing checkpoint for fold {fold}: {path}. Run the Kaggle GPU stage."
                )
            result[fold]=path
        return result

    @staticmethod
    def prediction_fingerprint(rows, checkpoint_map):
        payload={
            "schema": "simple-predictions-stable-v6",
            "thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75),
            "presence_thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80),
            "area": [0.003,0.65], "repair_offsets": (-0.20,-0.15,-0.10,-0.05,0.05,0.10,0.15,0.20),
            "tta_contrast": 1.10, "presence_override_peak": 0.80, "prior_z": 5.5,
            "dataset_fingerprint": DataStage.dataset_fingerprint(rows),
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        for fold, checkpoint_path in sorted(checkpoint_map.items()):
            digest.update(str(fold).encode("ascii"))
            digest.update(DataStage.sha256_file(checkpoint_path).encode("ascii"))
        return digest.hexdigest()

    @staticmethod
    def legacy_prediction_fingerprint(rows, checkpoint_map):
        digest=hashlib.sha256()
        settings_payload={
            "schema": "simple-predictions-2p5d-presence-v5",
            "thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75),
            "presence_thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80),
            "area": [0.003,0.65], "repair_offsets": (-0.20,-0.15,-0.10,-0.05,0.05,0.10,0.15,0.20),
            "tta_contrast": 1.10, "presence_override_peak": 0.80, "prior_z": 5.5,
        }
        digest.update(json.dumps(settings_payload, sort_keys=True).encode("utf-8"))
        digest.update(str(len(rows)).encode("ascii"))
        for row in rows:
            image_path=Path(row["image_path"]); image_stat=image_path.stat()
            digest.update(str(row["image_token"]).encode("utf-8"))
            digest.update(str(row["segmentation_fold"]).encode("ascii"))
            digest.update(str(row.get("previous_image_path", "")).encode("utf-8"))
            digest.update(str(row.get("next_image_path", "")).encode("utf-8"))
            digest.update(str(image_stat.st_size).encode("ascii"))
            digest.update(str(image_stat.st_mtime_ns).encode("ascii"))
        for fold, checkpoint_path in sorted(checkpoint_map.items()):
            digest.update(str(fold).encode("ascii"))
            digest.update(DataStage.sha256_file(checkpoint_path).encode("ascii"))
        return digest.hexdigest()

    @staticmethod
    def prediction_part_paths(workspace, fold):
        return (
            Path(workspace.prediction_parts_dir) / f"fold_{int(fold)}.csv",
            Path(workspace.prediction_parts_dir) / f"fold_{int(fold)}.json",
        )

    @staticmethod
    def prediction_part_fingerprint(fold, fold_rows, checkpoint_path):
        payload={
            "schema": "simple-prediction-part-stable-v6", "fold": int(fold),
            "thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75),
            "presence_thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80),
            "repair_offsets": (-0.20,-0.15,-0.10,-0.05,0.05,0.10,0.15,0.20),
            "morphology": 5, "tta_contrast": 1.10, "presence_override_peak": 0.80,
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        digest.update(DataStage.sha256_file(checkpoint_path).encode("ascii"))
        for row in sorted(fold_rows, key=lambda item: str(item["image_token"])):
            values=(
                str(row["image_token"]), DataStage.file_identity(row["image_path"]),
                DataStage.stable_path_key(row.get("previous_image_path") or row["image_path"]),
                DataStage.stable_path_key(row.get("next_image_path") or row["image_path"]),
            )
            digest.update(json.dumps(values, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))
        return digest.hexdigest()

    @staticmethod
    def legacy_prediction_part_fingerprint(fold, fold_rows, checkpoint_path):
        digest=hashlib.sha256()
        payload={
            "schema": "simple-prediction-part-2p5d-presence-v5", "fold": int(fold),
            "thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75),
            "presence_thresholds": (0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80),
            "repair_offsets": (-0.20,-0.15,-0.10,-0.05,0.05,0.10,0.15,0.20),
            "morphology": 5, "tta_contrast": 1.10, "presence_override_peak": 0.80,
        }
        digest.update(json.dumps(payload, sort_keys=True).encode("utf-8"))
        digest.update(DataStage.sha256_file(checkpoint_path).encode("ascii"))
        for row in fold_rows:
            image_path=Path(row["image_path"]); image_stat=image_path.stat()
            digest.update(str(row["image_token"]).encode("utf-8"))
            digest.update(str(image_stat.st_size).encode("ascii"))
            digest.update(str(image_stat.st_mtime_ns).encode("ascii"))
            digest.update(str(row.get("previous_image_path", "")).encode("utf-8"))
            digest.update(str(row.get("next_image_path", "")).encode("utf-8"))
        return digest.hexdigest()

    @staticmethod
    def prediction_part_cache_status(fold, fold_rows, workspace, checkpoint_path):
        part_csv, part_json=AttentionStage.prediction_part_paths(workspace, fold)
        metadata=DataStage.read_json(part_json, {}) or {}
        cached=DataStage.read_csv(part_csv)
        cached_by_token={
            str(row.get("image_token", "")): row
            for row in cached if str(row.get("image_token", ""))
        }
        expected_tokens={str(row["image_token"]) for row in fold_rows}
        required={
            "attention_presence_probability", "attention_uncertainty_score",
            "attention_sequence_inconsistency", "attention_centroid_x",
            "attention_centroid_y", "attention_heart_present",
        }
        masks_complete=all(
            (Path(workspace.predicted_masks) / f"{token}.png").is_file()
            for token in expected_tokens
        )
        rows_match=True
        current_by_token={str(row["image_token"]): row for row in fold_rows}
        for token, saved in cached_by_token.items():
            current=current_by_token.get(token, {})
            if (
                str(saved.get("patient_id", "")) != str(current.get("patient_id", ""))
                or str(saved.get("series_id", "")) != str(current.get("series_id", ""))
                or DataStage._as_int(saved.get("segmentation_fold"), -1) != int(fold)
            ):
                rows_match=False
                break
        complete=(
            len(cached_by_token) == len(fold_rows)
            and set(cached_by_token) == expected_tokens
            and bool(cached) and required.issubset(set(cached[0]))
            and masks_complete and rows_match
        )
        stable=AttentionStage.prediction_part_fingerprint(fold, fold_rows, checkpoint_path)
        legacy=AttentionStage.legacy_prediction_part_fingerprint(fold, fold_rows, checkpoint_path)
        exact=(
            metadata.get("stable_fingerprint") == stable
            or metadata.get("fingerprint") in {stable, legacy}
        )
        adopted=False
        legacy_adoptable=(
            metadata.get("schema") == "simple-prediction-part-2p5d-presence-v5"
            and not metadata.get("stable_fingerprint")
        )
        if complete and not exact and legacy_adoptable and bool(metadata.get("completed", False)):
            checkpoint=AttentionStage.torch_load(checkpoint_path)
            acceptable={
                str(checkpoint.get("fingerprint", "")),
                str(checkpoint.get("stable_fingerprint", "")),
            } - {""}
            cached_checkpoint_fingerprints={
                str(row.get("checkpoint_fingerprint", ""))
                for row in cached if str(row.get("checkpoint_fingerprint", ""))
            }
            adopted=bool(acceptable) and cached_checkpoint_fingerprints.issubset(acceptable)
        reusable=bool(complete and (exact or adopted))
        return {
            "reusable": reusable, "adopted": adopted, "complete": complete,
            "stable_fingerprint": stable, "legacy_fingerprint": legacy,
            "metadata": metadata, "cached": cached,
            "cached_by_token": cached_by_token,
            "part_csv": part_csv, "part_json": part_json,
            "reason": "compatible" if reusable else "fingerprint or completeness mismatch",
        }

    # Combine focal, Tversky, and heart-presence losses using per-target review weights.
    @staticmethod
    def segmentation_loss(
        logits,
        targets,
        presence_logits,
        presence_targets,
        sample_weights,
    ):
        probability=torch.sigmoid(logits)
        bce=F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt=probability * targets + (1.0 - probability) * (1.0 - targets)
        focal=((1.0 - pt).pow(2.0) * bce).mean(
            dim=(1, 2, 3)
        )

        tp=(probability * targets).sum(dim=(1, 2, 3))
        fp=(probability * (1.0 - targets)).sum(dim=(1, 2, 3))
        fn=((1.0 - probability) * targets).sum(dim=(1, 2, 3))
        tversky=(tp + 1e-6) / (
            tp
            + 0.65 * fp  # penalize false-positive mask area
            + 0.35 * fn  # penalize false-negative mask area
            + 1e-6
        )
        segmentation_per_sample=(
            0.40 * focal  # focal-loss contribution
            + 0.60 * (1.0 - tversky)  # Tversky-loss contribution
        )

        sample_weights=sample_weights.float().clamp_min(1e-3)
        segmentation_loss=(
            segmentation_per_sample * sample_weights
        ).sum() / sample_weights.sum().clamp_min(1e-6)
        presence_loss=F.binary_cross_entropy_with_logits(
            presence_logits.float(), presence_targets.float(), reduction="none"
        )
        presence_loss=(presence_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-6)
        return segmentation_loss + 0.30 * presence_loss  # auxiliary heart-presence loss
    # Select validation patients.
    @staticmethod
    def select_validation_patients(patient_ids, target_fold):
        ordered=sorted(
            set(map(str, patient_ids)),
            key=lambda patient_id: hashlib.sha256(
                f"{42}|validation|{target_fold}|{patient_id}".encode(
                    "utf-8"
                )
            ).hexdigest(),
        )
        count=max(
            1,
            int(
                round(
                    len(ordered)
                    * 0.20
                )
            ),
        )
        count=min(count, max(1, len(ordered) - 1))
        return set(ordered[:count])
    # Keep positive and negative presence examples represented in fold validation.
    @staticmethod
    def ensure_presence_coverage(
        rows,
        validation_patients,
        target_fold,
    ):
        patients=sorted({str(row["patient_id"]) for row in rows})
        has_label={0: set(), 1: set()}
        for row in rows:
            has_label[DataStage._as_int(row.get("heart_present"), 1)].add(
                str(row["patient_id"])
            )
        validation=set(validation_patients)

        def ordered(values, salt):
            return sorted(
                values,
                key=lambda patient: hashlib.sha256(
                    f"{42}|{target_fold}|{salt}|{patient}".encode(
                        "utf-8"
                    )
                ).hexdigest(),
            )

        for label in (0, 1):
            labelled=has_label[label]
            if len(labelled) < 2:
                continue
            if not (validation & labelled):
                incoming=ordered(labelled - validation, f"incoming-{label}")[0]
                removable=[
                    patient
                    for patient in validation
                    if patient not in labelled
                    and all(
                        len((validation - {patient}) & has_label[other]) > 0
                        for other in (0, 1)
                        if len(has_label[other]) >= 2
                        and (validation & has_label[other])
                    )
                ]
                if removable:
                    validation.remove(ordered(set(removable), f"remove-{label}")[0])
                validation.add(incoming)
            if not (labelled - validation):

                outgoing=ordered(validation & labelled, f"outgoing-{label}")[-1]
                validation.remove(outgoing)
                replacement_candidates=set(patients) - validation - {outgoing}
                if replacement_candidates:
                    validation.add(
                        ordered(replacement_candidates, f"replacement-{label}")[0]
                    )
        if not validation or len(validation) >= len(patients):
            return set(validation_patients)
        return validation
    # Balance patients, series, duplicate clusters, and explicit no-heart targets in training.
    @staticmethod
    def training_sampler(
        rows,
        seed,
    ):
        rows=list(rows)
        series_by_patient=defaultdict(set)
        clusters_by_series=defaultdict(set)
        cluster_counts=defaultdict(int)
        for row in rows:
            patient=str(row["patient_id"])
            series=str(row.get("sequence_group_id") or row["series_id"])
            cluster=str(row.get("perceptual_hash") or row["image_token"])
            series_by_patient[patient].add(series)
            clusters_by_series[(patient, series)].add(cluster)
            cluster_counts[(patient, series, cluster)] +=1

        base_weights=[]
        labels=[]
        for row in rows:
            patient=str(row["patient_id"])
            series=str(row.get("sequence_group_id") or row["series_id"])
            cluster=str(row.get("perceptual_hash") or row["image_token"])
            weight=1.0
            weight /=max(1, len(series_by_patient[patient]))
            weight /=max(1, len(clusters_by_series[(patient, series)]))
            weight /=max(1, cluster_counts[(patient, series, cluster)])
            base_weights.append(weight)
            labels.append(DataStage._as_int(row.get("heart_present"), 1))

        base=np.asarray(base_weights, dtype=np.float64)
        labels_array=np.asarray(labels, dtype=np.int64)
        positive=labels_array == 1
        negative=labels_array == 0
        if positive.any() and negative.any():
            target_negative=float(0.30)
            target_positive=1.0 - target_negative
            base[positive] *=target_positive / max(base[positive].sum(), 1e-12)
            base[negative] *=target_negative / max(base[negative].sum(), 1e-12)
        else:
            base /=max(base.sum(), 1e-12)

        generator=torch.Generator().manual_seed(int(seed))
        return WeightedRandomSampler(
            weights=torch.as_tensor(base, dtype=torch.double),
            num_samples=len(rows),
            replacement=True,
            generator=generator,
        )
    # Summarize mask area, boundary contact, centroid, and aspect ratio for QC.
    @staticmethod
    def mask_features(mask):
        mask=(np.asarray(mask) > 0).astype(np.uint8)
        area, boundary=AttentionStage.mask_geometry(mask)
        if not mask.any():
            return {
                "area": area,
                "boundary": boundary,
                "centroid_x": np.nan,
                "centroid_y": np.nan,
                "aspect_ratio": np.nan,
            }
        ys, xs=np.nonzero(mask)
        width=float(xs.max() - xs.min() + 1)
        height=float(ys.max() - ys.min() + 1)
        return {
            "area": area,
            "boundary": boundary,
            "centroid_x": float(xs.mean() / max(1.0, mask.shape[1] - 1)),
            "centroid_y": float(ys.mean() / max(1.0, mask.shape[0] - 1)),
            "aspect_ratio": float(width / max(height, 1.0)),
        }
    # Build geometry prior.
    @staticmethod
    def build_geometry_prior(rows):
        features=[]
        for row in rows:
            if DataStage._as_int(row.get("heart_present"), 1) != 1:
                continue
            try:
                features.append(
                    AttentionStage.mask_features(
                        DataStage.read_binary_mask(row["manual_mask_path"])
                    )
                )
            except Exception:
                continue
        keys=("area", "centroid_x", "centroid_y", "aspect_ratio")
        if not features:
            return {"count": 0, "median": {}, "scale": {}}
        median={}
        scale={}
        for key in keys:
            values=np.asarray([item[key] for item in features], dtype=np.float64)
            values=values[np.isfinite(values)]
            if not len(values):
                continue
            center=float(np.median(values))
            mad=float(np.median(np.abs(values - center)))
            median[key]=center
            scale[key]=max(1e-3, 1.4826 * mad)
        return {"count": len(features), "median": median, "scale": scale}
    # Calculate prior deviation.
    @staticmethod
    def calculate_prior_deviation(mask, prior):
        if not prior or DataStage._as_int(prior.get("count"), 0) < 5 or not np.asarray(mask).any():
            return 0.0
        features=AttentionStage.mask_features(mask)
        deviations=[]
        for key, center in dict(prior.get("median", {})).items():
            value=features.get(key, np.nan)
            scale=DataStage._as_float(dict(prior.get("scale", {})).get(key), 0.0)
            if np.isfinite(value) and scale > 0:
                deviations.append(abs(float(value) - float(center)) / scale)
        return float(max(deviations)) if deviations else 0.0
    # Average recall across available heart-present / no-heart classes.
    @staticmethod
    def balanced_accuracy(labels, predictions):
        labels=np.asarray(labels, dtype=np.int64)
        predictions=np.asarray(predictions, dtype=np.int64)
        values=[]
        for label in (0, 1):
            selector=labels == label
            if selector.any():
                values.append(float(np.mean(predictions[selector] == label)))
        return float(np.mean(values)) if values else 0.0
    # Choose mask and presence thresholds only on non-test validation patients.
    @staticmethod
    def calibrate_threshold(
        model,
        loader,
        device,
        geometry_prior,
    ):
        probabilities=[]
        targets=[]
        presence_probabilities=[]
        presence_targets=[]
        patients=[]
        model.eval()
        with torch.inference_mode():
            for images, masks, target_presence, _weights, patient_ids in loader:
                images=DataStage.move_tensor(images, device)
                with DataStage.autocast(device):
                    logits, presence_logits=model(images)
                    probability=torch.sigmoid(logits)
                    presence_probability=torch.sigmoid(presence_logits)
                probabilities.extend(probability.float().cpu().numpy()[:, 0])
                targets.extend(masks.numpy()[:, 0])
                presence_probabilities.extend(
                    presence_probability.float().cpu().numpy().tolist()
                )
                presence_targets.extend(target_presence.numpy().astype(int).tolist())
                patients.extend(map(str, patient_ids))

        labels_array=np.asarray(presence_targets, dtype=np.int64)
        presence_array=np.asarray(presence_probabilities, dtype=np.float64)
        best_presence={
            "threshold": 0.50,
            "balanced_accuracy": 0.0,
        }
        if len(np.unique(labels_array)) >= 2:
            for threshold in (0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80):
                prediction=(presence_array >= threshold).astype(np.int64)
                balanced=AttentionStage.balanced_accuracy(
                    labels_array, prediction
                )
                candidate={
                    "threshold": float(threshold),
                    "balanced_accuracy": balanced,
                }
                if (
                    candidate["balanced_accuracy"],
                    -abs(candidate["threshold"] - 0.5),
                ) > (
                    best_presence["balanced_accuracy"],
                    -abs(best_presence["threshold"] - 0.5),):
                    best_presence=candidate
        elif len(labels_array):
            best_presence["balanced_accuracy"]=1.0

        best=None
        for threshold in (0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75):
            by_patient=defaultdict(list)
            invalid=0
            for probability, target, target_presence, patient_id in zip(
                probabilities, targets, presence_targets, patients):
                prediction=AttentionStage.candidate_mask(
                    probability, threshold
                )
                if int(target_presence) == 1:
                    intersection=float(
                        np.logical_and(prediction, target > 0.5).sum()
                    )
                    denominator=float(
                        prediction.sum() + (target > 0.5).sum()
                    )
                    quality=(2.0 * intersection + 1e-6) / (
                        denominator + 1e-6
                    )
                    valid, _=AttentionStage.validate_mask(
                        prediction, probability, geometry_prior
                    )
                    invalid +=int(not valid)
                else:

                    quality=1.0 - min(1.0, float(prediction.mean()) / 0.10)
                by_patient[patient_id].append(float(quality))
            patient_scores=[
                float(np.mean(values)) for values in by_patient.values()
            ]
            mean_score=float(np.mean(patient_scores)) if patient_scores else 0.0
            invalid_rate=float(invalid / max(1, sum(presence_targets)))
            score=mean_score - 0.20 * invalid_rate
            candidate={
                "threshold": float(threshold),
                "patient_balanced_segmentation_score": mean_score,
                "invalid_rate_positive": invalid_rate,
                "score": score,
            }
            if best is None or (
                candidate["score"], -abs(threshold - 0.5)
            ) > (best["score"], -abs(best["threshold"] - 0.5)):
                best=candidate

        result=best or {
            "threshold": 0.50,
            "patient_balanced_segmentation_score": 0.0,
            "invalid_rate_positive": 1.0,
            "score": -1.0,
        }
        result["presence_threshold"]=float(best_presence["threshold"])
        result["presence_balanced_accuracy"]=float(
            best_presence["balanced_accuracy"]
        )
        return result
    # Exclude each test fold from fitting and calibration; retain best CPU weights in RAM.
    @staticmethod
    def train_attention_crossfit(accepted_rows, device, workspace=None, force=False):
        DataStage.seed_everything(include_cuda=device.type == "cuda")
        fold_states={}
        fold_summaries=[]
        all_patients=sorted({str(row["patient_id"]) for row in accepted_rows})
        previous_summary=(DataStage.read_json(workspace.training_summary, {}) or {}) if workspace is not None else {}
        previous_by_fold={
            DataStage._as_int(item.get("fold"), -1): item
            for item in previous_summary.get("folds", [])
            if isinstance(item, dict)
        }

        for fold in range(5):  # five patient-level folds
            fold_started=time.perf_counter()
            non_test_rows=[
                row
                for row in accepted_rows
                if int(row["segmentation_fold"]) != fold
            ]
            validation_patients=AttentionStage.select_validation_patients(
                (row["patient_id"] for row in non_test_rows), fold
            )
            validation_patients=AttentionStage.ensure_presence_coverage(
                non_test_rows, validation_patients, fold
            )
            train_rows=[
                row
                for row in non_test_rows
                if row["patient_id"] not in validation_patients
            ]
            validation_rows=[
                row
                for row in non_test_rows
                if row["patient_id"] in validation_patients
            ]
            stable_fingerprint=AttentionStage.manual_fingerprint(non_test_rows, fold)
            legacy_fingerprint=AttentionStage.legacy_manual_fingerprint(non_test_rows, fold)
            checkpoint_path=(
                AttentionStage.checkpoint_path(workspace, fold)
                if workspace is not None else None
            )
            if checkpoint_path is not None and checkpoint_path.is_file() and not force:
                try:
                    checkpoint=AttentionStage.torch_load(checkpoint_path)
                except Exception as error:
                    print(
                        f"[ATTENTION] Fold {fold}: checkpoint unreadable; retraining "
                        f"({type(error).__name__}: {error})."
                    )
                else:
                    reusable, cache_reason=AttentionStage.checkpoint_reuse_status(
                        checkpoint, checkpoint_path, previous_by_fold.get(fold, {}),
                        stable_fingerprint, legacy_fingerprint, train_rows,
                        validation_patients, accepted_rows, fold, workspace,
                    )
                    if reusable:
                        fold_states[fold]=checkpoint_path
                        fold_summaries.append({
                            "fold": fold, "status": "reused",
                            "checkpoint": str(checkpoint_path),
                            "fingerprint": checkpoint.get("fingerprint", ""),
                            "stable_fingerprint": stable_fingerprint,
                            "checkpoint_sha256": DataStage.sha256_file(checkpoint_path),
                            "cache_reason": cache_reason,
                            "calibration": checkpoint.get("calibration", {}),
                        })
                        print(f"[ATTENTION] Fold {fold}: compatible checkpoint reused ({cache_reason}).")
                        continue
                    print(f"[ATTENTION] Fold {fold}: {cache_reason}; retraining.")
            if not train_rows or not validation_rows:
                raise RuntimeError(f"Fold {fold}: train or validation split is empty.")

            train_positive=sum(
                DataStage._as_int(row.get("heart_present"), 1) == 1 for row in train_rows
            )
            train_negative=len(train_rows) - train_positive
            validation_positive=sum(
                DataStage._as_int(row.get("heart_present"), 1) == 1
                for row in validation_rows
            )
            validation_negative=len(validation_rows) - validation_positive
            print(
                f"[ATTENTION] Fold {fold}: train={len(train_rows)} "
                f"(+{train_positive}/-{train_negative}), validation={len(validation_rows)} "
                f"(+{validation_positive}/-{validation_negative}), device={device.type}"
            )

            train_dataset=SegmentationDataset(
                train_rows,
                augment=True,
                seed=42 + fold * 1000,
            )
            validation_dataset=SegmentationDataset(
                validation_rows,
                augment=False,
                seed=42,
            )
            train_sampler=AttentionStage.training_sampler(
                train_rows, 42 + fold
            )
            train_loader=DataLoader(
                train_dataset,
                batch_size=10 if device.type == "cuda" else 3,
                sampler=train_sampler,
                shuffle=False,
                num_workers=0,
                pin_memory=device.type == "cuda",
            )
            validation_loader=DataLoader(
                validation_dataset,
                batch_size=10 if device.type == "cuda" else 3,
                shuffle=False,
                num_workers=0,
                pin_memory=device.type == "cuda",
            )

            model=DataStage.prepare_model(AttentionUNet(), device)
            optimizer=torch.optim.AdamW(
                model.parameters(),
                lr=8e-4,  # initial AdamW learning rate used in the research run
                weight_decay=1e-4,  # small L2 regularization
            )
            scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="max",
                factor=0.5,
                patience=3,
                min_lr=1e-6,
            )
            scaler=DataStage.grad_scaler(device)
            best_state=None
            best_metric=-1.0
            best_epoch=0
            patience=0
            history=[]

            for epoch in range(1, 36 + 1):  # train at most 36 epochs per fold
                train_dataset.set_epoch(epoch)
                model.train()
                train_losses=[]
                for (
                    images,
                    masks,
                    presence_targets,
                    sample_weights,
                    _patient_ids,
                ) in train_loader:
                    images=DataStage.move_tensor(images, device)
                    masks=DataStage.move_tensor(masks, device)
                    presence_targets=DataStage.move_tensor(
                        presence_targets, device
                    )
                    sample_weights=DataStage.move_tensor(
                        sample_weights, device
                    )
                    optimizer.zero_grad(set_to_none=True)
                    with DataStage.autocast(device):
                        logits, presence_logits=model(images)
                        loss=AttentionStage.segmentation_loss(
                            logits,
                            masks,
                            presence_logits,
                            presence_targets,
                            sample_weights,
                        )
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    train_losses.append(float(loss.detach().cpu()))

                model.eval()
                validation_losses=[]
                by_patient_scores=defaultdict(list)
                presence_labels=[]
                presence_predictions=[]
                with torch.inference_mode():
                    for (
                        images,
                        masks,
                        presence_targets,
                        sample_weights,
                        patient_ids,
                    ) in validation_loader:
                        images=DataStage.move_tensor(images, device)
                        masks=DataStage.move_tensor(masks, device)
                        presence_targets_device=DataStage.move_tensor(
                            presence_targets, device
                        )
                        sample_weights=DataStage.move_tensor(
                            sample_weights, device
                        )
                        with DataStage.autocast(device):
                            logits, presence_logits=model(images)
                            loss=AttentionStage.segmentation_loss(
                                logits,
                                masks,
                                presence_logits,
                                presence_targets_device,
                                sample_weights,
                            )
                        validation_losses.append(float(loss.cpu()))
                        binary=(torch.sigmoid(logits) >= 0.5).float()
                        intersection=(binary * masks).sum(dim=(1, 2, 3))
                        denominator=binary.sum(dim=(1, 2, 3)) + masks.sum(
                            dim=(1, 2, 3)
                        )
                        dice=(2 * intersection + 1e-6) / (
                            denominator + 1e-6
                        )
                        negative_quality=1.0 - binary.mean(dim=(1, 2, 3))
                        target_presence_cpu=presence_targets.numpy().astype(int)
                        sample_scores=torch.where(
                            presence_targets_device > 0.5,
                            dice,
                            negative_quality,
                        ).float().cpu().numpy()
                        for patient_id, score in zip(patient_ids, sample_scores):
                            by_patient_scores[str(patient_id)].append(float(score))
                        presence_labels.extend(target_presence_cpu.tolist())
                        presence_predictions.extend(
                            (
                                torch.sigmoid(presence_logits) >= 0.5
                            ).long().cpu().numpy().tolist()
                        )

                mean_train=float(np.mean(train_losses))
                mean_val=float(np.mean(validation_losses))
                patient_balanced=float(
                    np.mean(
                        [np.mean(values) for values in by_patient_scores.values()]
                    )
                )
                presence_balanced=AttentionStage.balanced_accuracy(
                    np.asarray(presence_labels),
                    np.asarray(presence_predictions),
                )
                selection_metric=0.80 * patient_balanced + 0.20 * presence_balanced
                scheduler.step(selection_metric)
                current_lr=float(optimizer.param_groups[0]["lr"])
                history.append(
                    {
                        "epoch": epoch,
                        "train_loss": mean_train,
                        "validation_loss": mean_val,
                        "patient_balanced_segmentation_score_0_5": patient_balanced,
                        "presence_balanced_accuracy_0_5": presence_balanced,
                        "selection_metric": selection_metric,
                        "learning_rate": current_lr,
                    }
                )
                print(
                    f"  epoch={epoch:02d} train_loss={mean_train:.4f} "
                    f"val_loss={mean_val:.4f} seg={patient_balanced:.4f} "
                    f"presence={presence_balanced:.4f} metric={selection_metric:.4f} "
                    f"lr={current_lr:.2e}"
                )
                if selection_metric > best_metric + 1e-5:
                    best_metric=selection_metric
                    best_epoch=epoch
                    best_state={
                        key: value.detach().cpu().clone()
                        for key, value in model.state_dict().items()
                    }
                    patience=0
                else:
                    patience +=1
                    if patience >= 8:
                        print(f"  early stopping after epoch {epoch}.")
                        break

            if best_state is None:
                raise RuntimeError(f"Fold {fold}: no best model state was selected.")
            model.load_state_dict(best_state)
            geometry_prior=AttentionStage.build_geometry_prior(train_rows)
            calibration=AttentionStage.calibrate_threshold(
                model, validation_loader, device, geometry_prior
            )
            fold_state={
                "schema": "simple-attention-2p5d-presence-v3",
                "state_dict": best_state,
                "fingerprint": stable_fingerprint,
                "stable_fingerprint": stable_fingerprint,
                "fingerprint_schema": "stable-v4",
                    "fold": fold,
                "base_channels": 24,
                "input_channels": 3,
                "best_epoch": best_epoch,
                "best_selection_metric": best_metric,
                "calibration": calibration,
                "geometry_prior": geometry_prior,
                "train_positive_targets": train_positive,
                "train_negative_targets": train_negative,
                "train_patients": sorted(
                    {str(row["patient_id"]) for row in train_rows}
                ),
                "validation_patients": sorted(validation_patients),
                "excluded_test_patients": sorted(
                    patient
                    for patient in all_patients
                    if any(
                        str(row["patient_id"]) == patient
                        and int(row["segmentation_fold"]) == fold
                        for row in accepted_rows
                    )
                ),
                "history": history,
            }
            # Save each fold atomically so a disconnected Kaggle session resumes
            # from the first unfinished or incompatible fold.
            if checkpoint_path is not None:
                DataStage.write_torch(checkpoint_path, fold_state)
            fold_states[fold]=(checkpoint_path if checkpoint_path is not None else fold_state)
            fold_summaries.append({
                "fold": fold, "status": "trained",
                "checkpoint": str(checkpoint_path) if checkpoint_path is not None else "",
                "fingerprint": stable_fingerprint,
                "stable_fingerprint": stable_fingerprint,
                "checkpoint_sha256": (
                    DataStage.sha256_file(checkpoint_path)
                    if checkpoint_path is not None and checkpoint_path.is_file() else ""
                ),
                "best_epoch": best_epoch,
                "best_selection_metric": best_metric,
                "calibration": calibration,
                "elapsed": DataStage.format_seconds(time.perf_counter() - fold_started),
            })
            del model, optimizer, scheduler, scaler, train_loader, validation_loader
            del train_dataset, validation_dataset, fold_state, best_state, train_sampler
            DataStage.release_device(device)
            print(f"[ATTENTION] fold {fold} trained in "
                  f"{DataStage.format_seconds(time.perf_counter() - fold_started)}")

        if workspace is not None:
            DataStage.write_json(
                workspace.training_summary,
                {
                    "schema": "simple-attention-2p5d-presence-v3",
                    "device": device.type,
                    "accepted_training_targets": len(accepted_rows),
                    "heart_present_targets": sum(
                        DataStage._as_int(row.get("heart_present"), 1) == 1
                        for row in accepted_rows
                    ),
                    "no_heart_targets": sum(
                        DataStage._as_int(row.get("heart_present"), 1) == 0
                        for row in accepted_rows
                    ),
                    "folds": fold_summaries,
                },
            )
        DataStage.clear_image_cache()
        return fold_states
    # Keep the dominant connected mask component and discard small detached islands.
    @staticmethod
    def largest_component(mask):
        binary=(np.asarray(mask) > 0).astype(np.uint8)
        if not binary.any():
            return binary
        count, labels, stats, _=cv2.connectedComponentsWithStats(
            binary, connectivity=8
        )
        if count <= 1:
            return binary
        label=1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        return (labels == label).astype(np.uint8)
    # Measure mask area and the fraction touching the image boundary.
    @staticmethod
    def mask_geometry(mask):
        mask=(np.asarray(mask) > 0).astype(np.uint8)
        area=float(mask.mean())
        if not mask.any():
            return area, 0.0
        border=np.zeros_like(mask, dtype=bool)
        border[:2, :]=True
        border[-2:, :]=True
        border[:, :2]=True
        border[:, -2:]=True
        boundary_touch=float(
            np.logical_and(mask > 0, border).sum() / max(1, mask.sum())
        )
        return area, boundary_touch
    # Threshold, close small gaps, and keep the largest connected Attention region.
    @staticmethod
    def candidate_mask(probability, threshold):
        mask=(probability >= float(threshold)).astype(np.uint8)
        kernel_size=5  # close small holes/gaps in thresholded masks
        if kernel_size > 1:
            kernel=np.ones((kernel_size, kernel_size), dtype=np.uint8)
            mask=cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        return AttentionStage.largest_component(mask)
    # Validate mask.
    @staticmethod
    def validate_mask(
        mask,
        probability,
        geometry_prior=None,
    ):
        area, boundary=AttentionStage.mask_geometry(mask)
        peak=float(np.max(probability))
        reasons=[]
        if area < 0.003:
            reasons.append("area_too_small")
        if area > 0.65:  # reject implausibly large heart masks
            reasons.append("area_too_large")
        if peak < 0.50:
            reasons.append("low_peak_probability")
        if boundary > 0.35:
            reasons.append("touches_boundary")
        prior_deviation=AttentionStage.calculate_prior_deviation(mask, geometry_prior)
        if prior_deviation > 5.5:
            reasons.append("geometry_outlier")
        return not reasons, ";".join(reasons)
    # Try nearby thresholds and geometry priors when the first automatic mask is implausible.
    @staticmethod
    def repair_probability(
        probability,
        threshold,
        geometry_prior=None,
    ):
        candidates=[]
        thresholds=[threshold] + [
            float(np.clip(threshold + offset, 0.05, 0.95))
            for offset in (-0.20, -0.15, -0.10, -0.05, 0.05, 0.10, 0.15, 0.20)
        ]
        for candidate_threshold in sorted(set(thresholds)):
            mask=AttentionStage.candidate_mask(
                probability, candidate_threshold
            )
            valid, reason=AttentionStage.validate_mask(
                mask, probability, geometry_prior
            )
            _area, boundary=AttentionStage.mask_geometry(mask)
            prior_deviation=AttentionStage.calculate_prior_deviation(
                mask, geometry_prior
            )
            mean_inside=(
                float(probability[mask > 0].mean()) if mask.any() else 0.0
            )
            score=(
                mean_inside
                - 0.50 * boundary
                - 0.06 * min(prior_deviation, 10.0)
            )
            candidates.append(
                (
                    valid,
                    score,
                    mask,
                    candidate_threshold,
                    reason,
                    prior_deviation,
                )
            )
        valid_candidates=[candidate for candidate in candidates if candidate[0]]
        if valid_candidates:
            _, _, mask, used_threshold, _, prior_deviation=max(
                valid_candidates, key=lambda item: item[1]
            )
            return (
                mask,
                True,
                "adaptive_threshold_with_fold_prior",
                float(used_threshold),
                "",
                float(prior_deviation),
            )
        _, _, mask, used_threshold, reason, prior_deviation=max(
            candidates, key=lambda item: item[1]
        )
        return (
            mask,
            False,
            "unresolved",
            float(used_threshold),
            reason,
            float(prior_deviation),
        )
    # Combine entropy, TTA disagreement, presence ambiguity, and geometry deviation.
    @staticmethod
    def uncertainty_metrics(
        probability,
        alternate_probability,
        presence_probability,
        presence_threshold,
        prior_deviation,
        final_valid,
    ):
        probability=np.clip(probability.astype(np.float64), 1e-6, 1.0 - 1e-6)
        entropy=-(
            probability * np.log(probability)
            + (1.0 - probability) * np.log(1.0 - probability)
        ) / math.log(2.0)
        entropy_flat=entropy.reshape(-1)
        top_count=max(1, int(round(0.10 * entropy_flat.size)))
        mean_entropy=float(
            np.partition(entropy_flat, entropy_flat.size - top_count)[-top_count:].mean()
        )
        disagreement=float(
            np.mean(np.abs(probability - alternate_probability))
        )
        disagreement_normalized=min(
            1.0,
            disagreement
            / max(0.08, 1e-6),
        )
        presence_ambiguity=max(
            0.0,
            1.0
            - abs(float(presence_probability) - float(presence_threshold)) / 0.50,
        )
        prior_normalized=min(
            1.0,
            float(prior_deviation)
            / max(5.5, 1e-6),
        )
        uncertainty=(
            0.35 * mean_entropy
            + 0.35 * disagreement_normalized
            + 0.20 * presence_ambiguity
            + 0.10 * prior_normalized
            + (0.15 if not final_valid else 0.0)
        )
        return disagreement, mean_entropy, float(min(1.0, uncertainty))
    # Apply sequence consistency.
    @staticmethod
    def apply_sequence_consistency(
        prediction_rows,
        dataset_rows,
    ):
        base_by_token={
            str(row["image_token"]): row for row in dataset_rows
        }
        groups=defaultdict(list)
        for prediction in prediction_rows:
            token=str(prediction.get("image_token", ""))
            base=base_by_token.get(token, {})
            group=str(
                base.get("sequence_group_id")
                or f"{prediction.get('series_id', '')}::{Path(str(prediction.get('image_path', ''))).parent}"
            )
            prediction["_sequence_index"]=DataStage._as_int(
                base.get("sequence_index"), 0
            )
            groups[group].append(prediction)

        def pair_inconsistency(first, second):
            first_present=DataStage._as_int(first.get("attention_heart_present"), 1)
            second_present=DataStage._as_int(second.get("attention_heart_present"), 1)
            if first_present != second_present:
                return 1.0
            if first_present == 0:
                return 0.0
            first_area=DataStage._as_float(first.get("attention_area_ratio"), 0.0)
            second_area=DataStage._as_float(second.get("attention_area_ratio"), 0.0)
            area_change=min(
                1.0,
                abs(first_area - second_area)
                / max(0.01, 0.5 * (first_area + second_area)),
            )
            first_x=DataStage._as_float(first.get("attention_centroid_x"), np.nan)
            first_y=DataStage._as_float(first.get("attention_centroid_y"), np.nan)
            second_x=DataStage._as_float(second.get("attention_centroid_x"), np.nan)
            second_y=DataStage._as_float(second.get("attention_centroid_y"), np.nan)
            if all(np.isfinite(value) for value in (first_x, first_y, second_x, second_y)):
                centroid_change=min(
                    1.0,
                    math.hypot(first_x - second_x, first_y - second_y) / 0.25,
                )
            else:
                centroid_change=1.0
            return float(0.55 * area_change + 0.45 * centroid_change)

        for group_rows in groups.values():
            group_rows.sort(key=lambda row: DataStage._as_int(row.get("_sequence_index"), 0))
            for index, row in enumerate(group_rows):
                comparisons=[]
                if index > 0:
                    comparisons.append(pair_inconsistency(row, group_rows[index - 1]))
                if index + 1 < len(group_rows):
                    comparisons.append(pair_inconsistency(row, group_rows[index + 1]))
                inconsistency=float(np.mean(comparisons)) if comparisons else 0.0
                previous_inconsistency=DataStage._as_float(
                    row.get("attention_sequence_inconsistency"), 0.0
                )
                base_uncertainty=max(
                    0.0,
                    DataStage._as_float(row.get("attention_uncertainty_score"), 0.0)
                    - 0.20 * previous_inconsistency,
                )
                row["attention_sequence_inconsistency"]=inconsistency
                row["attention_uncertainty_score"]=min(
                    1.0, base_uncertainty + 0.20 * inconsistency
                )
                row.pop("_sequence_index", None)
    # Compare in-memory OOF predictions with accepted persistent manual targets.
    @staticmethod
    def evaluate_oof_segmentation(
        manual_audit,
        prediction_rows,
    ):
        predictions={
            str(row.get("image_token", "")): row for row in prediction_rows
        }
        audit_rows=[
            row
            for row in manual_audit
            if row.get("status") == "ACCEPTED"
        ]
        metrics=[]
        for target_row in audit_rows:
            token=str(target_row.get("image_token", ""))
            prediction_row=predictions.get(token)
            if prediction_row is None:
                continue
            target_path=Path(target_row.get("manual_mask_path", ""))
            if not target_path.is_file():
                continue
            predicted=AttentionStage.read_prediction_mask(prediction_row)
            target=DataStage.read_binary_mask(target_path)
            predicted_bool=predicted > 0
            target_bool=target > 0
            tp=float(np.logical_and(predicted_bool, target_bool).sum())
            fp=float(np.logical_and(predicted_bool, ~target_bool).sum())
            fn=float(np.logical_and(~predicted_bool, target_bool).sum())
            union=tp + fp + fn
            denominator=2.0 * tp + fp + fn
            dice=1.0 if denominator == 0 else 2.0 * tp / denominator
            iou=1.0 if union == 0 else tp / union
            precision=1.0 if tp + fp == 0 and not target_bool.any() else tp / max(1.0, tp + fp)
            recall=1.0 if tp + fn == 0 else tp / max(1.0, tp + fn)
            target_features=AttentionStage.mask_features(target)
            predicted_features=AttentionStage.mask_features(predicted)
            if target_bool.any() and predicted_bool.any():
                centroid_distance=float(
                    math.hypot(
                        predicted_features["centroid_x"] - target_features["centroid_x"],
                        predicted_features["centroid_y"] - target_features["centroid_y"],
                    )
                )
            else:
                centroid_distance=np.nan
            metrics.append(
                {
                    "image_token": token,
                    "patient_id": target_row.get("patient_id", ""),
                    "series_id": target_row.get("series_id", ""),
                    "target_type": target_row.get("target_type", ""),
                    "heart_present": DataStage._as_int(target_row.get("heart_present"), 1),
                    "dice": dice,
                    "iou": iou,
                    "precision": precision,
                    "recall": recall,
                    "target_area_ratio": float(target_bool.mean()),
                    "predicted_area_ratio": float(predicted_bool.mean()),
                    "absolute_area_error": abs(
                        float(predicted_bool.mean()) - float(target_bool.mean())
                    ),
                    "false_positive_area_ratio": float(fp / predicted_bool.size),
                    "centroid_distance_normalized": centroid_distance,
                    "presence_probability": DataStage._as_float(
                        prediction_row.get("attention_presence_probability"), np.nan
                    ),
                    "predicted_heart_present": DataStage._as_int(
                        prediction_row.get("attention_heart_present"), 1
                    ),
                    "uncertainty_score": DataStage._as_float(
                        prediction_row.get("attention_uncertainty_score"), np.nan
                    ),
                    "sequence_inconsistency": DataStage._as_float(
                        prediction_row.get("attention_sequence_inconsistency"), np.nan
                    ),
                }
            )

        positive=[row for row in metrics if DataStage._as_int(row["heart_present"], 1) == 1]
        negative=[row for row in metrics if DataStage._as_int(row["heart_present"], 1) == 0]

        def patient_balanced(rows, key):
            by_patient=defaultdict(list)
            for row in rows:
                value=DataStage._as_float(row.get(key), np.nan)
                if np.isfinite(value):
                    by_patient[str(row.get("patient_id", ""))].append(value)
            values=[np.mean(items) for items in by_patient.values() if items]
            return float(np.mean(values)) if values else np.nan

        summary={
            "evaluated_targets": len(metrics),
            "positive_targets": len(positive),
            "negative_targets": len(negative),
            "patient_balanced_dice_all": patient_balanced(metrics, "dice"),
            "patient_balanced_dice_positive": patient_balanced(positive, "dice"),
            "patient_balanced_iou_positive": patient_balanced(positive, "iou"),
            "patient_balanced_precision_positive": patient_balanced(
                positive, "precision"
            ),
            "patient_balanced_recall_positive": patient_balanced(
                positive, "recall"
            ),
            "no_heart_empty_prediction_rate": (
                float(
                    np.mean(
                        [
                            row["predicted_area_ratio"]
                            < 0.003
                            for row in negative
                        ]
                    )
                )
                if negative
                else np.nan
            ),
            "no_heart_mean_false_positive_area_ratio": (
                float(np.mean([row["false_positive_area_ratio"] for row in negative]))
                if negative
                else np.nan
            ),
        }
        if metrics:
            print(
                "[ATTENTION OOF] "
                f"targets={len(metrics)}, positive_dice_patient="
                f"{summary['patient_balanced_dice_positive']:.4f}, "
                f"negative_empty_rate={summary['no_heart_empty_prediction_rate']}"
            )
        return summary, pd.DataFrame(metrics)
    # Predict every image from freshly trained OOF states and save automatic masks.
    @staticmethod
    def _predict_attention_masks_in_memory(rows, workspace, device, fold_states):
        # Every run predicts the complete dataset from freshly trained OOF models.
        # Final binary masks are also persisted for HTML review after a restart.
        results=[None] * len(rows)
        started=time.perf_counter()

        for fold in range(5):  # five patient-level folds
            fold_indices=[
                index
                for index, row in enumerate(rows)
                if int(row["segmentation_fold"]) == fold
            ]
            fold_rows=[rows[index] for index in fold_indices]
            if not fold_rows:
                continue

            fold_state=fold_states.pop(fold)
            held_out={str(row["patient_id"]) for row in fold_rows}
            seen=set(fold_state["train_patients"]) | set(fold_state["validation_patients"])
            if held_out & seen:
                raise RuntimeError(f"Fold {fold}: OOF prediction would leak held-out patients.")
            model=AttentionUNet(
                base_channels=int(
                    fold_state.get(
                        "base_channels", 24
                    )
                ),
                input_channels=int(
                    fold_state.get(
                        "input_channels", 3
                    )
                ),
            )
            model.load_state_dict(fold_state["state_dict"])
            model=DataStage.prepare_model(model, device).eval()
            calibration=fold_state.get("calibration", {}) or {}
            threshold=float(
                calibration.get(
                    "threshold", 0.50
                )
            )
            presence_threshold=float(
                calibration.get(
                    "presence_threshold",
                    0.50,
                )
            )
            geometry_prior=fold_state.get("geometry_prior", {}) or {}
            loader=DataLoader(
                SegmentationInferenceDataset(fold_rows),
                batch_size=10 if device.type == "cuda" else 3,
                shuffle=False,
                num_workers=0,
                pin_memory=device.type == "cuda",
            )
            print(
                f"[ATTENTION] 2.5D prediction for fold {fold}: {len(fold_rows)} "
                f"images, device={device.type}"
            )
            fold_results=[]

            with torch.inference_mode():
                for images, local_indices in tqdm(
                    loader, desc=f"Attention 2.5D fold {fold}"):
                    images=DataStage.move_tensor(images, device)
                    with DataStage.autocast(device):
                        logits, presence_logits=model(images)
                    original_probability=torch.sigmoid(logits)[:, 0]
                    original_presence=torch.sigmoid(presence_logits)

                    factor=1.10  # second photometric view for uncertainty estimation
                    tta_images=torch.clamp(
                        (images - 0.5) * factor + 0.5, 0.0, 1.0
                    )
                    with DataStage.autocast(device):
                        tta_logits, tta_presence_logits=model(tta_images)
                    tta_probability=torch.sigmoid(tta_logits)[:, 0]
                    tta_presence=torch.sigmoid(tta_presence_logits)

                    averaged_probability=(
                        original_probability + tta_probability
                    ) / 2.0
                    averaged_presence=(
                        original_presence + tta_presence
                    ) / 2.0
                    probabilities=averaged_probability.float().cpu().numpy()
                    alternate_probabilities=tta_probability.float().cpu().numpy()
                    presence_probabilities=(
                        averaged_presence.float().cpu().numpy()
                    )

                    for batch_position, local_index_tensor in enumerate(
                        local_indices):
                        local_index=int(local_index_tensor)
                        global_index=fold_indices[local_index]
                        row=rows[global_index]
                        probability=probabilities[batch_position]
                        alternate_probability=alternate_probabilities[
                            batch_position
                        ]
                        presence_probability=float(
                            presence_probabilities[batch_position]
                        )
                        heart_present=bool(
                            presence_probability >= presence_threshold
                            or float(probability.max())
                            >= 0.80
                        )
                        initial_mask=AttentionStage.candidate_mask(
                            probability, threshold
                        )
                        initial_valid, initial_reason=(
                            AttentionStage.validate_mask(
                                initial_mask, probability, geometry_prior
                            )
                        )

                        if not heart_present:
                            final_mask=np.zeros_like(initial_mask, dtype=np.uint8)
                            final_valid=True
                            repair_method="presence_head_empty"
                            used_threshold=threshold
                            final_reason=""
                            prior_deviation=0.0
                        elif initial_valid:
                            final_mask=initial_mask
                            final_valid=True
                            repair_method="none"
                            used_threshold=threshold
                            final_reason=""
                            prior_deviation=AttentionStage.calculate_prior_deviation(
                                final_mask, geometry_prior
                            )
                        else:
                            (
                                final_mask,
                                final_valid,
                                repair_method,
                                used_threshold,
                                final_reason,
                                prior_deviation,
                            )=AttentionStage.repair_probability(
                                probability, threshold, geometry_prior
                            )
                            repair_method="photometric_tta+" + repair_method

                        geometry=AttentionStage.mask_features(final_mask)
                        area=float(geometry["area"])
                        boundary=float(geometry["boundary"])
                        centroid_x=geometry["centroid_x"]
                        centroid_y=geometry["centroid_y"]
                        disagreement, entropy, uncertainty=(
                            AttentionStage.uncertainty_metrics(
                                probability,
                                alternate_probability,
                                presence_probability,
                                presence_threshold,
                                prior_deviation,
                                final_valid,
                            )
                        )
                        mask_path=Path(row["predicted_attention_mask_path"])
                        DataStage.write_png(
                            mask_path, final_mask.astype(np.uint8) * 255
                        )
                        # Keep a compact RAM copy so the current run does not reread PNGs.
                        packed_mask=np.packbits(final_mask, axis=None)

                        result_row={
                            **row,
                            "attention_valid_initial": int(initial_valid),
                            "attention_valid_final": int(final_valid),
                            "attention_invalid_reason_initial": initial_reason,
                            "attention_invalid_reason_final": final_reason,
                            "attention_repair_method": repair_method,
                            "attention_threshold_used": float(used_threshold),
                            "attention_area_ratio": area,
                            "attention_peak_probability": float(
                                probability.max()
                            ),
                            "attention_boundary_touch_fraction": boundary,
                            "attention_heart_present": int(heart_present),
                            "attention_presence_probability": presence_probability,
                            "attention_presence_threshold": presence_threshold,
                            "attention_tta_disagreement": disagreement,
                            "attention_mean_entropy": entropy,
                            "attention_uncertainty_score": uncertainty,
                            "attention_sequence_inconsistency": 0.0,
                            "attention_centroid_x": centroid_x,
                            "attention_centroid_y": centroid_y,
                            "attention_prior_deviation": float(prior_deviation),
                            "attention_mask_bits": packed_mask,
                        }
                        results[global_index]=result_row
                        fold_results.append(result_row)

            AttentionStage.apply_sequence_consistency(
                fold_results, fold_rows
            )
            del model, loader, fold_state, fold_results
            DataStage.release_device(device)

        final_results=[result for result in results if result is not None]
        if len(final_results) != len(rows):
            raise RuntimeError(
                "Prediction did not produce exactly one row per image."
            )
        AttentionStage.apply_sequence_consistency(final_results, rows)
        DataStage.write_csv(
            workspace.prediction_audit,
            final_results,
            AttentionStage.PREDICTION_FIELDS,
        )
        invalid_rows=[
            row
            for row in final_results
            if DataStage._as_int(row.get("attention_valid_final"), 0) != 1
        ]
        summary={
            "device": device.type,
            "images": len(final_results),
            "valid_initial": sum(
                DataStage._as_int(row.get("attention_valid_initial"), 0)
                for row in final_results
            ),
            "valid_final": sum(
                DataStage._as_int(row.get("attention_valid_final"), 0)
                for row in final_results
            ),
            "predicted_heart_present": sum(
                DataStage._as_int(row.get("attention_heart_present"), 0)
                for row in final_results
            ),
            "predicted_no_heart": sum(
                DataStage._as_int(row.get("attention_heart_present"), 0) == 0
                for row in final_results
            ),
            "uncertain_above_review_threshold": sum(
                DataStage._as_float(row.get("attention_uncertainty_score"), 0.0)
                >= 0.18
                for row in final_results
            ),
            "invalid_final": len(invalid_rows),
            "predicted_masks": str(workspace.predicted_masks),
            "prediction_audit": str(workspace.prediction_audit),
            "elapsed": DataStage.format_seconds(
                time.perf_counter() - started
            ),
        }
        print(
            "[ATTENTION] "
            f"valid_final={summary['valid_final']}/{summary['images']}, "
            f"heart_present={summary['predicted_heart_present']}, "
            f"invalid={summary['invalid_final']} | masks={workspace.predicted_masks}"
        )
        DataStage.clear_image_cache()
        return final_results, summary

    @staticmethod
    def predict_attention_masks(rows, workspace, device, fold_states=None, force=False):
        """Run OOF inference per fold and adopt complete legacy parts without recomputation."""
        rows=list(rows)
        fold_states={} if fold_states is None else dict(fold_states)
        checkpoint_map=AttentionStage.load_checkpoint_map(workspace)
        stable_prediction_fingerprint=AttentionStage.prediction_fingerprint(rows, checkpoint_map)
        existing_summary=DataStage.read_json(workspace.prediction_summary, {}) or {}
        results=[None] * len(rows)
        reused_folds=[]
        computed_folds=[]
        adopted_folds=[]
        started=time.perf_counter()

        for fold in range(5):
            fold_indices=[index for index,row in enumerate(rows) if int(row["segmentation_fold"]) == fold]
            fold_rows=[rows[index] for index in fold_indices]
            if not fold_rows:
                continue
            checkpoint_path=checkpoint_map[fold]
            cache=AttentionStage.prediction_part_cache_status(
                fold, fold_rows, workspace, checkpoint_path
            )
            if not force and cache["reusable"]:
                for global_index, base in zip(fold_indices, fold_rows):
                    token=str(base["image_token"])
                    record=dict(base)
                    saved=cache["cached_by_token"][token]
                    for field in AttentionStage.PREDICTION_FIELDS:
                        if field.startswith("attention_") or field == "checkpoint_fingerprint":
                            record[field]=saved.get(field, record.get(field, ""))
                    record["predicted_attention_mask_path"]=str(
                        Path(workspace.predicted_masks) / f"{token}.png"
                    )
                    results[global_index]=record
                reused_folds.append(fold)
                if cache["adopted"]:
                    adopted_folds.append(fold)
                    metadata={
                        **cache["metadata"],
                        "stable_fingerprint": cache["stable_fingerprint"],
                        "checkpoint_sha256": DataStage.sha256_file(checkpoint_path),
                        "completed": True,
                        "legacy_cache_adopted": True,
                    }
                    DataStage.write_json(cache["part_json"], metadata)
                print(
                    f"[ATTENTION] Fold {fold}: persisted prediction part reused"
                    + (" (legacy metadata adopted)." if cache["adopted"] else ".")
                )
                fold_states.pop(fold, None)
                continue

            fold_state=fold_states.pop(fold, None)
            if isinstance(fold_state, (str, Path)):
                fold_state=AttentionStage.torch_load(Path(fold_state))
            if fold_state is None:
                fold_state=AttentionStage.torch_load(checkpoint_path)
            checkpoint_fingerprint=str(
                fold_state.get("stable_fingerprint") or fold_state.get("fingerprint", "")
            )
            part_workspace=workspace._replace(prediction_audit=cache["part_csv"])
            fold_results, _fold_summary=AttentionStage._predict_attention_masks_in_memory(
                fold_rows, part_workspace, device, {fold: fold_state}
            )
            for record in fold_results:
                record["checkpoint_fingerprint"]=checkpoint_fingerprint
            AttentionStage.apply_sequence_consistency(fold_results, fold_rows)
            DataStage.write_csv(cache["part_csv"], fold_results, AttentionStage.PREDICTION_FIELDS)
            DataStage.write_json(
                cache["part_json"],
                {
                    "schema": "simple-prediction-part-stable-v6",
                    "fingerprint": cache["stable_fingerprint"],
                    "stable_fingerprint": cache["stable_fingerprint"],
                    "fold": fold, "images": len(fold_results),
                    "checkpoint": str(checkpoint_path),
                    "checkpoint_sha256": DataStage.sha256_file(checkpoint_path),
                    "device_used": device.type, "completed": True,
                },
            )
            by_token={str(row["image_token"]): row for row in fold_results}
            for global_index, base in zip(fold_indices, fold_rows):
                results[global_index]=by_token[str(base["image_token"])]
            computed_folds.append(fold)
            del fold_state, fold_results, by_token
            DataStage.release_device(device)

        final_results=[result for result in results if result is not None]
        if len(final_results) != len(rows):
            raise RuntimeError("Prediction resume did not produce exactly one metadata row per image.")
        AttentionStage.apply_sequence_consistency(final_results, rows)
        DataStage.write_csv(workspace.prediction_audit, final_results, AttentionStage.PREDICTION_FIELDS)
        invalid_rows=[row for row in final_results if DataStage._as_int(row.get("attention_valid_final"),0) != 1]
        DataStage.write_csv(workspace.invalid_predictions, invalid_rows, AttentionStage.PREDICTION_FIELDS)

        all_reused=(
            not computed_folds
            and set(reused_folds) == set(range(5))
            and DataStage._as_int(existing_summary.get("images"), -1) == len(final_results)
        )
        if all_reused:
            # Keep the existing summary byte-for-byte so a harmless verification run
            # cannot invalidate an already computed feature bank.
            summary={
                **existing_summary,
                "reused_folds": reused_folds,
                "computed_folds": [],
                "adopted_legacy_folds": adopted_folds,
            }
        else:
            summary={
                "schema": "simple-predictions-stable-v6",
                "fingerprint": stable_prediction_fingerprint,
                "stable_fingerprint": stable_prediction_fingerprint,
                "device_used_for_computed_folds": device.type,
                "reused_folds": reused_folds, "computed_folds": computed_folds,
                "adopted_legacy_folds": adopted_folds,
                "images": len(final_results),
                "valid_initial": sum(DataStage._as_int(row.get("attention_valid_initial"),0) for row in final_results),
                "valid_final": sum(DataStage._as_int(row.get("attention_valid_final"),0) for row in final_results),
                "predicted_heart_present": sum(DataStage._as_int(row.get("attention_heart_present"),0) for row in final_results),
                "predicted_no_heart": sum(DataStage._as_int(row.get("attention_heart_present"),0)==0 for row in final_results),
                "invalid_final": len(invalid_rows),
                "prediction_audit": str(workspace.prediction_audit),
                "prediction_parts": str(workspace.prediction_parts_dir),
                "predicted_masks": str(workspace.predicted_masks),
                "elapsed": DataStage.format_seconds(time.perf_counter()-started),
            }
            DataStage.write_json(workspace.prediction_summary, summary)
        print(
            "[ATTENTION] resumable OOF prediction complete: "
            f"reused_folds={reused_folds}, computed_folds={computed_folds}, "
            f"valid={summary['valid_final']}/{summary['images']}"
        )
        DataStage.clear_image_cache()
        return final_results, summary

    @staticmethod
    def combine_complete_prediction_parts(rows, workspace):
        """Assemble a final audit from five complete parts, including legacy parts."""
        rows=list(rows)
        try:
            checkpoint_map=AttentionStage.load_checkpoint_map(workspace)
        except Exception:
            return None
        results=[None] * len(rows)
        adopted=[]
        for fold in range(5):
            fold_indices=[index for index,row in enumerate(rows) if int(row["segmentation_fold"]) == fold]
            fold_rows=[rows[index] for index in fold_indices]
            if not fold_rows:
                continue
            cache=AttentionStage.prediction_part_cache_status(
                fold, fold_rows, workspace, checkpoint_map[fold]
            )
            if not cache["reusable"]:
                return None
            if cache["adopted"]:
                adopted.append(fold)
                DataStage.write_json(
                    cache["part_json"],
                    {
                        **cache["metadata"],
                        "stable_fingerprint": cache["stable_fingerprint"],
                        "checkpoint_sha256": DataStage.sha256_file(checkpoint_map[fold]),
                        "completed": True, "legacy_cache_adopted": True,
                    },
                )
            for global_index, base in zip(fold_indices, fold_rows):
                token=str(base["image_token"])
                record=dict(base)
                saved=cache["cached_by_token"][token]
                for field in AttentionStage.PREDICTION_FIELDS:
                    if field.startswith("attention_") or field == "checkpoint_fingerprint":
                        record[field]=saved.get(field, record.get(field, ""))
                record["predicted_attention_mask_path"]=str(
                    Path(workspace.predicted_masks) / f"{token}.png"
                )
                results[global_index]=record

        final_results=[row for row in results if row is not None]
        if len(final_results) != len(rows):
            return None
        AttentionStage.apply_sequence_consistency(final_results, rows)
        DataStage.write_csv(workspace.prediction_audit, final_results, AttentionStage.PREDICTION_FIELDS)
        invalid_rows=[row for row in final_results if DataStage._as_int(row.get("attention_valid_final"),0) != 1]
        DataStage.write_csv(workspace.invalid_predictions, invalid_rows, AttentionStage.PREDICTION_FIELDS)
        stable=AttentionStage.prediction_fingerprint(rows, checkpoint_map)
        old=DataStage.read_json(workspace.prediction_summary, {}) or {}
        summary={
            "schema": "simple-predictions-stable-v6",
            "fingerprint": old.get("fingerprint") or stable,
            "stable_fingerprint": stable,
            "device_used_for_computed_folds": "none_part_assembly_only",
            "reused_folds": list(range(5)), "computed_folds": [],
            "adopted_legacy_folds": adopted,
            "images": len(final_results),
            "valid_initial": sum(DataStage._as_int(row.get("attention_valid_initial"),0) for row in final_results),
            "valid_final": sum(DataStage._as_int(row.get("attention_valid_final"),0) for row in final_results),
            "predicted_heart_present": sum(DataStage._as_int(row.get("attention_heart_present"),0) for row in final_results),
            "predicted_no_heart": sum(DataStage._as_int(row.get("attention_heart_present"),0)==0 for row in final_results),
            "invalid_final": len(invalid_rows),
            "prediction_audit": str(workspace.prediction_audit),
            "prediction_parts": str(workspace.prediction_parts_dir),
            "predicted_masks": str(workspace.predicted_masks),
            "elapsed": "assembled from completed fold parts",
        }
        DataStage.write_json(workspace.prediction_summary, summary)
        print("[ATTENTION] Combined audit reconstructed from five completed fold parts; no GPU used.")
        return final_results, summary

    # Read a current-session packed mask first, then fall back to its persistent PNG.
    @staticmethod
    def read_prediction_mask(row, size=256):
        packed=row.get("attention_mask_bits")
        if packed is not None:
            packed=np.asarray(packed, dtype=np.uint8)
            if packed.size != 8192:
                raise RuntimeError(
                    "An automatic mask must contain exactly 256 x 256 packed bits."
                )
            mask=np.unpackbits(packed).reshape(256, 256).astype(np.uint8)
        else:
            path=Path(str(row.get("predicted_attention_mask_path", "")))
            if not path.is_file():
                raise FileNotFoundError(
                    f"Missing automatic Attention mask for {row.get('image_token', '?')}: {path}"
                )
            mask=DataStage.read_binary_mask(path)
        if int(size) != 256:
            mask=cv2.resize(
                mask, (int(size), int(size)), interpolation=cv2.INTER_NEAREST
            )
        return (mask > 0).astype(np.uint8)

    # True when the editor can recover an automatic mask from RAM or disk.
    @staticmethod
    def has_prediction_mask(row):
        packed=row.get("attention_mask_bits")
        if packed is not None:
            try:
                return np.asarray(packed, dtype=np.uint8).size == 8192
            except Exception:
                return False
        path=Path(str(row.get("predicted_attention_mask_path", "")))
        return path.is_file()

    # Merge saved full-compatible prediction metadata into freshly discovered dataset rows.
    @staticmethod
    def load_saved_predictions(rows, workspace):
        candidates=[]
        for priority, path in (
            (2, workspace.prediction_audit),
            (1, workspace.root / "attention_unet_full_review_manifest.csv"),
        ):
            records=DataStage.read_csv(path)
            valid=[
                record for record in records
                if str(record.get("image_token", "")).strip()
            ]
            candidates.append((len(valid), priority, path, valid))
        _count, _priority, source_path, records=max(
            candidates, key=lambda item: (item[0], item[1])
        )
        by_token={}
        for record in records:
            token=str(record.get("image_token", "")).strip()
            if token in by_token:
                raise RuntimeError(
                    f"Duplicate image token in prediction audit {source_path}: {token}"
                )
            by_token[token]=record

        metadata_rows=0
        mask_rows=0
        normalized=[]
        for row in rows:
            token=str(row["image_token"])
            canonical=workspace.predicted_masks / f"{token}.png"
            record=by_token.get(token)
            source_mask=Path(str(record.get("predicted_attention_mask_path", ""))) if record else None
            mask_path=canonical if canonical.is_file() else source_mask
            if mask_path is None or not mask_path.is_file():
                row["predicted_attention_mask_path"]=str(canonical)
                continue
            row["predicted_attention_mask_path"]=str(mask_path)
            mask_rows +=1
            if record is not None:
                for field in AttentionStage.PREDICTION_FIELDS:
                    if field.startswith("attention_"):
                        row[field]=record.get(field, row.get(field, ""))
                metadata_rows +=1
                normalized.append(row)

        if records and source_path != workspace.prediction_audit and normalized:
            DataStage.write_csv(
                workspace.prediction_audit,
                normalized,
                AttentionStage.PREDICTION_FIELDS,
            )
            source_path=workspace.prediction_audit
        print(
            "[ATTENTION][PERSISTED] "
            f"mask files={mask_rows}/{len(rows)}, metadata={metadata_rows}/{len(rows)}, "
            f"audit={source_path if records else 'not found'}"
        )
        return rows, {
            "mask_files": mask_rows,
            "metadata_rows": metadata_rows,
            "audit_path": str(source_path) if records else "",
        }

# =============================================================================
# STAGE 3 — OPTIONAL HTML REVIEW
# =============================================================================
class HammingBKTree:
    """Search perceptual hashes by Hamming distance without scanning every hash."""

    def __init__(self):
        self.root=None

    def add(self, value):
        value=int(value)
        if self.root is None:
            self.root=(value, {})
            return
        node_value, children=self.root
        while True:
            distance=int(value ^ node_value).bit_count()
            child=children.get(distance)
            if child is None:
                children[distance]=(value, {})
                return
            node_value, children=child

    def has_near(self, value, maximum_distance):
        if self.root is None:
            return False
        stack=[self.root]
        while stack:
            node_value, children=stack.pop()
            distance=int(value ^ node_value).bit_count()
            if distance <= maximum_distance:
                return True
            low=distance - maximum_distance
            high=distance + maximum_distance
            stack.extend(child for edge, child in children.items() if low <= edge <= high)
        return False

class ReviewStage:
    """Select unresolved images and preserve review actions between sessions."""

    HISTORY_FIELDS=(
        "timestamp_utc", "review_round", "action", "image_token",
        "patient_id", "series_id", "queue_position", "queue_size",
        "mask_area_ratio", "manual_mask_path",
    )

    # Add the latest manual decisions to the current-session image records.
    @staticmethod
    def merge_review_rows(
        dataset_rows,
        workspace,
    ):
        annotations=DataStage.load_annotations(workspace)
        merged=[]
        for original in dataset_rows:
            row=dict(original)
            annotation=annotations.get(str(row["image_token"]), {})
            annotation_type=ReviewStage.normalize_target_type(
                annotation.get("target_type", "")
            )
            row["manual_annotation_type"]=annotation_type
            row["review_target_type"]=annotation_type or "UNLABELED"
            row["manual_annotation_source"]=annotation.get("source", "")
            mask_qc=DataStage.manual_mask_qc(row["manual_mask_path"])
            row.update(
                {
                    "manual_mask_exists": mask_qc["exists"],
                    "manual_mask_usable": mask_qc["usable"],
                    "manual_mask_area_ratio": mask_qc["area_ratio"],
                    "manual_mask_reason": mask_qc["reason"],
                }
            )
            merged.append(row)
        return merged
    # Normalize target type.
    @staticmethod
    def normalize_target_type(value):
        target=str(value or "").strip().upper()
        if target in {"", "UNLABELED", "NONE", "NAN"}:
            return ""
        return target
    # Read the normalized explicit review label attached to one image.
    @staticmethod
    def row_target_type(row):
        return ReviewStage.normalize_target_type(
            row.get("manual_annotation_type", row.get("target_type", ""))
        )

    # Exclude items already handled or explicitly skipped in the same review round.
    @staticmethod
    def reviewed_tokens(workspace, review_round):
        return {
            str(row.get("image_token", ""))
            for row in DataStage.read_csv(workspace.review_history)
            if DataStage._as_int(row.get("review_round"), 0) == int(review_round)
        }

    # Persist editor actions so a reopened full-style queue does not repeat skipped images.
    @staticmethod
    def log(workspace, row, review_round, action, mask, position, queue_size):
        history=DataStage.read_csv(workspace.review_history)
        history.append(
            {
                "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
                "review_round": int(review_round),
                "action": str(action),
                "image_token": row["image_token"],
                "patient_id": row["patient_id"],
                "series_id": row["series_id"],
                "queue_position": int(position),
                "queue_size": int(queue_size),
                "mask_area_ratio": "" if mask is None else float((mask > 0).mean()),
                "manual_mask_path": row["manual_mask_path"],
            }
        )
        DataStage.write_csv(
            workspace.review_history, history, ReviewStage.HISTORY_FIELDS
        )
    # Distribute review candidates across patients and series instead of oversampling one scan.
    @staticmethod
    def round_robin_review(
        rows,
        limit,
        per_patient,
        seed,
    ):
        def priority(row):
            explicit_priority=row.get("review_priority", "")
            if str(explicit_priority).strip() != "":
                primary=DataStage._as_float(explicit_priority, 0.0)
            else:
                uncertainty=DataStage._as_float(
                    row.get("attention_uncertainty_score"), 0.0
                )
                valid=DataStage._as_int(row.get("attention_valid_final"), 0)
                primary=-uncertainty if uncertainty > 0 else float(valid)
            peak=DataStage._as_float(row.get("attention_peak_probability"), 0.0)
            tie=hashlib.sha256(
                f"{seed}|{row['image_token']}".encode("utf-8")
            ).hexdigest()
            return (primary, peak, tie)

        by_patient_series=defaultdict(
            lambda: defaultdict(list)
        )
        for row in rows:
            sequence_group=str(
                row.get("sequence_group_id") or row["series_id"]
            )
            by_patient_series[str(row["patient_id"])][sequence_group].append(row)

        patient_queues={}
        for patient_id, series_map in by_patient_series.items():
            series_ids=sorted(series_map)
            for series_id in series_ids:
                series_map[series_id].sort(key=priority)
            queue=[]
            position=0
            while len(queue) < per_patient:
                added=False
                for series_id in series_ids:
                    if position < len(series_map[series_id]):
                        queue.append(series_map[series_id][position])
                        added=True
                        if len(queue) >= per_patient:
                            break
                if not added:
                    break
                position +=1
            patient_queues[patient_id]=queue

        selected=[]
        position=0
        while len(selected) < limit:
            added=False
            for patient_id in sorted(patient_queues):
                queue=patient_queues[patient_id]
                if position < len(queue):
                    selected.append(queue[position])
                    added=True
                    if len(selected) >= limit:
                        break
            if not added:
                break
            position +=1
        return selected
    # Build full-style queues from persisted automatic masks and prediction metadata.
    @staticmethod
    def select_review_rows(
        dataset_rows,
        workspace,
        scope="invalid",
        limit=None,
        seed=42,
        review_round=1,
    ):
        scope=str(scope).lower()
        allowed={"invalid", "uncertain", "empty", "novel", "manual", "all"}
        if scope not in allowed:
            raise ValueError("scope must be invalid/uncertain/empty/novel/manual/all.")
        limit=int(300 if limit is None else limit)
        all_rows=ReviewStage.merge_review_rows(dataset_rows, workspace)
        target_counts=defaultdict(int)
        for row in all_rows:
            target_counts[ReviewStage.row_target_type(row) or "UNLABELED"] +=1
        rows=[row for row in all_rows if ReviewStage.row_target_type(row) == ""]
        reviewed=ReviewStage.reviewed_tokens(workspace, review_round)
        print(
            "[REVIEW][TARGET FILTER] "
            f"UNLABELED={len(rows)}, already-labeled excluded={len(all_rows)-len(rows)}, "
            f"reviewed in round {int(review_round)}={len(reviewed)}, "
            f"distribution={dict(sorted(target_counts.items()))}"
        )

        def available(row):
            return AttentionStage.has_prediction_mask(row)

        if scope == "invalid":
            candidates=[
                row for row in rows
                if DataStage._as_int(row.get("attention_valid_final"), 1) == 0
                and available(row)
                and row["image_token"] not in reviewed
            ]
            for row in candidates:
                row["review_priority"]=DataStage._as_float(
                    row.get("attention_peak_probability"), 0.0
                )
            candidates=ReviewStage.round_robin_review(
                candidates, max(1, limit),
                max(10, int(math.ceil(limit / max(1, len({r['patient_id'] for r in candidates})))) if candidates else 10),
                seed,
            )
        elif scope == "uncertain":
            candidates=[]
            for row in rows:
                if row["image_token"] in reviewed:
                    continue
                if DataStage._as_int(row.get("quality_valid"), 0) != 1 or not available(row):
                    continue
                uncertainty=DataStage._as_float(row.get("attention_uncertainty_score"), 0.0)
                presence=DataStage._as_float(row.get("attention_presence_probability"), np.nan)
                threshold=DataStage._as_float(row.get("attention_presence_threshold"), 0.50)
                near_boundary=np.isfinite(presence) and abs(presence - threshold) <= 0.15
                if (
                    uncertainty < 0.18
                    and not near_boundary
                    and DataStage._as_int(row.get("attention_valid_final"), 1) == 1
                ):
                    continue
                row["review_priority"]=-uncertainty
                candidates.append(row)
            candidates=ReviewStage.round_robin_review(
                candidates, max(1, limit),
                max(10, int(math.ceil(limit / max(1, len({r['patient_id'] for r in candidates})))) if candidates else 10),
                seed,
            )
        elif scope == "empty":
            candidates=[
                row for row in rows
                if Path(row["manual_mask_path"]).is_file()
                and not bool(row.get("manual_annotation_type"))
                and str(row.get("manual_mask_reason", "")) == "empty_or_nearly_empty"
                and available(row)
                and row["image_token"] not in reviewed
            ]
            for row in candidates:
                row["review_priority"]=-DataStage._as_float(
                    row.get("attention_uncertainty_score"), 0.0
                )
            candidates=ReviewStage.round_robin_review(
                candidates, max(1, limit),
                max(10, int(math.ceil(limit / max(1, len({r['patient_id'] for r in candidates})))) if candidates else 10),
                seed,
            )
        elif scope == "manual":
            candidates=[
                row for row in rows
                if Path(row["manual_mask_path"]).is_file()
                and available(row)
                and row["image_token"] not in reviewed
            ]
            candidates.sort(key=lambda row: (row["patient_id"], row["series_id"], row["image_token"]))
        elif scope == "all":
            candidates=[
                row for row in rows
                if available(row) and row["image_token"] not in reviewed
            ]
            candidates.sort(key=lambda row: (row["patient_id"], row["series_id"], row["image_token"]))
        else:
            manual_hashes=[]
            for row in all_rows:
                annotated=bool(row.get("manual_annotation_type")) or Path(row["manual_mask_path"]).is_file()
                if annotated and row.get("perceptual_hash"):
                    manual_hashes.append(int(str(row["perceptual_hash"]), 16))
            if not manual_hashes:
                raise RuntimeError("The novel scope requires at least one labeled image.")
            tree=HammingBKTree()
            for value in sorted(set(manual_hashes)):
                tree.add(value)
            candidates=[]
            excluded=defaultdict(int)
            for row in rows:
                if bool(row.get("manual_annotation_type")) or Path(row["manual_mask_path"]).is_file():
                    excluded["already_annotated"] +=1
                    continue
                if row["image_token"] in reviewed:
                    excluded["already_reviewed_this_round"] +=1
                    continue
                if DataStage._as_int(row.get("quality_valid"), 0) != 1:
                    excluded["quality_invalid"] +=1
                    continue
                if not available(row):
                    excluded["missing_attention_mask"] +=1
                    continue
                phash=str(row.get("perceptual_hash", ""))
                if not phash:
                    excluded["missing_phash"] +=1
                    continue
                if tree.has_near(int(phash, 16), 6):
                    excluded["similar_to_annotated"] +=1
                    continue
                row["review_priority"]=-DataStage._as_float(
                    row.get("attention_uncertainty_score"), 0.0
                )
                candidates.append(row)
            candidates=ReviewStage.round_robin_review(
                candidates, max(1, limit), 10, seed
            )
            print(f"[REVIEW novel] excluded={dict(excluded)}")

        if limit > 0 and scope in {"manual", "all"}:
            candidates=candidates[:limit]
        if not candidates:
            raise RuntimeError(
                f"No target=UNLABELED images are eligible for scope={scope!r}. "
                "The queue requires saved automatic masks and excludes labeled or "
                f"already-reviewed images from round {int(review_round)}."
            )
        print(
            f"[REVIEW] scope={scope}, round={int(review_round)}, images={len(candidates)}"
        )
        return candidates

class MaskEditor:
    """Kaggle-safe HTML5 editor for correcting segmentation targets.

    Keyboard listeners are installed once and isolated from notebook command mode.
    Labeled images remain navigable during the current session but are excluded
    when a later review queue is constructed.
    """

    def __init__(
        self,
        rows,
        workspace,
        start_index=0,
        brush_radius=8,
        review_round=1,
        review_scope="invalid",):
        try:
            widgets=importlib.import_module("ipywidgets")
        except ModuleNotFoundError as error:
            raise RuntimeError(
                "The HTML review editor requires ipywidgets in the active Python/Jupyter "
                "environment. Install it with: python -m pip install ipywidgets"
            ) from error

        annotations=DataStage.load_annotations(workspace)
        unlabeled_rows=[]
        for source_row in rows:
            row=dict(source_row)
            annotation=annotations.get(str(row.get("image_token", "")), {})
            target=ReviewStage.normalize_target_type(
                annotation.get("target_type", row.get("manual_annotation_type", ""))
            )
            if target:
                continue
            row["manual_annotation_type"]=""
            row["review_target_type"]="UNLABELED"
            unlabeled_rows.append(row)
        if not unlabeled_rows:
            raise ValueError(
                "The editor has no remaining target=UNLABELED images. "
                "Previously decided targets are not reopened."
            )

        self.rows=unlabeled_rows
        self.workspace=workspace
        self.index=int(np.clip(start_index, 0, len(self.rows) - 1))
        self.brush_radius=max(1, int(brush_radius))
        self.review_round=int(review_round)
        self.review_scope=str(review_scope)
        self.image=None
        self.auto_mask=None
        self.base_mask=None
        self.mask=None
        self.widget_id=uuid.uuid4().hex[:12]
        self._shortcut_busy=False
        self._last_shortcut_payload=""
        self._last_shortcut_at=0.0

        self.output=widgets.Output()
        self.mask_sync=widgets.Textarea(
            value="",
            placeholder=f"CAD_MASK_SYNC_{self.widget_id}",
            layout=widgets.Layout(width="1px", height="1px", display="none"),
        )
        self.shortcut_sync=widgets.Textarea(
            value="",
            placeholder=f"CAD_SHORTCUT_SYNC_{self.widget_id}",
            layout=widgets.Layout(width="1px", height="1px", display="none"),
        )

        def compact_button(
            description,
            width,
            tooltip,
            button_style="",):
            return widgets.Button(
                description=description,
                tooltip=tooltip,
                button_style=button_style,
                layout=widgets.Layout(width=width, height="29px"),
                style={"font_weight": "500"},
            )

        self.previous_button=compact_button("← Prev [P]", "68px", "Previous — P")
        self.next_button=compact_button("Next [N] →", "72px", "Next — N")
        self.save_next_button=compact_button(
            "Save + Next [S]", "102px", "Save HEART_PRESENT mask and go next — S", "success"
        )
        self.raw_button=compact_button(
            "Image [I]", "74px", "Hide overlays and start from the image — I"
        )
        self.reset_button=compact_button(
            "Auto reset [O]", "94px", "Restore the automatic mask — O"
        )
        self.accept_button=compact_button(
            "Auto OK [A]", "84px", "Save confirmed auto mask for training — A", "info"
        )
        self.no_heart_button=compact_button(
            "No heart [H]", "92px", "Valid image, but no heart is visible — H", "info"
        )
        self.unusable_button=compact_button(
            "Unusable [U]", "92px", "Blur/noise/localizer: exclude from training — U", "warning"
        )
        self.skip_button=compact_button("Skip [K]", "66px", "Skip and go next — K")
        self.clear_button=compact_button("Clear [C]", "68px", "Clear editable mask — C")
        self.delete_button=compact_button(
            "Delete [D]", "76px", "Delete manual target and label — D", "danger"
        )
        self.brush_slider=widgets.IntSlider(
            description="Brush",
            value=self.brush_radius,
            min=1,
            max=30,
            step=1,
            continuous_update=False,
            layout=widgets.Layout(width="260px"),
        )
        self.status=widgets.HTML()

        self.previous_button.on_click(lambda _: self._safe("Previous", self.previous))
        self.next_button.on_click(lambda _: self._safe("Next", self.next))
        self.save_next_button.on_click(lambda _: self._safe("Save & Next", self.save_next))
        self.accept_button.on_click(lambda _: self._safe("Auto OK", self.accept_auto))
        self.no_heart_button.on_click(lambda _: self._safe("No heart", self.mark_no_heart))
        self.unusable_button.on_click(lambda _: self._safe("Unusable", self.mark_unusable))
        self.skip_button.on_click(lambda _: self._safe("Skip", self.skip))
        self.reset_button.on_click(lambda _: self._safe("Reset", self.reset_to_auto))
        self.raw_button.on_click(lambda _: self._safe("Reset to image", self.reset_to_image))
        self.clear_button.on_click(lambda _: self._safe("Clear", self.clear_editable))
        self.delete_button.on_click(lambda _: self._safe("Delete manual", self.delete_manual))
        self.brush_slider.observe(self._brush_changed, names="value")
        self.mask_sync.observe(self._mask_sync_changed, names="value")
        self.shortcut_sync.observe(self._shortcut_sync_changed, names="value")

        self.controls=widgets.VBox(
            [
                widgets.HBox(
                    [
                        self.previous_button,
                        self.next_button,
                        self.save_next_button,
                        self.raw_button,
                        self.reset_button,
                        self.accept_button,
                        self.no_heart_button,
                        self.unusable_button,
                        self.skip_button,
                        self.clear_button,
                        self.delete_button,
                    ],
                    layout=widgets.Layout(
                        flex_flow="row nowrap",
                        align_items="center",
                        overflow="auto hidden",
                        width="100%",
                    ),
                ),
                widgets.HBox([self.brush_slider, self.status]),
                self.mask_sync,
                self.shortcut_sync,
                self.output,
            ]
        )
        self.load_current()

    def _safe(self, action, function):
        try:
            function()
        except Exception as error:
            message=f"{action}: {type(error).__name__}: {error}"
            self.status.value=f"<span style='color:#b00020'><b>{message}</b></span>"
            print("[EDITOR ERROR]", message)

    def _advance_after_target(self, message):
        if not self.rows:
            return
        completed_index=int(self.index)
        completed_token=str(self.rows[completed_index].get("image_token", ""))
        if completed_index < len(self.rows) - 1:
            self.index=completed_index + 1
            self.load_current()
            self.update_status(
                f"{message} Image {completed_token} remains in the session queue; "
                "use Prev [P] to return to it."
            )
            return

        self.render()
        self.update_status(
            f"{message} End of queue; the image remains accessible, and "
            "Prev [P] returns to earlier images."
        )

    def _brush_changed(self, change):
        self.brush_radius=int(change["new"])
        if self.image is not None:
            self.render()

    @staticmethod
    def _image_uri(rgb):
        array=np.clip(np.round(rgb * 255.0), 0, 255).astype(np.uint8)
        ok, encoded=cv2.imencode(".png", cv2.cvtColor(array, cv2.COLOR_RGB2BGR))
        if not ok:
            raise RuntimeError("The editor image could not be encoded.")
        return "data:image/png;base64," + base64.b64encode(encoded.tobytes()).decode("ascii")

    @staticmethod
    def _mask_uri(mask):
        binary=(np.asarray(mask) > 0).astype(np.uint8)
        bgra=np.zeros((*binary.shape, 4), dtype=np.uint8)
        foreground=binary * 255
        bgra[..., :3]=foreground[..., None]
        bgra[..., 3]=foreground
        ok, encoded=cv2.imencode(".png", bgra)
        if not ok:
            raise RuntimeError("The editor mask could not be encoded.")
        return "data:image/png;base64," + base64.b64encode(encoded.tobytes()).decode("ascii")

    # Open the persisted automatic mask; before first training, use an empty guide.
    def load_current(self):
        row=self.rows[self.index]
        image=DataStage.standardized_uint8(row["image_path"])
        auto_mask=(
            AttentionStage.read_prediction_mask(row)
            if AttentionStage.has_prediction_mask(row)
            else np.zeros((256, 256), dtype=np.uint8)
        )
        manual_path=Path(row["manual_mask_path"])
        current=(
            DataStage.read_binary_mask(manual_path)
            if manual_path.is_file()
            else auto_mask.copy()
        )
        self.image=image.astype(np.float32) / 255.0
        self.auto_mask=auto_mask.astype(np.uint8)
        self.base_mask=self.auto_mask.copy()
        self.mask=current.astype(np.uint8)
        self.render()
        self.update_status()

    def render(self):
        row=self.rows[self.index]
        target_type=(
            ReviewStage.normalize_target_type(
                row.get("manual_annotation_type", row.get("target_type", ""))
            )
            or "UNLABELED"
        )
        image_uri=self._image_uri(np.stack([self.image] * 3, axis=-1))
        base_uri=self._mask_uri(self.base_mask)
        mask_uri=self._mask_uri(self.mask)
        canvas_id=f"cad_canvas_{self.widget_id}"
        keyboard_sink_id=f"cad_keyboard_sink_{self.widget_id}"
        placeholder=f"CAD_MASK_SYNC_{self.widget_id}"
        size=256
        html=f"""
        <div style="font-family:Arial,sans-serif;max-width:900px">
          <div style="margin-bottom:6px;font-size:14px">
            <b>{self.index + 1}/{len(self.rows)}</b> | {row['patient_id']} | {row['series_id']} |
            scope={self.review_scope} | target={target_type}
          </div>
          <input id="{keyboard_sink_id}" type="text" value="" tabindex="-1"
                 autocomplete="off" autocapitalize="off" spellcheck="false"
                 aria-label="CAD mask editor keyboard focus"
                 style="position:fixed;left:-10000px;top:0;width:1px;height:1px;
                        opacity:0;pointer-events:none;border:0;padding:0;margin:0" />
          <canvas id="{canvas_id}" width="{size * 3}" height="{size * 3}"
                  oncontextmenu="return false;"
                  style="width:768px;height:768px;max-width:100%;border:1px solid #999;
                         cursor:crosshair;touch-action:none;user-select:none;
                         -webkit-user-select:none;-webkit-touch-callout:none"></canvas>
          <div style="font-size:12px;margin-top:5px;line-height:1.4">
            Left drag=draw | Right drag=erase (Kaggle menu disabled) | S=Save & Next | P/N=Prev/Next |
            I=image | O=auto reset | A=Auto OK | H=no heart | U=unusable |
            K=Skip | C=Clear | D=Delete | [ / ]=Brush −/+
          </div>
        </div>
        """
        js=f"""
        (()=> {{
          const canvas=document.getElementById({json.dumps(canvas_id)});
          const keyboardSink=document.getElementById({json.dumps(keyboard_sink_id)});
          if (!canvas) return;
          const ctx=canvas.getContext('2d');
          const W={size}, H={size};
          const radius={int(self.brush_radius)};
          const placeholder={json.dumps(placeholder)};
          const image=new Image(), base=new Image(), initialMask=new Image();
          const maskCanvas=document.createElement('canvas');
          maskCanvas.width=W; maskCanvas.height=H;
          const maskCtx=maskCanvas.getContext('2d', {{willReadFrequently:true}});
          const pointerControllerKey='__cadMaskEditorPointerController';
          const oldPointerController=window[pointerControllerKey];
          if (oldPointerController && typeof oldPointerController.dispose === 'function') {{
            try {{ oldPointerController.dispose(); }} catch (_) {{}}
          }}

          let drawing=false, erase=false, last=null, rightDragActive=false;

          function focusKeyboardSink() {{
            if (!keyboardSink) return;
            try {{
              keyboardSink.focus({{preventScroll: true}});
            }} catch (_) {{
              try {{ keyboardSink.focus(); }} catch (_) {{}}
            }}
            keyboardSink.value='';
          }}

          function eventTargetsCanvas(event) {{
            if (event.target === canvas) return true;
            try {{
              const path=typeof event.composedPath === 'function'
                ? event.composedPath()
                : [];
              return path.includes(canvas);
            }} catch (_) {{
              return false;
            }}
          }}
          function blockEvent(event) {{
            if (!event) return false;
            if (typeof event.preventDefault === 'function') event.preventDefault();
            if (typeof event.stopPropagation === 'function') event.stopPropagation();
            if (typeof event.stopImmediatePropagation === 'function') {{
              event.stopImmediatePropagation();
            }}
            return false;
          }}
          function suppressContextMenu(event) {{
            // The window/capture listener runs before Kaggle's context menu.
            // Page behavior outside the canvas is unchanged.
            if (!rightDragActive && !eventTargetsCanvas(event)) return;
            return blockEvent(event);
          }}
          function releasePointerState() {{
            drawing=false;
            erase=false;
            rightDragActive=false;
            last=null;
          }}

          function hidden() {{
            return Array.from(document.querySelectorAll('textarea'))
              .find(x=> x.placeholder === placeholder);
          }}
          function overlay(source, color, alpha) {{
            const tmp=document.createElement('canvas');
            tmp.width=canvas.width; tmp.height=canvas.height;
            const c=tmp.getContext('2d');
            c.globalAlpha=alpha;
            c.drawImage(source, 0, 0, canvas.width, canvas.height);
            c.globalCompositeOperation='source-in';
            c.fillStyle=color;
            c.fillRect(0, 0, canvas.width, canvas.height);
            ctx.drawImage(tmp, 0, 0);
          }}
          function draw() {{
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.drawImage(image, 0, 0, canvas.width, canvas.height);
            overlay(base, 'rgb(255,165,0)', 0.35);
            overlay(maskCanvas, 'rgb(255,0,200)', 0.50);
          }}
          function sync() {{
            const target=hidden();
            if (!target) return;
            target.value=maskCanvas.toDataURL('image/png').split(',')[1];
            target.dispatchEvent(new Event('input', {{bubbles:true}}));
            target.dispatchEvent(new Event('change', {{bubbles:true}}));
          }}
          function point(e) {{
            const r=canvas.getBoundingClientRect();
            return {{
              x: Math.max(0, Math.min(W - 1, (e.clientX - r.left) / r.width * W)),
              y: Math.max(0, Math.min(H - 1, (e.clientY - r.top) / r.height * H))
            }};
          }}
          function paint(p) {{
            maskCtx.save();
            maskCtx.globalCompositeOperation=erase ? 'destination-out' : 'source-over';
            maskCtx.fillStyle='white'; maskCtx.strokeStyle='white';
            maskCtx.lineWidth=radius * 2; maskCtx.lineCap='round'; maskCtx.lineJoin='round';
            if (last) {{
              maskCtx.beginPath(); maskCtx.moveTo(last.x, last.y); maskCtx.lineTo(p.x, p.y); maskCtx.stroke();
            }} else {{
              maskCtx.beginPath(); maskCtx.arc(p.x, p.y, radius, 0, 2 * Math.PI); maskCtx.fill();
            }}
            maskCtx.restore(); last=p; draw();
          }}
          function begin(e) {{
            if (![0,2].includes(e.button)) return;
            focusKeyboardSink();
            const startsErase=e.button === 2;
            if (startsErase) blockEvent(e);
            else e.preventDefault();
            drawing=true;
            erase=startsErase;
            rightDragActive=startsErase;
            last=null;
            try {{ canvas.setPointerCapture?.(e.pointerId); }} catch (_) {{}}
            paint(point(e));
          }}
          function move(e) {{
            if (!drawing) return;
            const requiredButton=erase ? 2 : 1;
            if ((Number(e.buttons || 0) & requiredButton) === 0) return finish(e);
            if (erase) blockEvent(e);
            else e.preventDefault();
            paint(point(e));
          }}
          function finish(e) {{
            if (!drawing) {{
              if (e && e.button === 2) rightDragActive=false;
              return;
            }}
            const wasErase=erase;
            if (wasErase || (e && e.button === 2)) blockEvent(e);
            else e?.preventDefault?.();
            try {{ canvas.releasePointerCapture?.(e.pointerId); }} catch (_) {{}}
            drawing=false;
            erase=false;
            rightDragActive=false;
            last=null;
            sync();
            focusKeyboardSink();
          }}

          const activeOptions={{capture: true, passive: false}};
          // Local protection plus an early window listener blocks both the native
          // context menu and the menu installed by Kaggle/Jupyter.
          canvas.oncontextmenu=suppressContextMenu;
          canvas.addEventListener('contextmenu', suppressContextMenu, activeOptions);
          window.addEventListener('contextmenu', suppressContextMenu, activeOptions);
          canvas.addEventListener('auxclick', event=> {{
            if (event.button === 2) blockEvent(event);
          }}, activeOptions);
          canvas.addEventListener('dragstart', blockEvent, activeOptions);
          canvas.addEventListener('selectstart', blockEvent, activeOptions);

          // Pointer Events and Mouse Events are never installed together, avoiding
          // duplicate handling of a single press in Chrome/Kaggle.
          if (window.PointerEvent) {{
            canvas.addEventListener('pointerdown', begin, activeOptions);
            canvas.addEventListener('pointermove', move, activeOptions);
            canvas.addEventListener('pointerup', finish, activeOptions);
            canvas.addEventListener('pointercancel', finish, activeOptions);
          }} else {{
            canvas.addEventListener('mousedown', begin, activeOptions);
            canvas.addEventListener('mousemove', move, activeOptions);
            canvas.addEventListener('mouseup', finish, activeOptions);
            canvas.addEventListener('mouseleave', event=> {{
              if (drawing && Number(event.buttons || 0) === 0) finish(event);
            }}, activeOptions);
          }}

          window.addEventListener('blur', releasePointerState, true);
          document.addEventListener('visibilitychange', releasePointerState, true);
          window[pointerControllerKey]={{
            editorId: {json.dumps(self.widget_id)},
            dispose: ()=> {{
              window.removeEventListener('contextmenu', suppressContextMenu, true);
              window.removeEventListener('blur', releasePointerState, true);
              document.removeEventListener('visibilitychange', releasePointerState, true);
              releasePointerState();
            }}
          }};
          function load(target, source) {{
            return new Promise((resolve, reject)=> {{ target.onload=resolve; target.onerror=reject; target.src=source; }});
          }}
          // Focusing an invisible input places the notebook in typing mode before the
          // first shortcut and prevents Kaggle command mode from interpreting
          // O/P/N/etc. as notebook-cell operations.
          focusKeyboardSink();
          requestAnimationFrame(focusKeyboardSink);
          Promise.all([
            load(image, {json.dumps(image_uri)}),
            load(base, {json.dumps(base_uri)}),
            load(initialMask, {json.dumps(mask_uri)})
          ]).then(()=> {{
            maskCtx.clearRect(0,0,W,H);
            maskCtx.drawImage(initialMask,0,0,W,H);
            draw();
            focusKeyboardSink();
            requestAnimationFrame(focusKeyboardSink);
          }});
        }})();
        """
        with self.output:
            clear_output(wait=True)
            display(HTML(html))
            display(Javascript(js))

    def _shortcut_script(self):
        placeholder=f"CAD_SHORTCUT_SYNC_{self.widget_id}"
        keyboard_sink_id=f"cad_keyboard_sink_{self.widget_id}"
        controller_key="__cadMaskEditorKeyboardController"
        return f"""
        (()=> {{
          const controllerKey={json.dumps(controller_key)};
          const editorId={json.dumps(self.widget_id)};
          const placeholder={json.dumps(placeholder)};
          const keyboardSinkId={json.dumps(keyboard_sink_id)};

          const oldController=window[controllerKey];
          if (oldController && typeof oldController.dispose === 'function') {{
            try {{ oldController.dispose(); }} catch (_) {{}}
          }}

          const held=new Set();
          const lastFire=new Map();
          let serial=0;
          const minimumGapMs=220;
          const keyOptions={{capture: true, passive: false}};

          function commandTarget() {{
            return Array.from(document.querySelectorAll('textarea'))
              .find(node=> node.placeholder === placeholder) || null;
          }}

          function keyboardSink() {{
            return document.getElementById(keyboardSinkId) || null;
          }}

          function focusKeyboardSink() {{
            const sink=keyboardSink();
            if (!sink) return;
            try {{
              sink.focus({{preventScroll: true}});
            }} catch (_) {{
              try {{ sink.focus(); }} catch (_) {{}}
            }}
            sink.value='';
          }}

          function isEditorSink(target) {{
            return Boolean(target && String(target.id || '') === keyboardSinkId);
          }}

          function isTypingTarget(target) {{
            if (!target || isEditorSink(target)) return false;
            const tag=String(target.tagName || '').toLowerCase();
            return tag === 'input' || tag === 'textarea' || tag === 'select' ||
                   target.isContentEditable === true;
          }}

          function normalizeCode(event) {{
            const code=String(event.code || '');
            if (code) return code;
            const key=String(event.key || '').toLowerCase();
            const fallback={{
              p: 'KeyP', n: 'KeyN', s: 'KeyS', i: 'KeyI', o: 'KeyO',
              // R no longer resets the mask; it is retained only to block the legacy
              // Kaggle shortcut when pressed out of habit.
              r: 'KeyR',
              a: 'KeyA', h: 'KeyH', u: 'KeyU', k: 'KeyK', c: 'KeyC', d: 'KeyD',
              '[': 'BracketLeft', ']': 'BracketRight'
            }};
            return fallback[key] || '';
          }}

          const actions={{
            KeyP: 'previous',
            KeyN: 'next',
            KeyS: 'save_next',
            KeyI: 'reset_image',
            KeyO: 'reset_auto',
            KeyA: 'accept_auto',
            KeyH: 'no_heart',
            KeyU: 'unusable',
            KeyK: 'skip',
            KeyC: 'clear',
            KeyD: 'delete',
            BracketLeft: 'brush_down',
            BracketRight: 'brush_up',
          }};
          const legacyBlockedCodes=new Set(['KeyR']);

          function blockEvent(event) {{
            if (!event) return;
            if (typeof event.preventDefault === 'function') event.preventDefault();
            if (typeof event.stopPropagation === 'function') event.stopPropagation();
            if (typeof event.stopImmediatePropagation === 'function') {{
              event.stopImmediatePropagation();
            }}
          }}

          function shortcutInfo(event) {{
            const code=normalizeCode(event);
            const action=actions[code] || '';
            const legacyBlocked=legacyBlockedCodes.has(code);
            if ((!action && !legacyBlocked) || event.ctrlKey || event.metaKey ||
                event.altKey || isTypingTarget(event.target)) return null;
            return {{code, action, legacyBlocked}};
          }}

          function onKeyDown(event) {{
            const info=shortcutInfo(event);
            if (!info) return;

            // The event is stopped before any message reaches Python. Therefore O
            // (Kaggle's output shortcut) remains exclusive to the editor, and
            // legacy R cannot modify or create notebook cells.
            blockEvent(event);
            focusKeyboardSink();
            if (info.legacyBlocked) {{
              held.delete(info.code);
              return;
            }}
            if (event.repeat || held.has(info.code)) return;

            const now=performance.now();
            const previous=lastFire.get(info.code) || -Infinity;
            if (now - previous < minimumGapMs) return;

            const target=commandTarget();
            if (!target) return;
            held.add(info.code);
            lastFire.set(info.code, now);
            serial +=1;
            target.value=`${{info.action}}|${{Date.now()}}|${{serial}}`;
            target.dispatchEvent(new Event('input', {{bubbles: true}}));
            target.dispatchEvent(new Event('change', {{bubbles: true}}));
          }}

          function onKeyPress(event) {{
            const info=shortcutInfo(event);
            if (info) blockEvent(event);
          }}

          function onKeyUp(event) {{
            const info=shortcutInfo(event);
            if (!info) return;
            blockEvent(event);
            held.delete(info.code);
            focusKeyboardSink();
          }}

          function releaseAll() {{ held.clear(); }}

          // Window/capture fires before document/capture. Together with the focused
          // input, it isolates editor shortcuts from notebook command mode and
          // prevents accidental cell creation or modification.
          window.addEventListener('keydown', onKeyDown, keyOptions);
          window.addEventListener('keypress', onKeyPress, keyOptions);
          window.addEventListener('keyup', onKeyUp, keyOptions);
          window.addEventListener('blur', releaseAll, true);
          document.addEventListener('visibilitychange', releaseAll, true);

          window[controllerKey]={{
            editorId,
            dispose: ()=> {{
              window.removeEventListener('keydown', onKeyDown, true);
              window.removeEventListener('keypress', onKeyPress, true);
              window.removeEventListener('keyup', onKeyUp, true);
              window.removeEventListener('blur', releaseAll, true);
              document.removeEventListener('visibilitychange', releaseAll, true);
              const sink=keyboardSink();
              if (sink && document.activeElement === sink) {{
                try {{ sink.blur(); }} catch (_) {{}}
              }}
              held.clear();
            }}
          }};

          focusKeyboardSink();
          requestAnimationFrame(focusKeyboardSink);
          setTimeout(focusKeyboardSink, 0);
        }})();
        """

    def _shortcut_sync_changed(self, change):
        payload=str(change.get("new", "") or "").strip()
        if not self.rows:
            return
        if not payload or payload == self._last_shortcut_payload:
            return
        self._last_shortcut_payload=payload
        action=payload.split("|", 1)[0].strip().lower()

        now=time.monotonic()
        if self._shortcut_busy or now - self._last_shortcut_at < 0.12:
            return

        actions={
            "previous": ("Previous", self.previous),
            "next": ("Next", self.next),
            "save_next": ("Save & Next", self.save_next),
            "reset_image": ("Reset to image", self.reset_to_image),
            "reset_auto": ("Reset to auto", self.reset_to_auto),
            "accept_auto": ("Auto OK", self.accept_auto),
            "no_heart": ("No heart", self.mark_no_heart),
            "unusable": ("Unusable", self.mark_unusable),
            "skip": ("Skip", self.skip),
            "clear": ("Clear", self.clear_editable),
            "delete": ("Delete manual", self.delete_manual),
            "brush_down": (
                "Brush -",
                lambda: setattr(
                    self.brush_slider,
                    "value",
                    max(self.brush_slider.min, self.brush_slider.value - self.brush_slider.step),
                ),
            ),
            "brush_up": (
                "Brush +",
                lambda: setattr(
                    self.brush_slider,
                    "value",
                    min(self.brush_slider.max, self.brush_slider.value + self.brush_slider.step),
                ),
            ),
        }
        selected=actions.get(action)
        if selected is None:
            return

        self._shortcut_busy=True
        self._last_shortcut_at=now
        try:
            label, function=selected
            self._safe(label, function)
        finally:
            self._shortcut_busy=False

    def _mask_sync_changed(self, change):
        value=change.get("new", "")
        if not value:
            return
        try:
            decoded=cv2.imdecode(
                np.frombuffer(base64.b64decode(value), np.uint8), cv2.IMREAD_UNCHANGED
            )
            if decoded is None:
                return
            if decoded.ndim == 3 and decoded.shape[2] == 4:
                alpha=decoded[..., 3]
                gray=cv2.cvtColor(decoded[..., :3], cv2.COLOR_BGR2GRAY)
                binary=((alpha > 8) & (gray > 127)).astype(np.uint8)
            elif decoded.ndim == 3:
                binary=(cv2.cvtColor(decoded, cv2.COLOR_BGR2GRAY) > 127).astype(np.uint8)
            else:
                binary=(decoded > 127).astype(np.uint8)
            self.mask=cv2.resize(
                binary,
                (256, 256),
                interpolation=cv2.INTER_NEAREST,
            ).astype(np.uint8)
        except Exception as error:
            print("[EDITOR WARNING] Mask synchronization failed:", error)

    def _sync_python_mask(self):
        value=self._mask_uri(self.mask).split(",", 1)[1]
        if self.mask_sync.value != value:
            self.mask_sync.value=value

    def update_status(self, prefix=""):
        if not self.rows:
            self.status.value=(
                "<span style='margin-left:12px'><b>Review complete.</b> "
                "No target=UNLABELED images remain.</span>"
            )
            return
        row=self.rows[self.index]
        manual=DataStage.manual_mask_qc(row["manual_mask_path"])
        annotation=DataStage.load_annotations(self.workspace).get(
            str(row["image_token"]), {}
        )
        target_type=(
            ReviewStage.normalize_target_type(annotation.get("target_type", ""))
            or "UNLABELED"
        )
        self.status.value=(
            "<span style='margin-left:12px'>"
            f"<b>{prefix}</b> index={self.index + 1}/{len(self.rows)}; "
            f"target={target_type}; manual={'yes' if manual['exists'] else 'no'}; "
            f"usable={manual['usable']}; attention_valid={row.get('attention_valid_final', '')}; "
            f"presence={DataStage._as_float(row.get('attention_presence_probability'), np.nan):.3f}; "
            f"uncertainty={DataStage._as_float(row.get('attention_uncertainty_score'), np.nan):.3f}; "
            f"reason={row.get('attention_invalid_reason_final', '')}</span>"
        )

    # Save the corrected target, an inspection overlay, and the persistent review action.
    def _write_target(
        self,
        mask,
        target_type,
        source,
        action,
        sample_weight=None,
    ):
        row=self.rows[self.index]
        binary=(np.asarray(mask) > 0).astype(np.uint8)
        path=Path(row["manual_mask_path"])
        DataStage.write_png(path, binary * 255)
        DataStage.set_annotation(
            self.workspace,
            row,
            target_type=target_type,
            source=source,
            sample_weight=sample_weight,
        )
        row["manual_annotation_type"]=target_type
        row["manual_annotation_source"]=source

        overlay=np.stack([self.image] * 3, axis=-1)
        overlay[..., 0]=np.maximum(overlay[..., 0], binary * 0.90)
        overlay[..., 1] *=1.0 - 0.45 * binary
        overlay[..., 2] *=1.0 - 0.45 * binary
        DataStage.write_png(
            self.workspace.mask_overlays / f"{row['image_token']}.png",
            cv2.cvtColor(
                np.clip(np.round(overlay * 255.0), 0, 255).astype(np.uint8),
                cv2.COLOR_RGB2BGR,
            ),
        )
        ReviewStage.log(
            self.workspace,
            row,
            self.review_round,
            action,
            binary,
            self.index,
            len(self.rows),
        )
        self.mask=binary.copy()

    def save(self):
        if float((self.mask > 0).mean()) < 0.0005:
            raise ValueError(
                "The mask is empty. Use No heart [H] for a valid image "
                "without a visible heart, or Unusable [U] for blur/noise/localizer frames."
            )
        self._write_target(
            self.mask,
            HEART_PRESENT,
            source="human_drawn_or_corrected",
            action="save_manual_heart_present",
            sample_weight=1.0,
        )
        self.update_status("Saved HEART_PRESENT; weight=1.00.")
        print("[EDITOR]", self.rows[self.index]["manual_mask_path"], "| HEART_PRESENT")

    def save_next(self):
        self.save()
        self._advance_after_target("Saved HEART_PRESENT.")

    def accept_auto(self):
        if float((self.auto_mask > 0).mean()) < 0.0005:
            raise ValueError("The automatic mask is empty; use No heart [H], not Auto OK.")
        self._write_target(
            self.auto_mask,
            HEART_PRESENT,
            source="human_confirmed_auto",
            action="accept_auto_and_save",
            sample_weight=0.72,
        )
        self._advance_after_target("Auto saved HEART_PRESENT; weight=0.72.")

    def mark_no_heart(self):
        empty=np.zeros_like(self.auto_mask, dtype=np.uint8)
        self._write_target(
            empty,
            NO_HEART_VISIBLE,
            source="human_no_heart_visible",
            action="mark_no_heart_visible",
            sample_weight=0.90,
        )
        self._advance_after_target("Saved NO_HEART_VISIBLE negative target.")

    def mark_unusable(self):
        row=self.rows[self.index]
        Path(row["manual_mask_path"]).unlink(missing_ok=True)
        DataStage.set_annotation(
            self.workspace,
            row,
            target_type=UNUSABLE,
            source="human_unusable",
            sample_weight=0.0,
        )
        row["manual_annotation_type"]=UNUSABLE
        ReviewStage.log(
            self.workspace,
            row,
            self.review_round,
            "mark_unusable",
            None,
            self.index,
            len(self.rows),
        )
        self._advance_after_target("Marked UNUSABLE; excluded from training.")

    # Record a deliberate skip so reopening the same review round does not repeat it.
    def skip(self):
        row=self.rows[self.index]
        ReviewStage.log(
            self.workspace,
            row,
            self.review_round,
            "skip",
            None,
            self.index,
            len(self.rows),
        )
        self.next()

    def reset_to_auto(self):
        self.base_mask=self.auto_mask.copy()
        self.mask=self.auto_mask.copy()
        self._sync_python_mask()
        self.render()
        self.update_status("Automatic mask restored.")

    def reset_to_image(self):
        self.base_mask=np.zeros_like(self.auto_mask)
        self.mask=np.zeros_like(self.auto_mask)
        self._sync_python_mask()
        self.render()
        self.update_status("All overlays hidden; not saved yet.")

    def clear_editable(self):
        self.base_mask=self.auto_mask.copy()
        self.mask=np.zeros_like(self.auto_mask)
        self._sync_python_mask()
        self.render()
        self.update_status("Editable mask cleared; auto remains as guide.")

    def delete_manual(self):
        row=self.rows[self.index]
        Path(row["manual_mask_path"]).unlink(missing_ok=True)
        DataStage.remove_annotation(self.workspace, str(row["image_token"]))
        row["manual_annotation_type"]=""
        row["manual_annotation_source"]=""
        ReviewStage.log(
            self.workspace,
            row,
            self.review_round,
            "delete_manual_and_annotation",
            None,
            self.index,
            len(self.rows),
        )
        self.mask=self.auto_mask.copy()
        self.base_mask=self.auto_mask.copy()
        self._sync_python_mask()
        self.render()
        self.update_status("Manual target and annotation deleted.")

    def next(self):
        if not self.rows:
            return
        if self.index < len(self.rows) - 1:
            self.index +=1
            self.load_current()
        else:
            self.update_status("End of queue.")

    def previous(self):
        if not self.rows:
            return
        if self.index > 0:
            self.index -=1
            self.load_current()

    def show(self):
        display(self.controls)
        display(Javascript(self._shortcut_script()))
        return self

# =============================================================================
# STAGE 4 — SICK/NORMAL MATCHING
# =============================================================================
class MatchingStage:
    """Rebuild strict-core and expanded balanced matching from current-run data."""

    # Convert slice position to a normalized within-series coordinate for matching.
    @staticmethod
    def sequence_position(row):
        length=max(1, DataStage._as_int(row.get("sequence_length"), 1))
        index=int(np.clip(DataStage._as_int(row.get("sequence_index"), 0), 0, length - 1))
        if length <= 1:
            return 0.5
        return float(index / (length - 1))
    # Convert the perceptual hash string to bits used by the matching descriptor.
    @staticmethod
    def phash_bits(value):
        number=int(str(value), 16)
        return np.asarray(
            [(number >> shift) & 1 for shift in range(63, -1, -1)],
            dtype=np.float32,
        )
    # Median/MAD standardize matching variables so one descriptor block cannot dominate.
    @staticmethod
    def robust_standardize(values):
        values=np.asarray(values, dtype=np.float32).copy()
        if values.ndim == 1:
            values=values[:, None]
        for column in range(values.shape[1]):
            current=values[:, column]
            finite=np.isfinite(current)
            center=float(np.median(current[finite])) if finite.any() else 0.0
            current[~finite]=center
            mad=float(np.median(np.abs(current - center)))
            scale=max(1e-3, 1.4826 * mad)
            values[:, column]=np.clip((current - center) / scale, -5.0, 5.0)
        return values
    # Build acquisition/anatomy/quality descriptors without using the CAD label as a feature.
    @staticmethod
    def matching_descriptor(rows):
        phash=np.stack(
            [MatchingStage.phash_bits(row["perceptual_hash"]) for row in rows]
        )
        phash=phash * 2.0 - 1.0
        geometry=MatchingStage.robust_standardize(
            np.asarray(
                [
                    [
                        DataStage._as_float(row.get("attention_area_ratio"), np.nan),
                        DataStage._as_float(row.get("attention_centroid_x"), np.nan),
                        DataStage._as_float(row.get("attention_centroid_y"), np.nan),
                        DataStage._as_float(row.get("attention_boundary_touch_fraction"), np.nan),
                    ]
                    for row in rows
                ],
                dtype=np.float32,
            )
        )
        sequence=MatchingStage.robust_standardize(
            np.asarray(
                [[MatchingStage.sequence_position(row)] for row in rows],
                dtype=np.float32,
            )
        )
        quality=MatchingStage.robust_standardize(
            np.asarray(
                [
                    [
                        math.log1p(max(0.0, DataStage._as_float(row.get("sharpness"), 0.0))),
                        DataStage._as_float(row.get("noise_ratio"), np.nan),
                        DataStage._as_float(row.get("dynamic_range"), np.nan),
                    ]
                    for row in rows
                ],
                dtype=np.float32,
            )
        )

        def block_scale(weight, dimensions):
            return math.sqrt(max(float(weight), 0.0) / max(1, int(dimensions)))

        descriptor=np.concatenate(
            [
                phash * block_scale(0.45, phash.shape[1]),
                geometry
                * block_scale(0.25, geometry.shape[1]),
                sequence
                * block_scale(0.15, sequence.shape[1]),
                quality
                * block_scale(0.15, quality.shape[1]),
            ],
            axis=1,
        )
        return np.ascontiguousarray(descriptor, dtype=np.float32)
    # Require the same quality/geometry gates and a current in-memory automatic mask.
    @staticmethod
    def matching_eligibility_reason(row):
        reasons=[]
        if DataStage._as_int(row.get("quality_valid"), 0) != 1:
            reasons.append("quality_invalid")
        if DataStage._as_int(row.get("attention_valid_final"), 0) != 1:
            reasons.append("attention_invalid")
        if DataStage._as_int(row.get("attention_heart_present"), 1) != 1:
            reasons.append("heart_not_visible")
        if (
            DataStage._as_float(row.get("attention_area_ratio"), 0.0)
            < 0.003):
            reasons.append("attention_area_too_small")
        if not str(row.get("perceptual_hash", "")).strip():
            reasons.append("missing_phash")
        if not AttentionStage.has_prediction_mask(row):
            reasons.append("missing_attention_mask")
        return ";".join(reasons)
    # Choose acquisition-family granularity from cohort size while respecting class coverage.
    @staticmethod
    def choose_family_count(number_of_images):
        estimate=int(
            math.ceil(
                number_of_images
                / max(1, 1800)
            )
        )
        count=int(
            np.clip(
                estimate,
                8,
                32,
            )
        )
        return max(2, min(count, number_of_images))
    # Require each acquisition family to contain enough Sick and Normal patients/slices.
    @staticmethod
    def family_is_shared(rows):
        for label in (0, 1):
            class_rows=[row for row in rows if DataStage._as_int(row.get("label"), -1) == label]
            if len(class_rows) < 12:
                return False
            if (
                len({str(row.get("patient_id", "")) for row in class_rows})
                < 2):
                return False
        return True
    # Derive a distance cutoff from robust neighbour-distance statistics.
    @staticmethod
    def robust_caliper(
        distances,
        mad_multiplier,
        quantile,
    ):
        values=np.asarray(list(distances), dtype=np.float64)
        values=values[np.isfinite(values)]
        if not len(values):
            return np.nan
        median=float(np.median(values))
        mad=float(np.median(np.abs(values - median)))
        robust_scale=max(1e-9, 1.4826 * mad)
        mad_caliper=median + float(mad_multiplier) * robust_scale
        quantile_caliper=float(np.quantile(values, float(quantile)))
        return float(max(median, min(mad_caliper, quantile_caliper)))
    # Both core and extended matching use the same anatomical limits.
    @staticmethod
    def passes_anatomical_gates(
        sick_row,
        normal_row,
    ):
        sequence_limit=0.25
        area_limit=0.20
        if (
            abs(
                MatchingStage.sequence_position(sick_row)
                - MatchingStage.sequence_position(normal_row)
            )
            > sequence_limit):
            return False
        sick_area=DataStage._as_float(sick_row.get("attention_area_ratio"), np.nan)
        normal_area=DataStage._as_float(normal_row.get("attention_area_ratio"), np.nan)
        if (
            np.isfinite(sick_area)
            and np.isfinite(normal_area)
            and abs(sick_area - normal_area) > area_limit):
            return False
        return True
    # Return stable patient/series identifiers used by matching capacity limits.
    @staticmethod
    def pair_keys(
        sick_row, normal_row, family
    ):
        sick_patient=str(sick_row["patient_id"])
        normal_patient=str(normal_row["patient_id"])
        sick_sequence=str(
            sick_row.get("sequence_group_id") or sick_row.get("series_id", "")
        )
        normal_sequence=str(
            normal_row.get("sequence_group_id") or normal_row.get("series_id", "")
        )
        return (
            (sick_patient, family),
            (normal_patient, family),
            (sick_sequence, family),
            (normal_sequence, family),
            (sick_patient, normal_patient),
            (sick_sequence, normal_sequence),
        )
    # Rebuild the strict core and expanded balanced cohort from current RAM records.
    @staticmethod
    def build_cross_class_matching(dataset_rows):
        # Matching is intentionally rebuilt every run. It uses only the current OOF
        # masks, quality measurements, labels, and dataset geometry.
        manifest_by_token={}
        eligible_rows=[]
        for row in dataset_rows:
            token=str(row["image_token"])
            reason=MatchingStage.matching_eligibility_reason(row)
            record={
                "image_token": token,
                "image_path": str(row.get("image_path", "")),
                "patient_id": str(row.get("patient_id", "")),
                "series_id": str(row.get("series_id", "")),
                "sequence_group_id": str(
                    row.get("sequence_group_id") or row.get("series_id", "")
                ),
                "label": DataStage._as_int(row.get("label"), -1),
                "class_name": "Sick" if DataStage._as_int(row.get("label"), -1) == 1 else "Normal",
                "eligible_for_matching": int(not reason),
                "acquisition_family": "",
                "shared_family": 0,
                "selected_for_core_matched_cohort": 0,
                "selected_for_matched_cohort": 0,
                "match_stage": "",
                "pair_id": "",
                "matched_partner_token": "",
                "matched_partner_patient": "",
                "matched_partner_series": "",
                "match_distance": "",
                "mutual_rank_from_sick": "",
                "mutual_rank_from_normal": "",
                "reciprocal_extended_neighbor": "",
                "sequence_pair_seeded_by_core": "",
                "family_core_caliper": "",
                "family_extended_caliper": "",
                "family_caliper": "",
                "sequence_position": MatchingStage.sequence_position(row),
                "attention_area_ratio": DataStage._as_float(
                    row.get("attention_area_ratio"), np.nan
                ),
                "exclusion_reason": reason,
            }
            manifest_by_token[token]=record
            if not reason:
                eligible_rows.append(row)

        labels=np.asarray(
            [DataStage._as_int(row.get("label"), -1) for row in eligible_rows], dtype=np.int64
        )
        if len(eligible_rows) < 4 or set(labels.tolist()) != {0, 1}:
            raise RuntimeError(
                "Matching requires eligible images from both classes; "
                f"remaining Normal={int(np.sum(labels == 0))}, Sick={int(np.sum(labels == 1))}."
            )

        descriptor=MatchingStage.matching_descriptor(eligible_rows)
        family_count=MatchingStage.choose_family_count(len(eligible_rows))
        batch_size=min(4096, max(256, len(eligible_rows) // 10))
        clusterer=MiniBatchKMeans(
            n_clusters=family_count,
            random_state=42,
            batch_size=batch_size,
            n_init=5,
            max_iter=200,
            reassignment_ratio=0.01,
        )
        family_labels=clusterer.fit_predict(descriptor).astype(np.int64)

        family_indices=defaultdict(list)
        for index, family in enumerate(family_labels.tolist()):
            family_indices[int(family)].append(index)
            token=str(eligible_rows[index]["image_token"])
            manifest_by_token[token]["acquisition_family"]=int(family)

        shared_families=set()
        for family, indices in family_indices.items():
            family_rows=[eligible_rows[index] for index in indices]
            if MatchingStage.family_is_shared(family_rows):
                shared_families.add(int(family))
                for index in indices:
                    manifest_by_token[str(eligible_rows[index]["image_token"])][
                        "shared_family"
                    ]=1
            else:
                for index in indices:
                    manifest_by_token[str(eligible_rows[index]["image_token"])][
                        "exclusion_reason"
                    ]="family_not_shared_between_classes"

        if not shared_families:
            raise RuntimeError(
                "No acquisition family contains enough images and patients from both classes."
            )

        normal_count=int(np.sum(labels == 0))
        sick_count=int(np.sum(labels == 1))
        maximum_possible_pairs=min(normal_count, sick_count)
        requested_target_pairs=int(
            round(
                len(eligible_rows)
                * float(0.50)
                / 2.0
            )
        )
        requested_target_pairs=max(1, min(requested_target_pairs, maximum_possible_pairs))

        patients_by_label={
            label: sorted(
                {
                    str(row["patient_id"])
                    for row in eligible_rows
                    if DataStage._as_int(row.get("label"), -1) == label
                }
            )
            for label in (0, 1)
        }
        patient_total_caps={
            label: max(
                60,
                int(
                    math.ceil(
                        requested_target_pairs
                        / max(1, len(patients_by_label[label]))
                        * 1.70
                    )
                ),
            )
            for label in (0, 1)
        }

        used_tokens=set()
        patient_family_counts=defaultdict(int)
        sequence_counts=defaultdict(int)
        patient_total_counts=defaultdict(int)
        patient_pair_counts=defaultdict(int)
        sequence_pair_counts=defaultdict(int)
        core_sequence_pairs=set()
        selected_pair_distances=[]
        core_pair_distances=[]
        extended_pair_distances=[]
        family_summaries=[]
        pair_number=0
        core_pair_count=0
        extended_pair_count=0

        family_data={}
        token_candidate_stage={}

        # Build candidate graphs first. The extended graph is the union of the
        # top-K neighbours from both directions; the strict graph is its mutual
        # top-5 subset.
        for family in sorted(shared_families):
            indices=family_indices[family]
            normal_positions=[
                index for index in indices if DataStage._as_int(eligible_rows[index].get("label"), -1) == 0
            ]
            sick_positions=[
                index for index in indices if DataStage._as_int(eligible_rows[index].get("label"), -1) == 1
            ]
            normal_descriptor=descriptor[normal_positions]
            sick_descriptor=descriptor[sick_positions]
            extended_sick_to_normal=min(
                int(12), len(normal_positions)
            )
            extended_normal_to_sick=min(
                int(12), len(sick_positions)
            )

            normal_model=NearestNeighbors(
                n_neighbors=extended_sick_to_normal,
                metric="euclidean",
                algorithm="auto",
                n_jobs=-1,
            ).fit(normal_descriptor)
            sick_to_normal_index=normal_model.kneighbors(
                sick_descriptor, return_distance=False
            )
            sick_model=NearestNeighbors(
                n_neighbors=extended_normal_to_sick,
                metric="euclidean",
                algorithm="auto",
                n_jobs=-1,
            ).fit(sick_descriptor)
            normal_to_sick_index=sick_model.kneighbors(
                normal_descriptor, return_distance=False
            )

            sick_rank_maps=[
                {int(normal_local): int(rank + 1) for rank, normal_local in enumerate(neighbours)}
                for neighbours in sick_to_normal_index
            ]
            normal_rank_maps=[
                {int(sick_local): int(rank + 1) for rank, sick_local in enumerate(neighbours)}
                for neighbours in normal_to_sick_index
            ]
            edge_keys=set()
            for sick_local, neighbours in enumerate(sick_to_normal_index):
                edge_keys.update((int(sick_local), int(value)) for value in neighbours)
            for normal_local, neighbours in enumerate(normal_to_sick_index):
                edge_keys.update((int(value), int(normal_local)) for value in neighbours)

            all_edges=[]
            core_edges=[]
            for sick_local, normal_local in edge_keys:
                sick_global=sick_positions[sick_local]
                normal_global=normal_positions[normal_local]
                sick_row=eligible_rows[sick_global]
                normal_row=eligible_rows[normal_global]
                if not MatchingStage.passes_anatomical_gates(
                    sick_row, normal_row):
                    continue
                rank_from_sick=sick_rank_maps[sick_local].get(normal_local)
                rank_from_normal=normal_rank_maps[normal_local].get(sick_local)
                distance=float(
                    np.linalg.norm(
                        descriptor[sick_global] - descriptor[normal_global]
                    )
                )
                reciprocal=rank_from_sick is not None and rank_from_normal is not None
                is_core=bool(
                    reciprocal
                    and rank_from_sick <= 5
                    and rank_from_normal <= 5
                )
                sick_token=str(sick_row["image_token"])
                normal_token=str(normal_row["image_token"])
                token_candidate_stage.setdefault(sick_token, "extended_candidate")
                token_candidate_stage.setdefault(normal_token, "extended_candidate")
                if is_core:
                    token_candidate_stage[sick_token]="core_candidate"
                    token_candidate_stage[normal_token]="core_candidate"
                edge={
                    "family": int(family),
                    "distance": distance,
                    "sick_global": int(sick_global),
                    "normal_global": int(normal_global),
                    "rank_from_sick": rank_from_sick,
                    "rank_from_normal": rank_from_normal,
                    "reciprocal": int(reciprocal),
                    "is_core": int(is_core),
                    "tie": hashlib.sha256(
                        f"{family}|{sick_token}|{normal_token}".encode("utf-8")
                    ).hexdigest(),
                }
                all_edges.append(edge)
                if is_core:
                    core_edges.append(edge)

            core_caliper=MatchingStage.robust_caliper(
                [edge["distance"] for edge in core_edges],
                2.5,
                0.90,
            )
            extended_caliper=MatchingStage.robust_caliper(
                [edge["distance"] for edge in all_edges],
                3.5,
                0.97,
            )
            if np.isfinite(core_caliper):
                maximum_extended=(
                    float(core_caliper)
                    * float(1.18)
                )
                if np.isfinite(extended_caliper):
                    extended_caliper=max(
                        float(core_caliper),
                        min(float(extended_caliper), maximum_extended),
                    )
                else:
                    extended_caliper=maximum_extended

            family_data[family]={
                "normal_positions": normal_positions,
                "sick_positions": sick_positions,
                "core_edges": core_edges,
                "all_edges": all_edges,
                "core_caliper": core_caliper,
                "extended_caliper": extended_caliper,
                "core_selected": 0,
                "extended_selected": 0,
            }
            for index in indices:
                record=manifest_by_token[str(eligible_rows[index]["image_token"])]
                if np.isfinite(core_caliper):
                    record["family_core_caliper"]=float(core_caliper)
                if np.isfinite(extended_caliper):
                    record["family_extended_caliper"]=float(extended_caliper)

        def can_select(
            edge,
            stage,
            patient_quota=None,
        ):
            family=int(edge["family"])
            sick_row=eligible_rows[int(edge["sick_global"])]
            normal_row=eligible_rows[int(edge["normal_global"])]
            sick_token=str(sick_row["image_token"])
            normal_token=str(normal_row["image_token"])
            if sick_token in used_tokens or normal_token in used_tokens:
                return False
            (
                sick_patient_key,
                normal_patient_key,
                sick_sequence_key,
                normal_sequence_key,
                patient_pair_key,
                sequence_pair_key,
            )=MatchingStage.pair_keys(sick_row, normal_row, family)
            if stage == "core_mutual":
                patient_family_limit=20
                sequence_limit=5
            else:
                patient_family_limit=60
                sequence_limit=15
            if (
                patient_family_counts[sick_patient_key] >= patient_family_limit
                or patient_family_counts[normal_patient_key] >= patient_family_limit
                or sequence_counts[sick_sequence_key] >= sequence_limit
                or sequence_counts[normal_sequence_key] >= sequence_limit):
                return False
            if stage != "core_mutual":
                if (
                    patient_pair_counts[patient_pair_key]
                    >= 60
                    or sequence_pair_counts[sequence_pair_key]
                    >= 12):
                    return False
                for row in (sick_row, normal_row):
                    label=DataStage._as_int(row.get("label"), -1)
                    patient=str(row["patient_id"])
                    cap=int(patient_total_caps[label])
                    if patient_total_counts[patient] >= cap:
                        return False
                    if patient_quota is not None and patient_total_counts[patient] >= int(
                        patient_quota[label]):
                        return False
            return True

        def select_edge(edge, stage):
            nonlocal pair_number, core_pair_count, extended_pair_count
            family=int(edge["family"])
            sick_row=eligible_rows[int(edge["sick_global"])]
            normal_row=eligible_rows[int(edge["normal_global"])]
            sick_token=str(sick_row["image_token"])
            normal_token=str(normal_row["image_token"])
            (
                sick_patient_key,
                normal_patient_key,
                sick_sequence_key,
                normal_sequence_key,
                patient_pair_key,
                sequence_pair_key,
            )=MatchingStage.pair_keys(sick_row, normal_row, family)
            seeded=int(sequence_pair_key in core_sequence_pairs)
            pair_number +=1
            prefix="CORE" if stage == "core_mutual" else "EXT"
            pair_id=f"CCM_{prefix}_F{family:02d}_P{pair_number:06d}"
            applied_caliper=(
                family_data[family]["core_caliper"]
                if stage == "core_mutual"
                else family_data[family]["extended_caliper"]
            )
            for source_row, partner_row in (
                (sick_row, normal_row),
                (normal_row, sick_row),):
                source_token=str(source_row["image_token"])
                record=manifest_by_token[source_token]
                record.update(
                    {
                        "selected_for_core_matched_cohort": int(stage == "core_mutual"),
                        "selected_for_matched_cohort": 1,
                        "match_stage": stage,
                        "pair_id": pair_id,
                        "matched_partner_token": str(partner_row["image_token"]),
                        "matched_partner_patient": str(partner_row["patient_id"]),
                        "matched_partner_series": str(partner_row["series_id"]),
                        "match_distance": float(edge["distance"]),
                        "mutual_rank_from_sick": (
                            "" if edge.get("rank_from_sick") is None else int(edge["rank_from_sick"])
                        ),
                        "mutual_rank_from_normal": (
                            "" if edge.get("rank_from_normal") is None else int(edge["rank_from_normal"])
                        ),
                        "reciprocal_extended_neighbor": int(edge.get("reciprocal", 0)),
                        "sequence_pair_seeded_by_core": seeded,
                        "family_caliper": float(applied_caliper),
                        "exclusion_reason": "",
                    }
                )
            used_tokens.update((sick_token, normal_token))
            patient_family_counts[sick_patient_key] +=1
            patient_family_counts[normal_patient_key] +=1
            sequence_counts[sick_sequence_key] +=1
            sequence_counts[normal_sequence_key] +=1
            patient_total_counts[str(sick_row["patient_id"])] +=1
            patient_total_counts[str(normal_row["patient_id"])] +=1
            patient_pair_counts[patient_pair_key] +=1
            sequence_pair_counts[sequence_pair_key] +=1
            selected_pair_distances.append(float(edge["distance"]))
            if stage == "core_mutual":
                core_sequence_pairs.add(sequence_pair_key)
                core_pair_distances.append(float(edge["distance"]))
                core_pair_count +=1
                family_data[family]["core_selected"] +=1
            else:
                extended_pair_distances.append(float(edge["distance"]))
                extended_pair_count +=1
                family_data[family]["extended_selected"] +=1

        # Stage 1: preserve the previous strict cohort.
        for family in sorted(shared_families):
            caliper=family_data[family]["core_caliper"]
            if not np.isfinite(caliper):
                continue
            for edge in sorted(
                family_data[family]["core_edges"],
                key=lambda item: (item["distance"], item["tie"]),):
                if float(edge["distance"]) > float(caliper):
                    continue
                if can_select(edge, "core_mutual"):
                    select_edge(edge, "core_mutual")

        # Allocate the remaining target proportionally to the unused common
        # support of each family. This prevents a few large families from taking
        # the whole extension.
        remaining_target=max(0, requested_target_pairs - core_pair_count)
        remaining_capacity={
            family: max(
                0,
                min(
                    len(family_data[family]["normal_positions"]),
                    len(family_data[family]["sick_positions"]),
                )
                - int(family_data[family]["core_selected"]),
            )
            for family in sorted(shared_families)
        }
        total_remaining_capacity=sum(remaining_capacity.values())
        extension_targets={family: 0 for family in shared_families}
        if remaining_target > 0 and total_remaining_capacity > 0:
            raw_targets={
                family: remaining_target
                * remaining_capacity[family]
                / total_remaining_capacity
                for family in shared_families
            }
            extension_targets={
                family: min(
                    remaining_capacity[family], int(math.floor(raw_targets[family]))
                )
                for family in shared_families
            }
            unassigned=remaining_target - sum(extension_targets.values())
            for family in sorted(
                shared_families,
                key=lambda value: (
                    raw_targets[value] - math.floor(raw_targets[value]),
                    remaining_capacity[value],
                ),
                reverse=True,):
                if unassigned <= 0:
                    break
                if extension_targets[family] < remaining_capacity[family]:
                    extension_targets[family] +=1
                    unassigned -=1

        # Stage 2: add close one-sided or reciprocal top-K neighbours. Core-seeded
        # sequence pairs are considered first, then reciprocal candidates, then
        # distance/rank. Quota passes stop a single patient from monopolising a family.
        for family in sorted(shared_families):
            family_target=int(extension_targets.get(family, 0))
            if family_target <= 0:
                continue
            extended_caliper=family_data[family]["extended_caliper"]
            if not np.isfinite(extended_caliper):
                continue
            candidates=[]
            for edge in family_data[family]["all_edges"]:
                if float(edge["distance"]) > float(extended_caliper):
                    continue
                sick_row=eligible_rows[int(edge["sick_global"])]
                normal_row=eligible_rows[int(edge["normal_global"])]
                sequence_pair=MatchingStage.pair_keys(
                    sick_row, normal_row, family
                )[-1]
                seeded=int(sequence_pair in core_sequence_pairs)
                rank_sick=edge.get("rank_from_sick")
                rank_normal=edge.get("rank_from_normal")
                rank_sum=int(rank_sick or 12 + 1) + int(
                    rank_normal or 12 + 1
                )
                candidates.append(
                    {
                        **edge,
                        "seeded": seeded,
                        "rank_sum": rank_sum,
                        "normalized_distance": float(edge["distance"])
                        / max(float(extended_caliper), 1e-9),
                    }
                )
            candidates.sort(
                key=lambda item: (
                    0
                    if (
                        True
                        and item["seeded"]
                    )
                    else 1,
                    0 if item.get("reciprocal", 0) else 1,
                    item["normalized_distance"],
                    item["rank_sum"],
                    item["tie"],
                )
            )

            selected_here=0
            max_cap=max(patient_total_caps.values())
            starting_quota=max(
                1,
                min(
                    [patient_total_counts.get(patient, 0) for patient in patient_total_counts]
                    or [1]
                ),
            )
            quota_values=np.unique(
                np.ceil(
                    np.linspace(
                        starting_quota,
                        max_cap,
                        max(2, int(12)),
                    )
                ).astype(int)
            )
            for quota in quota_values:
                if selected_here >= family_target or pair_number >= requested_target_pairs:
                    break
                patient_quota={0: int(quota), 1: int(quota)}
                for edge in candidates:
                    if selected_here >= family_target or pair_number >= requested_target_pairs:
                        break
                    if can_select(edge, "extended_neighbor", patient_quota=patient_quota):
                        select_edge(edge, "extended_neighbor")
                        selected_here +=1

        # Spillover pass: if a family could not fill its proportional allocation,
        # other families with unused high-quality edges may contribute the remainder.
        # All one-to-one and capacity constraints remain active.
        if pair_number < requested_target_pairs:
            spillover_added=0
            for family in sorted(shared_families):
                if pair_number >= requested_target_pairs:
                    break
                extended_caliper=family_data[family]["extended_caliper"]
                if not np.isfinite(extended_caliper):
                    continue
                spillover_candidates=[]
                for edge in family_data[family]["all_edges"]:
                    if float(edge["distance"]) > float(extended_caliper):
                        continue
                    sick_row=eligible_rows[int(edge["sick_global"])]
                    normal_row=eligible_rows[int(edge["normal_global"])]
                    sequence_pair=MatchingStage.pair_keys(
                        sick_row, normal_row, family
                    )[-1]
                    rank_sick=edge.get("rank_from_sick")
                    rank_normal=edge.get("rank_from_normal")
                    spillover_candidates.append(
                        {
                            **edge,
                            "seeded": int(sequence_pair in core_sequence_pairs),
                            "rank_sum": int(
                                rank_sick or 12 + 1
                            )
                            + int(
                                rank_normal or 12 + 1
                            ),
                            "normalized_distance": float(edge["distance"])
                            / max(float(extended_caliper), 1e-9),
                        }
                    )
                spillover_candidates.sort(
                    key=lambda item: (
                        0
                        if (
                            True
                            and item["seeded"]
                        )
                        else 1,
                        0 if item.get("reciprocal", 0) else 1,
                        item["normalized_distance"],
                        item["rank_sum"],
                        item["tie"],
                    )
                )
                for edge in spillover_candidates:
                    if pair_number >= requested_target_pairs:
                        break
                    if can_select(edge, "extended_neighbor", patient_quota=None):
                        select_edge(edge, "extended_neighbor")
                        spillover_added +=1
            if spillover_added:
                print(
                    "[MATCHING] Spillover redistributed "
                    f"{spillover_added} extended pairs across families."
                )

        # Explain every non-selected image in the manifest.
        for family in sorted(shared_families):
            data=family_data[family]
            candidate_tokens=set()
            below_extended_tokens=set()
            for edge in data["all_edges"]:
                sick_token=str(eligible_rows[int(edge["sick_global"])]["image_token"])
                normal_token=str(eligible_rows[int(edge["normal_global"])]["image_token"])
                candidate_tokens.update((sick_token, normal_token))
                if np.isfinite(data["extended_caliper"]) and float(edge["distance"]) <= float(
                    data["extended_caliper"]):
                    below_extended_tokens.update((sick_token, normal_token))
            for index in family_indices[family]:
                token=str(eligible_rows[index]["image_token"])
                record=manifest_by_token[token]
                if DataStage._as_int(record.get("selected_for_matched_cohort"), 0) == 1:
                    continue
                if token not in candidate_tokens:
                    record["exclusion_reason"]="no_cross_class_neighbor_in_extended_top_k"
                elif token not in below_extended_tokens:
                    record["exclusion_reason"]="above_extended_family_caliper"
                elif pair_number >= requested_target_pairs:
                    record["exclusion_reason"]="expanded_target_reached"
                else:
                    record["exclusion_reason"]="one_to_one_or_capacity_limit"

            family_summaries.append(
                {
                    "family": family,
                    "normal_images": len(data["normal_positions"]),
                    "sick_images": len(data["sick_positions"]),
                    "normal_patients": len(
                        {
                            str(eligible_rows[index]["patient_id"])
                            for index in data["normal_positions"]
                        }
                    ),
                    "sick_patients": len(
                        {
                            str(eligible_rows[index]["patient_id"])
                            for index in data["sick_positions"]
                        }
                    ),
                    "core_candidate_edges": len(data["core_edges"]),
                    "extended_candidate_edges": len(data["all_edges"]),
                    "core_caliper": (
                        float(data["core_caliper"])
                        if np.isfinite(data["core_caliper"])
                        else None
                    ),
                    "extended_caliper": (
                        float(data["extended_caliper"])
                        if np.isfinite(data["extended_caliper"])
                        else None
                    ),
                    "core_selected_pairs": int(data["core_selected"]),
                    "extended_added_pairs": int(data["extended_selected"]),
                    "selected_pairs": int(data["core_selected"] + data["extended_selected"]),
                    "extended_target_pairs": int(extension_targets.get(family, 0)),
                }
            )

        manifest=[
            manifest_by_token[str(row["image_token"])] for row in dataset_rows
        ]
        selected=[
            row
            for row in manifest
            if DataStage._as_int(row.get("selected_for_matched_cohort"), 0) == 1
        ]
        core_selected=[
            row
            for row in manifest
            if DataStage._as_int(row.get("selected_for_core_matched_cohort"), 0) == 1
        ]
        selected_normal=[row for row in selected if DataStage._as_int(row.get("label"), -1) == 0]
        selected_sick=[row for row in selected if DataStage._as_int(row.get("label"), -1) == 1]
        normal_patients=sorted({str(row["patient_id"]) for row in selected_normal})
        sick_patients=sorted({str(row["patient_id"]) for row in selected_sick})
        if len(selected_normal) != len(selected_sick):
            raise RuntimeError("Internal matching produced unequal class sizes.")
        if len(normal_patients) < 2 or len(sick_patients) < 2:
            raise RuntimeError(
                "The matched cohort has too few patients for evaluation: "
                f"Normal={len(normal_patients)}, Sick={len(sick_patients)}."
            )

        per_patient=defaultdict(int)
        for row in selected:
            per_patient[str(row["patient_id"])] +=1
        selected_fraction_eligible=len(selected) / max(1, len(eligible_rows))
        selected_fraction_dataset=len(selected) / max(1, len(dataset_rows))
        summary={
            "dataset_images": len(dataset_rows),
            "eligible_images": len(eligible_rows),
            "acquisition_families": family_count,
            "shared_families": len(shared_families),
            "requested_target_pairs": requested_target_pairs,
            "requested_target_images": requested_target_pairs * 2,
            "core_matched_pairs": len(core_selected) // 2,
            "core_selected_images": len(core_selected),
            "extended_added_pairs": extended_pair_count,
            "extended_added_images": extended_pair_count * 2,
            "matched_pairs": len(selected_normal),
            "selected_images": len(selected),
            "selected_fraction_of_eligible": float(selected_fraction_eligible),
            "selected_fraction_of_dataset": float(selected_fraction_dataset),
            "target_achieved": bool(len(selected_normal) >= requested_target_pairs),
            "selected_normal_images": len(selected_normal),
            "selected_sick_images": len(selected_sick),
            "selected_normal_patients": len(normal_patients),
            "selected_sick_patients": len(sick_patients),
            "normal_patients": normal_patients,
            "sick_patients": sick_patients,
            "patient_total_caps_by_class": {
                "Normal": int(patient_total_caps[0]),
                "Sick": int(patient_total_caps[1]),
            },
            "selected_images_per_patient": dict(sorted(per_patient.items())),
            "distance_median": (
                float(np.median(selected_pair_distances))
                if selected_pair_distances
                else None
            ),
            "distance_p90": (
                float(np.quantile(selected_pair_distances, 0.90))
                if selected_pair_distances
                else None
            ),
            "core_distance_median": (
                float(np.median(core_pair_distances)) if core_pair_distances else None
            ),
            "core_distance_p90": (
                float(np.quantile(core_pair_distances, 0.90))
                if core_pair_distances
                else None
            ),
            "extended_distance_median": (
                float(np.median(extended_pair_distances))
                if extended_pair_distances
                else None
            ),
            "extended_distance_p90": (
                float(np.quantile(extended_pair_distances, 0.90))
                if extended_pair_distances
                else None
            ),
            "families": family_summaries,
            "interpretation": (
                "B1/AU2/C2 use the expanded balanced cohort. The previous strict "
                "mutual-neighbour cohort is retained in selected_for_core_matched_cohort "
                "for audit and sensitivity checks."
            ),
        }
        if not summary["target_achieved"]:
            print(
                "[MATCHING][WARNING] The requested expanded target could not be fully "
                "reached without violating one-to-one, anatomical, patient, or sequence caps."
            )
        print(
            "[MATCHING] Sick↔Normal core+extended: "
            f"eligible={len(eligible_rows)}, core_pairs={len(core_selected) // 2}, "
            f"total_pairs={len(selected_normal)}, images={len(selected)} "
            f"({100.0 * selected_fraction_dataset:.1f}% dataset), "
            f"patients Normal={len(normal_patients)}, Sick={len(sick_patients)}"
        )
        return manifest, summary

# =============================================================================
# STAGE 5 — FROZEN EFFICIENTNET FEATURES
# =============================================================================
# PyTorch Dataset/model/pool classes hold actual runtime state, so classes are natural here.
class FeatureDataset(Dataset):
    """Load full-image, automatic-mask, and manual-mask tensors for feature extraction."""

    def __init__(self, rows):
        self.rows=list(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row=self.rows[index]
        robust, raw, content=DataStage.classifier_views(row["image_path"])
        attention=AttentionStage.read_prediction_mask(row, size=224).astype(np.float32)
        manual_path=Path(row["manual_mask_path"])
        if manual_path.is_file():
            manual=DataStage.read_binary_mask(
                manual_path, size=224
            ).astype(np.float32)
        else:
            manual=np.zeros_like(attention)

        robust_tensor=torch.from_numpy(np.stack([robust] * 3)).float()
        raw_tensor=torch.from_numpy(np.stack([raw] * 3)).float()
        return (
            robust_tensor,
            raw_tensor,
            torch.from_numpy(content).unsqueeze(0).float(),
            torch.from_numpy(attention).unsqueeze(0).float(),
            torch.from_numpy(manual).unsqueeze(0).float(),
            int(index),
        )
class FrozenEfficientNet(nn.Module):
    """Expose pretrained EfficientNet-B0 as a frozen embedding extractor."""

    def __init__(self):
        super().__init__()
        weights=EfficientNet_B0_Weights.IMAGENET1K_V1
        try:
            model=efficientnet_b0(weights=weights)
        except Exception as error:
            raise RuntimeError(
                "EfficientNet-B0 weights could not be loaded. "
                "Enable Internet in Kaggle or place the weights in the Torch cache. "
                f"Original error: {type(error).__name__}: {error}"
            ) from error
        model.classifier=nn.Identity()
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        # CPU/float32 is a deliberate experiment protocol, not automatic device selection.
        self.model=model.to(device="cpu", dtype=torch.float32,
                            memory_format=torch.contiguous_format).eval()
        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32, device="cpu").view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32, device="cpu").view(1, 3, 1, 1)
        )

    def forward(self, images):
        if images.device.type != "cpu":
            raise ValueError("EfficientNet requires CPU inputs in this CPU/float32 pipeline.")
        # An outer notebook autocast context must not silently enable CPU bfloat16.
        with torch.autocast(device_type="cpu", enabled=False):
            images=(images.to(dtype=torch.float32) - self.mean) / self.std
            return self.model(images.contiguous())
class StreamingPatientPool:
    """Aggregate slice embeddings first by series and then by patient."""

    def __init__(self, modes):
        self.series_data={
            mode: {} for mode in modes
        }
        self.source_slices=defaultdict(int)

    def add(
        self,
        mode,
        embeddings,
        rows,
    ):
        embeddings=np.asarray(embeddings, dtype=np.float32)
        for embedding, row in zip(embeddings, rows):
            key=(str(row["patient_id"]), str(row["series_id"]))
            entry=self.series_data[mode].get(key)
            if entry is None:
                entry=[np.zeros_like(embedding, dtype=np.float32), 0, int(row["label"])]
                self.series_data[mode][key]=entry
            if int(entry[2]) != int(row["label"]):
                raise RuntimeError(f"Inconsistent labels for patient {row['patient_id']}.")
            entry[0] +=embedding
            entry[1] +=1
            self.source_slices[mode] +=1

    def finalize(self, mode):
        patient_series=defaultdict(list)
        patient_labels={}
        for (patient_id, _series_id), (embedding_sum, count, label) in self.series_data[mode].items():
            patient_series[patient_id].append(embedding_sum / max(1, count))
            patient_labels[patient_id]=int(label)
        patients=sorted(patient_series)
        if not patients:
            raise RuntimeError(
                f"Mode {mode} contains no patients. For M1/C3/AU3/C4, "
                "inspect [FEATURES][MANUAL] and the current manual-target audit."
            )
        X=np.stack(
            [np.mean(np.stack(patient_series[patient]), axis=0) for patient in patients]
        ).astype(np.float32)
        y=np.asarray([patient_labels[patient] for patient in patients], dtype=np.int64)
        return {
            "X": X,
            "y": y,
            "patient_ids": np.asarray(patients),
            "source_slices": int(self.source_slices[mode]),
            "series_proxies": int(len(self.series_data[mode])),
        }

class FeatureStage:
    """Build all eleven full-image/ROI/complement patient feature sets from scratch."""

    # Full-pipeline CPU defaults. Both sizes affect forward grouping, so retain 4/4.
    BATCH_SIZE = 4
    FORWARD_BATCH_SIZE = 4

    # These names define the research experiments; they are not tunable settings.
    MODES = (
        "B0_FULL_IMAGE",
        "B2_ATTENTION_ELIGIBLE_FULL_IMAGE",
        "AU1_ATTENTION_ROI",
        "C1_ATTENTION_COMPLEMENT",
        "B1_MATCHED_FULL_IMAGE",
        "AU2_MATCHED_ATTENTION_ROI",
        "C2_MATCHED_ATTENTION_COMPLEMENT",
        "M1_MANUAL_ROI",
        "C3_MANUAL_COMPLEMENT",
        "AU3_ATTENTION_ROI_MANUAL_SUBSET",
        "C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET",
    )
    MODE_DESCRIPTIONS = {
    "B0_FULL_IMAGE": "Full-image contextual baseline on every dataset slice",
    "B2_ATTENTION_ELIGIBLE_FULL_IMAGE": "Full image on exactly the AU1/C1 attention-eligible slices",
    "AU1_ATTENTION_ROI": "Automatic Attention U-Net heart ROI on the same attention-eligible slices",
    "C1_ATTENTION_COMPLEMENT": "Pixels outside the automatic heart ROI on the same attention-eligible slices",
    "B1_MATCHED_FULL_IMAGE": "Full-image baseline on the expanded balanced Sick/Normal cohort",
    "AU2_MATCHED_ATTENTION_ROI": "Automatic heart ROI on the expanded balanced matched cohort",
    "C2_MATCHED_ATTENTION_COMPLEMENT": "Automatic-ROI complement on the expanded balanced matched cohort",
    "M1_MANUAL_ROI": "Manual heart ROI on the annotated subset",
    "C3_MANUAL_COMPLEMENT": "Complement of the manual ROI",
    "AU3_ATTENTION_ROI_MANUAL_SUBSET": "Automatic ROI on the same manually annotated images",
    "C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET": "Automatic complement on the same manually annotated images",
}
    MODE_COHORTS = {
    "B0_FULL_IMAGE": "all_dataset_slices",
    "B2_ATTENTION_ELIGIBLE_FULL_IMAGE": "attention_eligible_same_slices",
    "AU1_ATTENTION_ROI": "attention_eligible_same_slices",
    "C1_ATTENTION_COMPLEMENT": "attention_eligible_same_slices",
    "B1_MATCHED_FULL_IMAGE": "cross_class_matched_same_slices",
    "AU2_MATCHED_ATTENTION_ROI": "cross_class_matched_same_slices",
    "C2_MATCHED_ATTENTION_COMPLEMENT": "cross_class_matched_same_slices",
    "M1_MANUAL_ROI": "manual_positive_same_slices",
    "C3_MANUAL_COMPLEMENT": "manual_positive_same_slices",
    "AU3_ATTENTION_ROI_MANUAL_SUBSET": "manual_positive_same_slices",
    "C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET": "manual_positive_same_slices",
}

    @staticmethod
    def region_normalize(images, masks):
        if masks.ndim == 3:
            masks=masks.unsqueeze(1)
        visible=masks > 0.5
        batch_size=images.shape[0]
        bins=int(256)
        mask_flat=visible[:, 0].reshape(batch_size, -1)
        counts=mask_flat.sum(dim=1).long()
        values=images[:, 0].float().clamp(0.0, 1.0)
        indices=torch.round(values * float(bins - 1)).long().clamp_(0, bins - 1)
        indices=indices.reshape(batch_size, -1)
        histogram=torch.zeros(batch_size, bins, device=images.device, dtype=torch.float32)
        histogram.scatter_add_(1, indices, mask_flat.float())
        cumulative=torch.cumsum(histogram, dim=1)
        safe_counts=counts.clamp_min(1)
        lower_rank=(
            torch.floor(
                1.0 / 100.0
                * (safe_counts - 1).float()
            ).long()
            + 1
        )
        upper_rank=(
            torch.floor(
                99.0 / 100.0
                * (safe_counts - 1).float()
            ).long()
            + 1
        )
        lower_bin=(cumulative >= lower_rank[:, None].float()).long().argmax(dim=1)
        upper_bin=(cumulative >= upper_rank[:, None].float()).long().argmax(dim=1)
        lower=lower_bin.float().view(-1, 1, 1, 1) / float(bins - 1)
        upper=upper_bin.float().view(-1, 1, 1, 1) / float(bins - 1)
        valid=(
            (counts >= 64)
            & (upper[:, 0, 0, 0] > lower[:, 0, 0, 0])
        )
        scaled=(images.float() - lower) / (upper - lower).clamp_min(
            8.0 / 255.0
        )
        scaled=scaled.clamp(0.0, 1.0) * visible.float()
        return scaled * valid.view(-1, 1, 1, 1).float()
    # Dilate the heart mask slightly before ROI/complement feature extraction.
    @staticmethod
    def support_mask(mask, content):
        kernel=15  # expand ROI slightly before extracting ROI/complement features
        support=F.max_pool2d(
            (mask > 0.5).float(), kernel_size=kernel, stride=1, padding=kernel // 2
        )
        return (support > 0.5).float() * (content > 0.5).float()
    @staticmethod
    def assert_same_cohort(
        bank,
        modes,
        cohort_name,
    ):

        modes=tuple(modes)
        reference_mode=modes[0]
        reference=bank[reference_mode]
        reference_patients=np.asarray(reference["patient_ids"]).astype(str)
        reference_labels=np.asarray(reference["y"], dtype=np.int64)
        reference_slices=int(reference["source_slices"])
        reference_series=int(reference["series_proxies"])

        for mode in modes[1:]:
            current=bank[mode]
            checks={
                "patient_ids": np.array_equal(
                    reference_patients,
                    np.asarray(current["patient_ids"]).astype(str),
                ),
                "labels": np.array_equal(
                    reference_labels,
                    np.asarray(current["y"], dtype=np.int64),
                ),
                "source_slices": reference_slices == int(current["source_slices"]),
                "series_proxies": reference_series == int(current["series_proxies"]),
            }
            failed=[name for name, passed in checks.items() if not passed]
            if failed:
                raise RuntimeError(
                    f"Same-slice cohort {cohort_name!r} is inconsistent between "
                    f"{reference_mode} and {mode}: {failed}. Rebuild the affected "
                    "feature-bank modes before interpreting their AUC difference."
                )

        print(
            f"[FEATURE BANK][COHORT] {cohort_name}: modes={list(modes)}, "
            f"patients={len(reference_patients)}, slices={reference_slices}, "
            f"series={reference_series}"
        )
    @staticmethod
    def validate_cohort_alignment(bank):
        FeatureStage.assert_same_cohort(
            bank,
            (
                "B2_ATTENTION_ELIGIBLE_FULL_IMAGE",
                "AU1_ATTENTION_ROI",
                "C1_ATTENTION_COMPLEMENT",
            ),
            "attention_eligible_same_slices",
        )
        FeatureStage.assert_same_cohort(
            bank,
            (
                "B1_MATCHED_FULL_IMAGE",
                "AU2_MATCHED_ATTENTION_ROI",
                "C2_MATCHED_ATTENTION_COMPLEMENT",
            ),
            "cross_class_matched_same_slices",
        )
        FeatureStage.assert_same_cohort(
            bank,
            (
                "M1_MANUAL_ROI",
                "C3_MANUAL_COMPLEMENT",
                "AU3_ATTENTION_ROI_MANUAL_SUBSET",
                "C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET",
            ),
            "manual_positive_same_slices",
        )
    # Compute all eleven modes every run; share identical embeddings across exact subsets.
    @staticmethod
    def build_feature_bank(dataset_rows, accepted_rows, matching_manifest, device=None):
        device=torch.device("cpu" if device is None else device)
        if device.type != "cpu":
            raise ValueError(
                "EfficientNet is locked to CPU/float32 to match the full pipeline. "
                "Pass device='cpu'; CUDA remains enabled only for Attention U-Net."
            )
        accepted_manual = {
            str(row["image_token"])
            for row in accepted_rows
            if row.get("segmentation_target_type") == HEART_PRESENT
            and DataStage.manual_mask_qc(row["manual_mask_path"])["usable"]
        }
        print(f"[FEATURES][MANUAL] accepted positive masks={len(accepted_manual)}")
        if not accepted_manual:
            raise RuntimeError("No usable accepted HEART_PRESENT masks remain for M1/C3/AU3/C4.")
        matched_tokens = {
            str(row["image_token"])
            for row in matching_manifest
            if DataStage._as_int(row.get("selected_for_matched_cohort"), 0) == 1
        }
        if not matched_tokens:
            raise RuntimeError("The current matching stage produced no usable matched pairs.")
        rows = []
        for original in dataset_rows:
            row = dict(original)
            if not AttentionStage.has_prediction_mask(row):
                raise RuntimeError(f"Missing persisted OOF prediction for {row['image_token']}.")
            row["keep_attention"] = int(
                DataStage._as_int(row.get("attention_valid_final"), 0) == 1
                and DataStage._as_int(row.get("attention_heart_present"), 1) == 1
                and DataStage._as_int(row.get("quality_valid"), 0) == 1
                and DataStage._as_float(row.get("attention_area_ratio"), 0.0) >= 0.003
            )
            row["keep_manual_matched"] = int(row["image_token"] in accepted_manual)
            row["keep_cross_class_matched"] = int(
                row["image_token"] in matched_tokens and row["keep_attention"] == 1
            )
            rows.append(row)
        if not rows:
            raise RuntimeError("No images are eligible for feature extraction.")
        print(f"[FEATURES] rebuilding all 11 experiments from {len(rows)} source images")
        extractor = FrozenEfficientNet().to(
            device=device, dtype=torch.float32, memory_format=torch.contiguous_format
        ).eval()
        print(f"[FEATURES][NUMERICS] device=cpu | dtype=float32 | autocast=False | "
              f"batch={FeatureStage.BATCH_SIZE} | forward_batch={FeatureStage.FORWARD_BATCH_SIZE} | "
              "weights=IMAGENET1K_V1")
        loader = DataLoader(
            FeatureDataset(rows),
            batch_size=FeatureStage.BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=False,  # CPU feature extraction never transfers batches to CUDA
        )
        pool = StreamingPatientPool(FeatureStage.MODES)
        started = time.perf_counter()

        # Reuse embeddings only inside this run, never from a previous run or file.
        alias_rules = {
            "B0_FULL_IMAGE": (
                ("B2_ATTENTION_ELIGIBLE_FULL_IMAGE", "keep_attention"),
                ("B1_MATCHED_FULL_IMAGE", "keep_cross_class_matched"),
            ),
            "AU1_ATTENTION_ROI": (("AU2_MATCHED_ATTENTION_ROI", "keep_cross_class_matched"),),
            "C1_ATTENTION_COMPLEMENT": (("C2_MATCHED_ATTENTION_COMPLEMENT", "keep_cross_class_matched"),),
        }

        def encode_groups(
            groups,
        ):
            valid_groups=[
                (mode, images, selected_rows)
                for mode, images, selected_rows in groups
                if len(selected_rows) > 0
                and images.shape[0] > 0
            ]
            if not valid_groups:
                return

            combined=torch.cat(
                [images.contiguous() for _, images, _ in valid_groups], dim=0
            )
            forward_batch=FeatureStage.FORWARD_BATCH_SIZE
            embedding_chunks=[]
            for begin in range(0, combined.shape[0], forward_batch):
                images_device=combined[begin : begin + forward_batch].to(
                    device=device, dtype=torch.float32
                ).contiguous()
                output=extractor(images_device)
                # Check the actual computation output; a final .float() alone would
                # hide a preceding low-precision forward instead of correcting it.
                if output.device.type != "cpu" or output.dtype != torch.float32:
                    raise RuntimeError("EfficientNet must produce CPU/float32 embeddings.")
                embedding_chunks.append(output)
                del images_device, output

            embeddings=torch.cat(embedding_chunks, dim=0).numpy().astype(np.float32)
            cursor=0
            for mode, images, selected_rows in valid_groups:
                count=int(images.shape[0])
                current_embeddings=embeddings[cursor : cursor + count]
                pool.add(mode, current_embeddings, selected_rows)

                for alias_mode, selector_field in alias_rules.get(mode, ()):
                    alias_positions=[
                        position
                        for position, row in enumerate(selected_rows)
                        if DataStage._as_int(row.get(selector_field), 0) == 1
                    ]
                    if alias_positions:
                        index_array=np.asarray(alias_positions, dtype=np.int64)
                        pool.add(
                            alias_mode,
                            current_embeddings[index_array],
                            [selected_rows[position] for position in alias_positions],
                        )
                cursor +=count
            del combined, embeddings, embedding_chunks

        # Disable even an inherited CPU autocast context for the complete feature path.
        with torch.inference_mode(), torch.autocast(device_type="cpu", enabled=False):
            for robust, raw, content, attention, manual, indices in tqdm(
                loader, desc=f"EfficientNet feature bank ({device.type})"
            ):
                batch_rows = [rows[int(index)] for index in indices]
                attention_positions = [i for i, row in enumerate(batch_rows) if row["keep_attention"]]
                manual_positions = [i for i, row in enumerate(batch_rows) if row["keep_manual_matched"]]
                groups = [("B0_FULL_IMAGE", robust, batch_rows)]

                # Automatic ROI/complement plus aliases for the matched cohort.
                if attention_positions:
                    positions = torch.as_tensor(attention_positions, dtype=torch.long)
                    selected_raw = raw.index_select(0, positions)
                    selected_content = content.index_select(0, positions)
                    selected_mask = attention.index_select(0, positions)
                    support = FeatureStage.support_mask(selected_mask, selected_content)
                    selected_rows = [batch_rows[i] for i in attention_positions]
                    groups.extend([
                        ("AU1_ATTENTION_ROI", FeatureStage.region_normalize(selected_raw, support), selected_rows),
                        ("C1_ATTENTION_COMPLEMENT", FeatureStage.region_normalize(
                            selected_raw, (selected_content > 0.5).float() * (1.0 - support)
                        ), selected_rows),
                    ])

                # All four manual-reference experiments use exactly these same slices.
                if manual_positions:
                    positions = torch.as_tensor(manual_positions, dtype=torch.long)
                    selected_raw = raw.index_select(0, positions)
                    selected_content = content.index_select(0, positions)
                    manual_support = FeatureStage.support_mask(manual.index_select(0, positions), selected_content)
                    attention_support = FeatureStage.support_mask(attention.index_select(0, positions), selected_content)
                    selected_rows = [batch_rows[i] for i in manual_positions]
                    visible_content = (selected_content > 0.5).float()
                    regions = (
                        ("M1_MANUAL_ROI", manual_support),
                        ("C3_MANUAL_COMPLEMENT", visible_content * (1.0 - manual_support)),
                        ("AU3_ATTENTION_ROI_MANUAL_SUBSET", attention_support),
                        ("C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET", visible_content * (1.0 - attention_support)),
                    )
                    groups.extend(
                        (mode, FeatureStage.region_normalize(selected_raw, support), selected_rows)
                        for mode, support in regions
                    )
                encode_groups(groups)
                del groups

        bank = {mode: pool.finalize(mode) for mode in FeatureStage.MODES}
        FeatureStage.validate_cohort_alignment(bank)
        print("[FEATURES] all experiment representations built in RAM | "
              + DataStage.format_seconds(time.perf_counter() - started))
        del extractor, loader, pool
        DataStage.release_device(device)
        return bank

# =============================================================================
# STAGE 6 — PATIENT-LEVEL EVALUATION
# =============================================================================
class EvaluationStage:
    """Nested CV, confidence intervals, and paired AUC comparisons."""

    # Build classifier pipeline.
    @staticmethod
    def build_classifier_pipeline(c_value):
        return SklearnPipeline(
            [
                ("scale", StandardScaler()),
                (
                    "pca",
                    PCA(
                        n_components=0.95,
                        svd_solver="full",
                    ),
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=float(c_value),
                        class_weight="balanced",
                        solver="liblinear",
                        max_iter=5000,
                        random_state=42,
                    ),
                ),
            ]
        )
    # Choose the probability threshold maximizing sensitivity + specificity - 1.
    @staticmethod
    def youden_threshold(labels, scores):
        fpr, tpr, thresholds=roc_curve(labels, scores)
        finite=np.isfinite(thresholds)
        if not np.any(finite):
            return 0.5
        index=np.argmax((tpr - fpr)[finite])
        return float(thresholds[finite][index])
    # Select c and threshold.
    @staticmethod
    def select_c_and_threshold(X, y, seed):
        class_counts=np.bincount(y, minlength=2)
        n_splits=min(3, int(class_counts.min()))
        if n_splits < 2:
            return 1.0, 0.5
        splitter=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        best_c, best_auc, best_scores=1.0, -np.inf, None
        for c_value in (0.01, 0.1, 1.0, 10.0):
            scores=np.full(len(y), np.nan, dtype=np.float64)
            for train_index, valid_index in splitter.split(X, y):
                model=EvaluationStage.build_classifier_pipeline(c_value)
                model.fit(X[train_index], y[train_index])
                scores[valid_index]=model.predict_proba(X[valid_index])[:, 1]
            auc=roc_auc_score(y, scores)
            if (auc, -abs(math.log10(c_value))) > (best_auc, -abs(math.log10(best_c))):
                best_c, best_auc, best_scores=float(c_value), float(auc), scores.copy()
        threshold=EvaluationStage.youden_threshold(y, best_scores)
        return best_c, threshold
    # Create deterministic stratified outer/inner folds over patients, never slices.
    @staticmethod
    def patient_fold_map(reference):
        patients=np.asarray(reference["patient_ids"]).astype(str)
        labels=np.asarray(reference["y"], dtype=np.int64)
        class_counts=np.bincount(labels, minlength=2)
        n_splits=min(5, int(class_counts.min()))
        if n_splits < 2:
            raise RuntimeError("At least two patients are required in each class.")
        splitter=StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=42,
        )
        mapping={}
        for fold, (_, valid_index) in enumerate(splitter.split(np.zeros(len(labels)), labels), start=1):
            for index in valid_index:
                mapping[patients[index]]=fold
        return mapping
    # Bootstrap patient-level AUC to report a sampling confidence interval.
    @staticmethod
    def auc_confidence_interval(labels, scores, repeats, seed):
        rng=np.random.default_rng(seed)
        values=[]
        for _ in range(int(repeats)):
            indices=rng.integers(0, len(labels), size=len(labels))
            if len(np.unique(labels[indices])) < 2:
                continue
            values.append(roc_auc_score(labels[indices], scores[indices]))
        if not values:
            return np.nan, np.nan
        return float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))
    # Calculate classification metrics.
    @staticmethod
    def classification_metrics(labels, scores, predictions):
        tn, fp, fn, tp=confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
        ci_low, ci_high=EvaluationStage.auc_confidence_interval(
            labels,
            scores,
            2000,
            42 + 9000,
        )
        return {
            "auc": float(roc_auc_score(labels, scores)),
            "auc_ci_low": ci_low,
            "auc_ci_high": ci_high,
            "average_precision": float(average_precision_score(labels, scores)),
            "brier": float(brier_score_loss(labels, scores)),
            "accuracy": float(np.mean(predictions == labels)),
            "sensitivity": float(tp / max(1, tp + fn)),
            "specificity": float(tn / max(1, tn + fp)),
        }
    # Calculate paired auc difference.
    @staticmethod
    def paired_auc_difference(
        first,
        second,
        repeats,
        seed,
    ):
        merged=first.merge(second, on=["patient_id", "true_label"], suffixes=("_first", "_second"))
        labels=merged["true_label"].to_numpy(dtype=np.int64)
        first_scores=merged["score_first"].to_numpy(dtype=np.float64)
        second_scores=merged["score_second"].to_numpy(dtype=np.float64)
        if len(merged) < 4 or len(np.unique(labels)) < 2:
            return {
                "n_patients": len(merged),
                "auc_difference_first_minus_second": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan,
            }
        observed=float(roc_auc_score(labels, first_scores) - roc_auc_score(labels, second_scores))
        rng=np.random.default_rng(seed)
        differences=[]
        for _ in range(int(repeats)):
            indices=rng.integers(0, len(labels), size=len(labels))
            if len(np.unique(labels[indices])) < 2:
                continue
            differences.append(
                roc_auc_score(labels[indices], first_scores[indices])
                - roc_auc_score(labels[indices], second_scores[indices])
            )
        return {
            "n_patients": len(merged),
            "auc_difference_first_minus_second": observed,
            "ci_low": float(np.quantile(differences, 0.025)) if differences else np.nan,
            "ci_high": float(np.quantile(differences, 0.975)) if differences else np.nan,
        }
    # Select classifier settings inside outer training folds and return all results in RAM.
    @staticmethod
    def evaluate_experiments(bank):
        fold_map=EvaluationStage.patient_fold_map(bank["B0_FULL_IMAGE"])
        summary_rows=[]
        prediction_tables={}

        for mode in FeatureStage.MODES:
            values=bank[mode]
            X=np.asarray(values["X"], dtype=np.float32)
            y=np.asarray(values["y"], dtype=np.int64)
            patients=np.asarray(values["patient_ids"]).astype(str)
            if len(np.unique(y)) < 2:
                raise RuntimeError(f"{mode}: one class is missing.")
            unknown=sorted(set(patients) - set(fold_map))
            if unknown:
                raise RuntimeError(f"{mode}: patients missing from the fold map: {unknown}")
            patient_folds=np.asarray([fold_map[patient] for patient in patients], dtype=np.int64)
            scores=np.full(len(y), np.nan, dtype=np.float64)
            predictions=np.full(len(y), -1, dtype=np.int64)
            selected_cs=np.full(len(y), np.nan, dtype=np.float64)
            thresholds=np.full(len(y), np.nan, dtype=np.float64)

            for fold in sorted(np.unique(patient_folds)):
                train_index=np.flatnonzero(patient_folds != fold)
                valid_index=np.flatnonzero(patient_folds == fold)
                if len(np.unique(y[train_index])) < 2:
                    raise RuntimeError(
                        f"{mode}: fold {fold} does not contain both classes in training. "
                        "The matched cohort is too sparse; inspect the matching summary."
                    )
                best_c, threshold=EvaluationStage.select_c_and_threshold(
                    X[train_index], y[train_index], 42 + int(fold)
                )
                model=EvaluationStage.build_classifier_pipeline(best_c)
                model.fit(X[train_index], y[train_index])
                fold_scores=model.predict_proba(X[valid_index])[:, 1]
                scores[valid_index]=fold_scores
                predictions[valid_index]=(fold_scores >= threshold).astype(np.int64)
                selected_cs[valid_index]=best_c
                thresholds[valid_index]=threshold

            if not np.all(np.isfinite(scores)) or np.any(predictions < 0):
                raise RuntimeError(f"{mode}: incomplete OOF predictions.")
            metrics=EvaluationStage.classification_metrics(y, scores, predictions)
            table=pd.DataFrame(
                {
                    "patient_id": patients,
                    "true_label": y,
                    "outer_fold": patient_folds,
                    "score": scores,
                    "predicted_label": predictions,
                    "selected_c": selected_cs,
                    "training_threshold": thresholds,
                }
            ).sort_values("patient_id")
            prediction_tables[mode]=table
            summary_rows.append(
                {
                    "mode": mode,
                    "experiment_group": (
                        "primary"
                        if mode in FeatureStage.MODES[:7]
                        else "mask_validation"
                    ),
                    "description": FeatureStage.MODE_DESCRIPTIONS[mode],
                    "cohort": FeatureStage.MODE_COHORTS[mode],
                    "patients": len(patients),
                    "source_slices": values["source_slices"],
                    "series_proxies": values["series_proxies"],
                    **metrics,
                }
            )
            print(
                f"[EVALUATION] {mode}: AUC={metrics['auc']:.3f} "
                f"AP={metrics['average_precision']:.3f} Brier={metrics['brier']:.3f}"
            )

        summary=pd.DataFrame(summary_rows).sort_values("auc", ascending=False)

        comparisons=[]
        for first_mode, second_mode, question in (
            (
                "AU1_ATTENTION_ROI",
                "B2_ATTENTION_ELIGIBLE_FULL_IMAGE",
                "same_attention_eligible_slices_roi_vs_full_image",
            ),
            (
                "B2_ATTENTION_ELIGIBLE_FULL_IMAGE",
                "C1_ATTENTION_COMPLEMENT",
                "same_attention_eligible_slices_full_image_vs_complement",
            ),
            (
                "AU1_ATTENTION_ROI",
                "C1_ATTENTION_COMPLEMENT",
                "same_attention_eligible_slices_roi_vs_complement",
            ),
            (
                "B2_ATTENTION_ELIGIBLE_FULL_IMAGE",
                "B0_FULL_IMAGE",
                "attention_eligible_selection_effect_on_full_image",
            ),
            (
                "AU2_MATCHED_ATTENTION_ROI",
                "B1_MATCHED_FULL_IMAGE",
                "cross_class_matched_roi_vs_full_image",
            ),
            (
                "AU2_MATCHED_ATTENTION_ROI",
                "C2_MATCHED_ATTENTION_COMPLEMENT",
                "cross_class_matched_roi_vs_complement",
            ),
            (
                "B1_MATCHED_FULL_IMAGE",
                "C2_MATCHED_ATTENTION_COMPLEMENT",
                "cross_class_matched_full_image_vs_complement",
            ),
            (
                "B0_FULL_IMAGE",
                "B1_MATCHED_FULL_IMAGE",
                "all_slices_vs_cross_class_matched_full_image",
            ),
            (
                "AU1_ATTENTION_ROI",
                "AU2_MATCHED_ATTENTION_ROI",
                "attention_eligible_vs_cross_class_matched_roi",
            ),
            (
                "C1_ATTENTION_COMPLEMENT",
                "C2_MATCHED_ATTENTION_COMPLEMENT",
                "attention_eligible_vs_cross_class_matched_complement",
            ),
            (
                "M1_MANUAL_ROI",
                "AU3_ATTENTION_ROI_MANUAL_SUBSET",
                "manual_vs_attention_roi_same_images",
            ),
            (
                "C3_MANUAL_COMPLEMENT",
                "C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET",
                "manual_vs_attention_complement_same_images",
            ),):
            comparison=EvaluationStage.paired_auc_difference(
                prediction_tables[first_mode],
                prediction_tables[second_mode],
                2000,
                42 + len(comparisons) * 100,
            )
            comparisons.append(
                {
                    "comparison": question,
                    "first_mode": first_mode,
                    "second_mode": second_mode,
                    **comparison,
                }
            )
        return summary, pd.DataFrame(comparisons), prediction_tables


# =============================================================================
# PERSISTENT EXECUTION LAYER — NON-NEURAL CACHES AND RESTART RECOVERY
# =============================================================================
# Neural checkpoint and prediction persistence lives directly in AttentionStage;
# the class below owns matching, feature-bank, and evaluation caches.
class PersistenceStage:
    """Validate and persist non-neural stages without changing their algorithms."""

    LEGACY_MODE_ALIASES={
        "B0_FULL_IMAGE": "FULL_IMAGE",
        "B2_ATTENTION_ELIGIBLE_FULL_IMAGE": "B2_ATTENTION_ELIGIBLE_FULL_IMAGE",
        "AU1_ATTENTION_ROI": "AU1_ATTENTION_ROI",
        "C1_ATTENTION_COMPLEMENT": "AU5_ATTENTION_COMPLEMENT",
        "B1_MATCHED_FULL_IMAGE": "CROSS_CLASS_MATCHED_FULL_IMAGE",
        "AU2_MATCHED_ATTENTION_ROI": "CROSS_CLASS_MATCHED_AU1_ATTENTION_ROI",
        "C2_MATCHED_ATTENTION_COMPLEMENT": "CROSS_CLASS_MATCHED_AU5_ATTENTION_COMPLEMENT",
        "M1_MANUAL_ROI": "AU6_MANUAL_ROI",
        "C3_MANUAL_COMPLEMENT": "AU7_MANUAL_COMPLEMENT",
        "AU3_ATTENTION_ROI_MANUAL_SUBSET": "AU8_ATTENTION_MATCHED_MANUAL_ROI",
        "C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET": "AU9_ATTENTION_MATCHED_MANUAL_COMPLEMENT",
    }

    @staticmethod
    def mark(workspace, stage, status="complete", **details):
        state=DataStage.read_json(workspace.stage_state, {}) or {}
        state.update({
            "last_stage": str(stage),
            "last_status": str(status),
            "updated_utc": pd.Timestamp.utcnow().isoformat(),
        })
        state.setdefault("stages", {})[str(stage)]={
            "status": str(status),
            "updated_utc": state["updated_utc"],
            **details,
        }
        DataStage.write_json(workspace.stage_state, state)
        return state

    @staticmethod
    def save_dataset_manifest(rows, workspace):
        if rows:
            DataStage.write_csv(
                workspace.dataset_manifest,
                rows,
                DataStage.csv_fields(rows, rows[0].keys()),
            )

    @staticmethod
    def save_manual_audit(audit, summary, workspace):
        fields=(
            "image_token", "patient_id", "series_id", "manual_mask_path",
            "status", "target_type", "heart_present", "annotation_source",
            "sample_weight", "area_ratio", "quality_valid", "quality_reason",
            "reason",
        )
        DataStage.write_csv(workspace.manual_audit, audit, fields)
        DataStage.write_json(workspace.manual_summary, summary)

    @staticmethod
    def save_segmentation_metrics(summary, metrics, workspace):
        if isinstance(metrics, pd.DataFrame):
            rows=metrics.to_dict("records")
        else:
            rows=list(metrics or [])
        if rows:
            DataStage.write_csv(
                workspace.segmentation_metrics,
                rows,
                DataStage.csv_fields(rows),
            )
        DataStage.write_json(workspace.segmentation_summary, summary)

    @staticmethod
    def matching_fingerprint(rows, workspace=None):
        """Semantic matching identity; ignores CSV formatting and Kaggle mount mtimes."""
        payload={
            "schema": "cross-class-matching-stable-v3",
            "min_families":8,"max_families":32,"target_images_per_family":1800,
            "minimum_images_per_class_per_family":12,
            "minimum_patients_per_class_per_family":2,
            "core_mutual_neighbors":5,"core_caliper_mad_multiplier":2.5,
            "core_caliper_quantile":0.90,"maximum_sequence_position_difference":0.25,
            "maximum_area_ratio_difference":0.20,
            "core_maximum_matches_per_patient_per_family":20,
            "core_maximum_matches_per_sequence_group":5,
            "target_matched_image_fraction":0.50,"extended_neighbors":12,
            "extended_caliper_mad_multiplier":3.5,"extended_caliper_quantile":0.97,
            "extended_max_caliper_multiplier":1.18,
            "extended_maximum_sequence_position_difference":0.25,
            "extended_maximum_area_ratio_difference":0.20,
            "extended_maximum_matches_per_patient_per_family":60,
            "extended_maximum_matches_per_sequence_group":15,
            "extended_maximum_matches_per_sequence_pair":12,
            "extended_maximum_matches_per_patient_pair":60,
            "patient_total_cap_multiplier":1.70,"prioritize_core_sequence_pairs":True,
            "block_weights":{"phash":0.45,"geometry":0.25,"sequence":0.15,"quality":0.15},
            "random_seed":42,
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        for row in sorted(rows, key=lambda item: str(item.get("image_token", ""))):
            values=(
                str(row.get("image_token", "")), DataStage._as_int(row.get("label"),-1),
                str(row.get("patient_id", "")), str(row.get("series_id", "")),
                DataStage.stable_sequence_group(row),
                DataStage._as_int(row.get("sequence_index"),0),
                DataStage._as_int(row.get("sequence_length"),1),
                DataStage._as_int(row.get("quality_valid"),0), str(row.get("perceptual_hash", "")),
                DataStage.canonical_float(row.get("sharpness")),
                DataStage.canonical_float(row.get("noise_ratio")),
                DataStage.canonical_float(row.get("dynamic_range")),
                DataStage._as_int(row.get("attention_valid_final"),0),
                DataStage._as_int(row.get("attention_heart_present"),0),
                DataStage.canonical_float(row.get("attention_area_ratio")),
                DataStage.canonical_float(row.get("attention_centroid_x")),
                DataStage.canonical_float(row.get("attention_centroid_y")),
                DataStage.canonical_float(row.get("attention_boundary_touch_fraction")),
            )
            digest.update(json.dumps(values, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))
        return digest.hexdigest()

    @staticmethod
    def legacy_matching_fingerprint(rows, workspace):
        payload={
            "schema":"cross-class-matching-v2-core-plus-extended","min_families":8,
            "max_families":32,"target_images_per_family":1800,
            "minimum_images_per_class_per_family":12,"minimum_patients_per_class_per_family":2,
            "core_mutual_neighbors":5,"core_caliper_mad_multiplier":2.5,
            "core_caliper_quantile":0.90,"maximum_sequence_position_difference":0.25,
            "maximum_area_ratio_difference":0.20,"core_maximum_matches_per_patient_per_family":20,
            "core_maximum_matches_per_sequence_group":5,"target_matched_image_fraction":0.50,
            "extended_neighbors":12,"extended_caliper_mad_multiplier":3.5,
            "extended_caliper_quantile":0.97,"extended_max_caliper_multiplier":1.18,
            "extended_maximum_sequence_position_difference":0.25,
            "extended_maximum_area_ratio_difference":0.20,
            "extended_maximum_matches_per_patient_per_family":60,
            "extended_maximum_matches_per_sequence_group":15,
            "extended_maximum_matches_per_sequence_pair":12,
            "extended_maximum_matches_per_patient_pair":60,"patient_total_cap_multiplier":1.70,
            "prioritize_core_sequence_pairs":True,
            "block_weights":{"phash":0.45,"geometry":0.25,"sequence":0.15,"quality":0.15},
            "random_seed":42,
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        for row in rows:
            for key in ("image_token","label","patient_id","sequence_group_id","sequence_index","sequence_length"):
                digest.update(str(row.get(key, "")).encode("utf-8"))
        for audit_path in (workspace.quality_audit, workspace.prediction_audit):
            if Path(audit_path).is_file():
                digest.update(Path(audit_path).name.encode("utf-8"))
                digest.update(DataStage.sha256_file(audit_path).encode("ascii"))
        return digest.hexdigest()

    @staticmethod
    def matching_cache_status(rows, workspace):
        summary=DataStage.read_json(workspace.matching_summary, {}) or {}
        stable=PersistenceStage.matching_fingerprint(rows, workspace)
        legacy=(
            PersistenceStage.legacy_matching_fingerprint(rows, workspace)
            if not summary.get("stable_fingerprint")
            else ""
        )
        cached=DataStage.read_csv(workspace.matching_manifest)
        expected_tokens={str(row["image_token"]) for row in rows}
        cached_tokens=[str(row.get("image_token", "")) for row in cached]
        required={"image_token","selected_for_matched_cohort","label","patient_id","series_id"}
        structural=(
            bool(cached) and len(cached_tokens)==len(expected_tokens)
            and len(set(cached_tokens))==len(cached_tokens)
            and set(cached_tokens)==expected_tokens
            and required.issubset(set(cached[0]))
        )
        if structural:
            selected=sum(DataStage._as_int(row.get("selected_for_matched_cohort"),0) for row in cached)
            structural=(selected == DataStage._as_int(summary.get("selected_images"), selected))
        fingerprint_match=(
            summary.get("stable_fingerprint") == stable
            or summary.get("fingerprint") in {stable, legacy}
        )
        freshness=False
        if structural and Path(workspace.matching_manifest).is_file():
            manifest_mtime=Path(workspace.matching_manifest).stat().st_mtime_ns
            input_mtime=DataStage.newest_mtime_ns([workspace.quality_audit, workspace.prediction_audit])
            freshness=manifest_mtime >= input_mtime
        legacy_adoptable=(
            summary.get("schema") == "cross-class-matching-v2-core-plus-extended"
            and not summary.get("stable_fingerprint")
        )
        reusable=bool(structural and (fingerprint_match or (legacy_adoptable and freshness)))
        reason=("fingerprint" if fingerprint_match else "legacy-structural+freshness" if reusable else "invalid")
        return {
            "reusable":reusable,"reason":reason,"stable_fingerprint":stable,
            "legacy_fingerprint":legacy,"summary":summary,"cached":cached,
        }

    @staticmethod
    def build_matching(rows, workspace, force=False):
        status=PersistenceStage.matching_cache_status(rows, workspace)
        if not force and status["reusable"]:
            if status["summary"].get("stable_fingerprint") != status["stable_fingerprint"]:
                DataStage.write_json(
                    workspace.matching_summary,
                    {**status["summary"], "stable_fingerprint": status["stable_fingerprint"],
                     "cache_adoption": status["reason"]},
                )
            print(
                f"[MATCHING] Compatible persisted manifest reused ({status['reason']}): "
                f"{workspace.matching_manifest}"
            )
            return status["cached"], status["summary"]
        manifest, summary=MatchingStage.build_cross_class_matching(rows)
        summary={
            **summary,
            "fingerprint": status["stable_fingerprint"],
            "stable_fingerprint": status["stable_fingerprint"],
            "schema": "cross-class-matching-stable-v3",
        }
        DataStage.write_csv(workspace.matching_manifest, manifest, DataStage.csv_fields(manifest))
        DataStage.write_json(workspace.matching_summary, summary)
        print(f"[MATCHING] Manifest persisted: {workspace.matching_manifest}")
        return manifest, summary

    @staticmethod
    def feature_fingerprint(rows, accepted_rows, workspace):
        prediction_summary=DataStage.read_json(workspace.prediction_summary, {}) or {}
        quality_summary=DataStage.read_json(workspace.quality_summary, {}) or {}
        manual_summary=DataStage.read_json(workspace.manual_summary, {}) or {}
        payload={
            "schema":"simple-patient-feature-bank-stable-v7",
            "modes":tuple(PersistenceStage.LEGACY_MODE_ALIASES.values()),
            "prediction_fingerprint":prediction_summary.get("stable_fingerprint") or prediction_summary.get("fingerprint", ""),
            "manual_input_fingerprint":manual_summary.get("input_fingerprint", ""),
            "support_dilation":15,"imagenet_weights":"IMAGENET1K_V1",
            "classification_size":224,"dtype":"float32","device":"cpu",
            "batch":4,"forward_batch":4,
            "quality_thresholds":{"dynamic_range":12.0,"laplacian_variance":4.0,
                "noise_ratio":0.30,"patient_blur_quantile":0.03,"patient_noise_quantile":0.97},
            "dataset_fingerprint":quality_summary.get("dataset_fingerprint") or DataStage.dataset_fingerprint(rows),
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        for row in sorted(rows, key=lambda item: str(item.get("image_token", ""))):
            values=(
                str(row.get("image_token", "")), DataStage._as_int(row.get("label"),-1),
                str(row.get("patient_id", "")), str(row.get("series_id", "")),
                DataStage._as_int(row.get("quality_valid"),0),
                DataStage._as_int(row.get("attention_valid_final"),0),
                DataStage._as_int(row.get("attention_heart_present"),0),
                DataStage.canonical_float(row.get("attention_area_ratio")),
            )
            digest.update(json.dumps(values, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))
        matching=DataStage.read_csv(workspace.matching_manifest)
        for row in sorted(matching, key=lambda item: str(item.get("image_token", ""))):
            values=(str(row.get("image_token", "")),
                    DataStage._as_int(row.get("selected_for_matched_cohort"),0),
                    str(row.get("pair_id", "")))
            digest.update(json.dumps(values, separators=(",", ":")).encode("utf-8"))
        for row in sorted(accepted_rows, key=lambda item: str(item["image_token"])):
            if row.get("segmentation_target_type") != HEART_PRESENT:
                continue
            values=(str(row["image_token"]), str(row.get("segmentation_target_type", "")),
                    DataStage.canonical_float(row.get("sample_weight", 1.0)))
            digest.update(json.dumps(values, separators=(",", ":")).encode("utf-8"))
        return digest.hexdigest()

    @staticmethod
    def legacy_feature_fingerprint(rows, accepted_rows, workspace):
        prediction_summary=DataStage.read_json(workspace.prediction_summary, {}) or {}
        payload={
            "schema":"simple-patient-feature-bank-cross-class-v6-same-slice-baseline",
            "modes":tuple(PersistenceStage.LEGACY_MODE_ALIASES.values()),
            "prediction_fingerprint":prediction_summary.get("fingerprint", ""),
            "support_dilation":15,"imagenet_weights":True,"classification_size":224,
            "quality_thresholds":{"dynamic_range":12.0,"laplacian_variance":4.0,
                "noise_ratio":0.30,"patient_blur_quantile":0.03,"patient_noise_quantile":0.97},
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        digest.update(str(len(rows)).encode("ascii"))
        for row in rows:
            digest.update(str(row["image_token"]).encode("utf-8"))
        for path in (workspace.quality_audit,workspace.manual_audit,workspace.manual_annotations,
                     workspace.prediction_audit,workspace.matching_manifest):
            if Path(path).is_file():
                digest.update(Path(path).name.encode("utf-8"))
                digest.update(DataStage.sha256_file(path).encode("ascii"))
        for row in sorted(accepted_rows, key=lambda item: str(item["image_token"])):
            if row.get("segmentation_target_type") != HEART_PRESENT:
                continue
            mask_path=Path(row["manual_mask_path"])
            if mask_path.is_file():
                digest.update(str(row["image_token"]).encode("utf-8"))
                digest.update(DataStage.sha256_file(mask_path).encode("ascii"))
        return digest.hexdigest()

    @staticmethod
    def expected_feature_cohorts(rows, accepted_rows, matching):
        accepted_manual={
            str(row["image_token"]) for row in accepted_rows
            if row.get("segmentation_target_type") == HEART_PRESENT
            and DataStage.manual_mask_qc(row["manual_mask_path"])["usable"]
        }
        matched={
            str(row.get("image_token", "")) for row in matching
            if DataStage._as_int(row.get("selected_for_matched_cohort"),0)==1
        }
        prepared=[]
        for source in rows:
            row=dict(source)
            keep_attention=(
                DataStage._as_int(row.get("attention_valid_final"),0)==1
                and DataStage._as_int(row.get("attention_heart_present"),1)==1
                and DataStage._as_int(row.get("quality_valid"),0)==1
                and DataStage._as_float(row.get("attention_area_ratio"),0.0)>=0.003
            )
            row["keep_attention"]=int(keep_attention)
            row["keep_manual"]=int(str(row["image_token"]) in accepted_manual)
            row["keep_matched"]=int(str(row["image_token"]) in matched and keep_attention)
            prepared.append(row)
        selectors={
            "B0_FULL_IMAGE":lambda row: True,
            "B2_ATTENTION_ELIGIBLE_FULL_IMAGE":lambda row: row["keep_attention"],
            "AU1_ATTENTION_ROI":lambda row: row["keep_attention"],
            "C1_ATTENTION_COMPLEMENT":lambda row: row["keep_attention"],
            "B1_MATCHED_FULL_IMAGE":lambda row: row["keep_matched"],
            "AU2_MATCHED_ATTENTION_ROI":lambda row: row["keep_matched"],
            "C2_MATCHED_ATTENTION_COMPLEMENT":lambda row: row["keep_matched"],
            "M1_MANUAL_ROI":lambda row: row["keep_manual"],
            "C3_MANUAL_COMPLEMENT":lambda row: row["keep_manual"],
            "AU3_ATTENTION_ROI_MANUAL_SUBSET":lambda row: row["keep_manual"],
            "C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET":lambda row: row["keep_manual"],
        }
        expected={}
        for mode, selector in selectors.items():
            selected=[row for row in prepared if selector(row)]
            patients=sorted({str(row["patient_id"]) for row in selected})
            labels={str(row["patient_id"]):DataStage._as_int(row.get("label"),-1) for row in selected}
            expected[mode]={
                "patient_ids":np.asarray(patients),
                "y":np.asarray([labels[patient] for patient in patients], dtype=np.int64),
                "source_slices":len(selected),
                "series_proxies":len({(str(row["patient_id"]),str(row["series_id"])) for row in selected}),
            }
        return expected

    @staticmethod
    def feature_upstream_state(rows, accepted_rows, workspace):
        """Return semantic lineage consumed by the feature bank.

        Metadata files may be migrated or rewritten without changing the image/mask
        inputs. These fingerprints describe the underlying pipeline state instead of
        relying on volatile file modification times.
        """
        quality_summary=DataStage.read_json(workspace.quality_summary, {}) or {}
        manual_summary=DataStage.read_json(workspace.manual_summary, {}) or {}
        prediction_summary=DataStage.read_json(workspace.prediction_summary, {}) or {}
        matching_summary=DataStage.read_json(workspace.matching_summary, {}) or {}
        return {
            "dataset_fingerprint": (
                quality_summary.get("dataset_fingerprint")
                or DataStage.dataset_fingerprint(rows)
            ),
            "manual_input_fingerprint": (
                manual_summary.get("input_fingerprint")
                or DataStage.manual_state_fingerprint(rows, workspace)
            ),
            "prediction_fingerprint": (
                prediction_summary.get("stable_fingerprint")
                or prediction_summary.get("fingerprint", "")
            ),
            "matching_fingerprint": (
                matching_summary.get("stable_fingerprint")
                or matching_summary.get("fingerprint", "")
            ),
            "accepted_training_targets": int(len(accepted_rows)),
            "positive_manual_targets": int(sum(
                row.get("segmentation_target_type") == HEART_PRESENT
                for row in accepted_rows
            )),
        }

    @staticmethod
    def feature_bank_structurally_compatible(bank, rows, accepted_rows, matching, workspace):
        """Safely adopt a legacy feature bank when its direct inputs are unchanged.

        ``simple_prediction_summary.json`` is deliberately not compared by mtime:
        migrating its metadata does not alter any EfficientNet input. For the one-time
        legacy adoption we instead validate every cohort and ensure that the actual
        automatic/manual mask PNG files have not changed after the feature bank.
        Future runs use the stable semantic fingerprint written during adoption.
        """
        expected=PersistenceStage.expected_feature_cohorts(rows, accepted_rows, matching)
        for mode in FeatureStage.MODES:
            if mode not in bank:
                return False, f"missing mode {mode}"
            current=bank[mode]
            wanted=expected[mode]
            current_patients=np.asarray(current["patient_ids"]).astype(str)
            current_labels=np.asarray(current["y"], dtype=np.int64)
            current_features=np.asarray(current["X"])
            if current_features.ndim != 2 or current_features.shape[0] != len(current_patients):
                return False, f"invalid feature matrix for {mode}"
            if not np.all(np.isfinite(current_features)):
                return False, f"non-finite feature values for {mode}"
            if not np.array_equal(current_patients, wanted["patient_ids"].astype(str)):
                return False, f"patient cohort changed for {mode}"
            if not np.array_equal(current_labels, wanted["y"]):
                return False, f"labels changed for {mode}"
            if int(current["source_slices"]) != int(wanted["source_slices"]):
                return False, f"slice cohort changed for {mode}"
            if int(current["series_proxies"]) != int(wanted["series_proxies"]):
                return False, f"series cohort changed for {mode}"

        bank_path=Path(workspace.feature_bank)
        if not bank_path.is_file():
            return False, "feature bank file is missing"
        bank_mtime=int(bank_path.stat().st_mtime_ns)

        positive_masks=[
            Path(row["manual_mask_path"])
            for row in accepted_rows
            if row.get("segmentation_target_type") == HEART_PRESENT
        ]
        if DataStage.newest_mtime_ns(
            positive_masks + [Path(workspace.manual_annotations)]
        ) > bank_mtime:
            return False, "manual target changed after feature bank"

        predicted_directory=Path(workspace.predicted_masks)
        predicted_masks=list(predicted_directory.glob("*.png"))
        if len(predicted_masks) != len(rows):
            return False, f"automatic mask set incomplete ({len(predicted_masks)}/{len(rows)})"
        if DataStage.newest_mtime_ns(predicted_masks) > bank_mtime:
            return False, "automatic OOF mask changed after feature bank"

        # These tables directly determine cohort membership. JSON summaries and
        # pipeline-stage state are intentionally excluded because metadata-only
        # migration must not invalidate expensive embeddings.
        direct_tables=(
            Path(workspace.quality_audit),
            Path(workspace.prediction_audit),
            Path(workspace.matching_manifest),
        )
        missing=[str(path) for path in direct_tables if not path.is_file()]
        if missing:
            return False, "missing upstream table: " + ", ".join(missing)

        return True, "cohorts and direct image/mask inputs unchanged"

    @staticmethod
    def feature_metadata_payload(
        bank, fingerprint, workspace, elapsed="", upstream_state=None,
        cache_adoption="",
    ):
        modes={}
        for mode in FeatureStage.MODES:
            values=bank[mode]
            storage=PersistenceStage.LEGACY_MODE_ALIASES[mode]
            modes[mode]={
                "description": FeatureStage.MODE_DESCRIPTIONS[mode],
                "cohort": FeatureStage.MODE_COHORTS[mode],
                "storage_key": storage,
                "patients": int(len(values["patient_ids"])),
                "source_slices": int(values["source_slices"]),
                "series_proxies": int(values["series_proxies"]),
            }
        payload={
            "schema": "simple-patient-feature-bank-stable-v7",
            "fingerprint": fingerprint,
            "stable_fingerprint": fingerprint,
            "preprocessing_device": "cpu",
            "extractor_device": "cpu",
            "dtype": "float32",
            "numeric_protocol": {
                "device": "cpu", "dtype": "float32", "autocast": False,
                "batch_size": int(FeatureStage.BATCH_SIZE),
                "forward_batch_size": int(FeatureStage.FORWARD_BATCH_SIZE),
                "weights": "EfficientNet_B0_Weights.IMAGENET1K_V1",
            },
            "modes": modes,
            "elapsed": str(elapsed or ""),
            "feature_bank": str(workspace.feature_bank),
            "upstream_state": dict(upstream_state or {}),
        }
        if cache_adoption:
            payload["cache_adoption"]=str(cache_adoption)
        return payload

    @staticmethod
    def feature_bank_cache_status(rows, accepted_rows, matching, workspace):
        """Validate one feature-bank cache through a single shared code path."""
        metadata=DataStage.read_json(workspace.feature_metadata, {}) or {}
        stable=PersistenceStage.feature_fingerprint(rows, accepted_rows, workspace)
        legacy=(
            PersistenceStage.legacy_feature_fingerprint(rows, accepted_rows, workspace)
            if not metadata.get("stable_fingerprint")
            else ""
        )
        bank=None
        load_error=None
        if Path(workspace.feature_bank).is_file():
            try:
                bank=PersistenceStage.load_feature_bank(workspace)
            except Exception as error:
                load_error=error
        if bank is None:
            reason=(
                f"{type(load_error).__name__}: {load_error}"
                if load_error is not None
                else "feature bank file is missing"
            )
            return {
                "reusable": False, "reason": reason, "exact": False,
                "adopted": False, "stable_fingerprint": stable,
                "legacy_fingerprint": legacy, "metadata": metadata,
                "bank": None,
            }

        exact=(
            metadata.get("stable_fingerprint") == stable
            or metadata.get("fingerprint") in {stable, legacy}
        )
        if exact:
            return {
                "reusable": True, "reason": "fingerprint", "exact": True,
                "adopted": False, "stable_fingerprint": stable,
                "legacy_fingerprint": legacy, "metadata": metadata,
                "bank": bank,
            }

        legacy_adoptable=(
            metadata.get("schema") == "simple-patient-feature-bank-cross-class-v6-same-slice-baseline"
            and not metadata.get("stable_fingerprint")
        )
        if legacy_adoptable:
            structural, reason=PersistenceStage.feature_bank_structurally_compatible(
                bank, rows, accepted_rows, matching, workspace
            )
            return {
                "reusable": bool(structural), "reason": reason,
                "exact": False, "adopted": bool(structural),
                "stable_fingerprint": stable, "legacy_fingerprint": legacy,
                "metadata": metadata, "bank": bank,
            }

        return {
            "reusable": False, "reason": "stable fingerprint changed",
            "exact": False, "adopted": False,
            "stable_fingerprint": stable, "legacy_fingerprint": legacy,
            "metadata": metadata, "bank": bank,
        }

    @staticmethod
    def load_feature_bank(workspace):
        if not Path(workspace.feature_bank).is_file():
            raise FileNotFoundError(workspace.feature_bank)
        bank={}
        with np.load(workspace.feature_bank, allow_pickle=False) as archive:
            available=set(archive.files)
            for mode in FeatureStage.MODES:
                alias=PersistenceStage.LEGACY_MODE_ALIASES[mode]
                storage=None
                for candidate in dict.fromkeys((mode, alias)):
                    required={
                        f"{candidate}__X", f"{candidate}__y",
                        f"{candidate}__patient_ids", f"{candidate}__source_slices",
                        f"{candidate}__series_proxies",
                    }
                    if required.issubset(available):
                        storage=candidate
                        break
                if storage is None:
                    raise KeyError(f"Feature bank is missing mode {mode}.")
                bank[mode]={
                    "X": archive[f"{storage}__X"],
                    "y": archive[f"{storage}__y"],
                    "patient_ids": archive[f"{storage}__patient_ids"].astype(str),
                    "source_slices": int(archive[f"{storage}__source_slices"][0]),
                    "series_proxies": int(archive[f"{storage}__series_proxies"][0]),
                }
        FeatureStage.validate_cohort_alignment(bank)
        return bank

    @staticmethod
    def save_feature_bank(
        bank, fingerprint, workspace, elapsed, upstream_state=None,
    ):
        arrays={}
        for mode in FeatureStage.MODES:
            values=bank[mode]
            storage=PersistenceStage.LEGACY_MODE_ALIASES[mode]
            arrays[f"{storage}__X"]=np.asarray(values["X"], dtype=np.float32)
            arrays[f"{storage}__y"]=np.asarray(values["y"], dtype=np.int64)
            arrays[f"{storage}__patient_ids"]=np.asarray(values["patient_ids"]).astype("U")
            arrays[f"{storage}__source_slices"]=np.asarray([values["source_slices"]], dtype=np.int64)
            arrays[f"{storage}__series_proxies"]=np.asarray([values["series_proxies"]], dtype=np.int64)
        DataStage.write_npz(workspace.feature_bank, arrays)
        DataStage.write_json(
            workspace.feature_metadata,
            PersistenceStage.feature_metadata_payload(
                bank, fingerprint, workspace, elapsed=elapsed,
                upstream_state=upstream_state,
            ),
        )

    @staticmethod
    def build_feature_bank(rows, accepted_rows, matching, workspace, force=False):
        status=PersistenceStage.feature_bank_cache_status(
            rows, accepted_rows, matching, workspace
        )
        if not force and status["reusable"]:
            if not status["exact"]:
                # One-time metadata migration only. The .npz embeddings remain untouched.
                old_elapsed=str(status["metadata"].get("elapsed", ""))
                DataStage.write_json(
                    workspace.feature_metadata,
                    PersistenceStage.feature_metadata_payload(
                        status["bank"], status["stable_fingerprint"], workspace,
                        elapsed=old_elapsed,
                        upstream_state=PersistenceStage.feature_upstream_state(
                            rows, accepted_rows, workspace
                        ),
                        cache_adoption=status["reason"],
                    ),
                )
            print(
                "[FEATURE BANK] Compatible CPU/float32 bank reused"
                + (f" ({status['reason']})" if status["reason"] != "fingerprint" else "")
                + f": {workspace.feature_bank}"
            )
            return status["bank"]

        if status["bank"] is not None and not force:
            print(f"[FEATURE BANK] Cache invalid; rebuilding ({status['reason']}).")
        elif status["bank"] is None and Path(workspace.feature_bank).is_file():
            print(f"[FEATURE BANK] Cache invalid; rebuilding ({status['reason']}).")

        started=time.perf_counter()
        device=DataStage.start_device_stage("cpu", "EfficientNet feature extraction")
        try:
            bank=FeatureStage.build_feature_bank(rows, accepted_rows, matching, device)
        finally:
            DataStage.finish_device_stage(device, "EfficientNet feature extraction")
        PersistenceStage.save_feature_bank(
            bank,
            status["stable_fingerprint"],
            workspace,
            DataStage.format_seconds(time.perf_counter()-started),
            upstream_state=PersistenceStage.feature_upstream_state(
                rows, accepted_rows, workspace
            ),
        )
        print(f"[FEATURE BANK] Saved: {workspace.feature_bank}")
        return bank

    @staticmethod
    def evaluation_fingerprint(workspace):
        payload={
            "schema":"patient-evaluation-stable-v2","pca_variance":0.95,
            "outer_folds":5,"inner_folds":3,"c_grid":(0.01,0.1,1.0,10.0),
            "bootstrap_repeats":2000,"random_seed":42,"modes":FeatureStage.MODES,
        }
        digest=hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8"))
        if Path(workspace.feature_bank).is_file():
            digest.update(DataStage.sha256_file(workspace.feature_bank).encode("ascii"))
        return digest.hexdigest()

    @staticmethod
    def legacy_evaluation_fingerprint(workspace):
        digest=hashlib.sha256(b"patient-evaluation-v1")
        for path in (workspace.feature_bank, workspace.feature_metadata):
            if Path(path).is_file():
                digest.update(DataStage.sha256_file(path).encode("ascii"))
        return digest.hexdigest()

    @staticmethod
    def evaluation_cache_status(bank, workspace):
        metadata=DataStage.read_json(workspace.evaluation_metadata, {}) or {}
        stable=PersistenceStage.evaluation_fingerprint(workspace)
        legacy=(
            PersistenceStage.legacy_evaluation_fingerprint(workspace)
            if not metadata.get("stable_fingerprint")
            else ""
        )
        result_path=Path(workspace.results_csv)
        comparison_path=Path(workspace.comparisons_csv)
        prediction_paths={mode:Path(workspace.predictions_dir)/f"{mode}.csv" for mode in FeatureStage.MODES}
        files_complete=result_path.is_file() and comparison_path.is_file() and all(path.is_file() for path in prediction_paths.values())
        structural=False
        if files_complete:
            try:
                results=pd.read_csv(result_path)
                comparisons=pd.read_csv(comparison_path)
                predictions={mode:pd.read_csv(path) for mode,path in prediction_paths.items()}
                structural=set(results.get("mode", pd.Series(dtype=str)).astype(str)) == set(FeatureStage.MODES)
                for mode, table in predictions.items():
                    expected_patients=np.asarray(bank[mode]["patient_ids"]).astype(str)
                    expected_labels=np.asarray(bank[mode]["y"], dtype=np.int64)
                    table=table.sort_values("patient_id")
                    order=np.argsort(expected_patients)
                    if not np.array_equal(table["patient_id"].astype(str).to_numpy(), expected_patients[order]):
                        structural=False; break
                    if not np.array_equal(table["true_label"].to_numpy(dtype=np.int64), expected_labels[order]):
                        structural=False; break
            except Exception:
                structural=False
                results=comparisons=predictions=None
        else:
            results=comparisons=predictions=None
        fingerprint_match=(
            metadata.get("stable_fingerprint") == stable
            or metadata.get("fingerprint") in {stable, legacy}
        )
        freshness=False
        if structural:
            output_mtime=min(
                [result_path.stat().st_mtime_ns, comparison_path.stat().st_mtime_ns]
                + [path.stat().st_mtime_ns for path in prediction_paths.values()]
            )
            freshness=output_mtime >= Path(workspace.feature_bank).stat().st_mtime_ns
        legacy_adoptable=(
            metadata.get("schema") == "patient-evaluation-cache-v1"
            and not metadata.get("stable_fingerprint")
        )
        reusable=bool(structural and (fingerprint_match or (legacy_adoptable and freshness)))
        return {
            "reusable":reusable,"reason":"fingerprint" if fingerprint_match else "legacy-structural+freshness" if reusable else "invalid",
            "stable_fingerprint":stable,"legacy_fingerprint":legacy,"metadata":metadata,
            "results":results,"comparisons":comparisons,"predictions":predictions,
        }

    @staticmethod
    def evaluate(bank, workspace, force=False):
        status=PersistenceStage.evaluation_cache_status(bank, workspace)
        if not force and status["reusable"]:
            if status["metadata"].get("stable_fingerprint") != status["stable_fingerprint"]:
                DataStage.write_json(
                    workspace.evaluation_metadata,
                    {**status["metadata"], "stable_fingerprint":status["stable_fingerprint"],
                     "cache_adoption":status["reason"]},
                )
            print(
                f"[EVALUATION] Compatible persisted results reused ({status['reason']}): "
                f"{workspace.results_csv}"
            )
            return status["results"], status["comparisons"], status["predictions"]
        results, comparisons, predictions=EvaluationStage.evaluate_experiments(bank)
        results.to_csv(workspace.results_csv, index=False)
        comparisons.to_csv(workspace.comparisons_csv, index=False)
        for mode, table in predictions.items():
            table.to_csv(Path(workspace.predictions_dir)/f"{mode}.csv", index=False)
        DataStage.write_json(
            workspace.evaluation_metadata,
            {
                "schema":"patient-evaluation-stable-v2",
                "fingerprint":status["stable_fingerprint"],
                "stable_fingerprint":status["stable_fingerprint"],
                "modes":list(FeatureStage.MODES),
                "results":str(workspace.results_csv),
                "comparisons":str(workspace.comparisons_csv),
            },
        )
        print(f"[EVALUATION] Results persisted: {workspace.results_csv}")
        return results, comparisons, predictions




# =============================================================================
# STAGE 7 — SIMPLE USER API
# =============================================================================

class Pipeline:
    """Public API with independent, resumable CPU and GPU stages."""

    last_run=None

    @staticmethod
    def _prepare_context(
        dataset_path=None,
        workspace_root=None,
        refresh_quality=False,
        minimum_masks=None,
        require_manual=True,
        load_predictions=False,
        require_complete_predictions=False,
    ):
        DataStage.seed_everything(include_cuda=False)
        workspace=DataStage.create_workspace(workspace_root)
        dataset_path=Path(dataset_path or DataStage._default_dataset_path())
        base_rows=DataStage.discover_dataset(dataset_path, workspace)
        PersistenceStage.save_dataset_manifest(base_rows, workspace)
        rows=DataStage.build_quality_audit(
            base_rows,
            workspace=workspace,
            refresh=refresh_quality,
        )
        accepted=[]
        manual_audit=[]
        manual_summary={}
        if require_manual:
            accepted, manual_audit, manual_summary=DataStage.audit_manual_masks(
                rows,
                workspace,
                minimum_masks=minimum_masks,
            )
            PersistenceStage.save_manual_audit(
                manual_audit, manual_summary, workspace
            )
        persisted={}
        if load_predictions:
            existing_audit=DataStage.read_csv(workspace.prediction_audit)
            existing_tokens={
                str(row.get("image_token", ""))
                for row in existing_audit if str(row.get("image_token", ""))
            }
            expected_tokens={str(row["image_token"]) for row in rows}
            if existing_tokens != expected_tokens:
                AttentionStage.combine_complete_prediction_parts(rows, workspace)
            rows, persisted=AttentionStage.load_saved_predictions(rows, workspace)
            if require_complete_predictions and (
                persisted.get("mask_files", 0) != len(rows)
                or persisted.get("metadata_rows", 0) != len(rows)
            ):
                raise RuntimeError(
                    "The final CPU stage requires a complete persisted OOF prediction set. "
                    f"Found masks={persisted.get('mask_files', 0)}/{len(rows)}, "
                    f"metadata={persisted.get('metadata_rows', 0)}/{len(rows)}. "
                    "Run Pipeline.run_kaggle_gpu_stage() again; completed folds will be reused."
                )
        return {
            "dataset_path": dataset_path,
            "workspace": workspace,
            "rows": rows,
            "accepted": accepted,
            "manual_audit": manual_audit,
            "manual_summary": manual_summary,
            "persisted_predictions": persisted,
        }

    @staticmethod
    def cpu_prepare(
        dataset_path=None,
        workspace_root=None,
        refresh_quality=False,
        minimum_masks=None,
    ):
        print("\n" + "=" * 92)
        print("CARDIAC MRI CAD — KAGGLE CPU INITIAL STAGE")
        print("=" * 92)
        print("[ORDER] dataset manifest -> quality audit -> manual-target audit")
        context=Pipeline._prepare_context(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            refresh_quality=refresh_quality,
            minimum_masks=minimum_masks,
            require_manual=True,
            load_predictions=False,
        )
        workspace=context["workspace"]
        PersistenceStage.mark(
            workspace,
            "cpu_prepare",
            images=len(context["rows"]),
            accepted_targets=len(context["accepted"]),
        )
        context["status"]="cpu_prepare_complete"
        Pipeline.last_run=context
        print(f"[WORKSPACE] {workspace.root}")
        print("[NEXT] Enable a Kaggle GPU and run Pipeline.run_kaggle_gpu_stage().")
        return context

    @staticmethod
    def gpu_attention(
        dataset_path=None,
        workspace_root=None,
        force_training=False,
        force_prediction=False,
        refresh_quality=False,
        minimum_masks=None,
        device="cuda",
    ):
        print("\n" + "=" * 92)
        print("CARDIAC MRI CAD — KAGGLE GPU ATTENTION STAGE")
        print("=" * 92)
        print("[RESUME] Compatible checkpoints and completed prediction folds are reused.")
        context=Pipeline._prepare_context(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            refresh_quality=refresh_quality,
            minimum_masks=minimum_masks,
            require_manual=True,
            load_predictions=False,
        )
        workspace=context["workspace"]
        attention_device=DataStage.start_device_stage(device, "Attention U-Net training + OOF prediction")
        try:
            fold_states=AttentionStage.train_attention_crossfit(
                context["accepted"],
                attention_device,
                workspace=workspace,
                force=force_training,
            )
            rows, prediction_summary=AttentionStage.predict_attention_masks(
                context["rows"],
                workspace,
                attention_device,
                fold_states=fold_states,
                force=force_prediction,
            )
        finally:
            DataStage.clear_image_cache()
            DataStage.finish_device_stage(
                attention_device,
                "Attention U-Net training + OOF prediction",
            )
        segmentation_summary, segmentation_metrics=AttentionStage.evaluate_oof_segmentation(
            context["manual_audit"], rows
        )
        PersistenceStage.save_segmentation_metrics(
            segmentation_summary, segmentation_metrics, workspace
        )
        context.update(
            rows=rows,
            predictions=rows,
            prediction_summary=prediction_summary,
            segmentation_summary=segmentation_summary,
            segmentation_metrics=segmentation_metrics,
            status="gpu_attention_complete",
        )
        PersistenceStage.mark(
            workspace,
            "gpu_attention",
            images=len(rows),
            reused_folds=prediction_summary.get("reused_folds", []),
            computed_folds=prediction_summary.get("computed_folds", []),
        )
        Pipeline.last_run=context
        print("[NEXT] Disable the Kaggle accelerator and run Pipeline.run_kaggle_cpu_final_stage().")
        return context

    @staticmethod
    def cpu_finalize(
        dataset_path=None,
        workspace_root=None,
        action="evaluate",
        force_matching=False,
        force_feature_bank=False,
        force_evaluation=False,
        refresh_quality=False,
        minimum_masks=None,
        review_scope="invalid",
        review_limit=300,
        review_start_index=0,
        review_round=1,
        review_seed=42,
    ):
        action=str(action).strip().lower()
        if action not in {"matching", "features", "evaluate", "review", "status"}:
            raise ValueError("action must be matching/features/evaluate/review/status.")
        if action == "review":
            editor=Pipeline.review(
                scope=review_scope,
                limit=review_limit,
                start_index=review_start_index,
                seed=review_seed,
                review_round=review_round,
                dataset_path=dataset_path,
                workspace_root=workspace_root,
            )
            return {"editor": editor, "status": Pipeline.status(workspace_root)}
        if action == "status":
            return {"status": Pipeline.status(workspace_root)}

        print("\n" + "=" * 92)
        print(f"CARDIAC MRI CAD — KAGGLE CPU FINAL STAGE: {action.upper()}")
        print("=" * 92)
        context=Pipeline._prepare_context(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            refresh_quality=refresh_quality,
            minimum_masks=minimum_masks,
            require_manual=True,
            load_predictions=True,
            require_complete_predictions=True,
        )
        workspace=context["workspace"]
        rows=context["rows"]
        matching, matching_summary=PersistenceStage.build_matching(
            rows, workspace, force=force_matching
        )
        context.update(matching=matching, matching_summary=matching_summary)
        PersistenceStage.mark(
            workspace, "matching", selected_images=matching_summary.get("selected_images", "")
        )
        if action == "matching":
            context["status"]="matching_complete"
            Pipeline.last_run=context
            return context

        bank=PersistenceStage.build_feature_bank(
            rows,
            context["accepted"],
            matching,
            workspace,
            force=force_feature_bank,
        )
        context["feature_bank"]=bank
        PersistenceStage.mark(workspace, "features", modes=len(bank))
        if action == "features":
            context["status"]="features_complete"
            Pipeline.last_run=context
            return context

        results, comparisons, predictions=PersistenceStage.evaluate(
            bank, workspace, force=force_evaluation
        )
        context.update(
            results=results,
            comparisons=comparisons,
            patient_predictions=predictions,
            status="complete",
        )
        PersistenceStage.mark(
            workspace,
            "evaluation",
            modes=len(results),
            results=str(workspace.results_csv),
        )
        Pipeline.last_run=context
        print("\n[EVALUATION SUMMARY]")
        display(results)
        print("\n[PAIRED AUC COMPARISONS]")
        display(comparisons)
        return context

    @staticmethod
    def run(
        dataset_path=None,
        workspace_root=None,
        manual_root=None,
        minimum_masks=None,
        force=False,
    ):
        """Convenience path for one active GPU session; artifacts remain resumable.

        ``manual_root`` is retained as a backward-compatible alias for the complete
        persistent workspace path used by earlier simplified notebooks.
        """
        workspace_root=workspace_root or manual_root
        Pipeline.cpu_prepare(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            minimum_masks=minimum_masks,
        )
        Pipeline.gpu_attention(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            minimum_masks=minimum_masks,
            force_training=force,
            force_prediction=force,
            device="cuda",
        )
        return Pipeline.cpu_finalize(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            minimum_masks=minimum_masks,
            action="evaluate",
            force_matching=force,
            force_feature_bank=force,
            force_evaluation=force,
        )

    @staticmethod
    def review(
        scope="invalid",
        limit=300,
        start_index=0,
        seed=42,
        review_round=1,
        dataset_path=None,
        workspace_root=None,
        manual_root=None,
    ):
        """Open the HTML editor from persisted masks and prediction metadata."""
        workspace_root=workspace_root or manual_root
        context=Pipeline._prepare_context(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            refresh_quality=False,
            require_manual=False,
            load_predictions=True,
            require_complete_predictions=False,
        )
        rows=context["rows"]
        workspace=context["workspace"]
        persisted=context["persisted_predictions"]
        DataStage.register_existing_manual_masks(rows, workspace)
        if scope in {"invalid", "uncertain"} and persisted.get("metadata_rows", 0) == 0:
            print("[REVIEW] Saved prediction metadata are required. Run the GPU stage first.")
            return None
        try:
            queue=ReviewStage.select_review_rows(
                rows,
                workspace,
                scope=scope,
                limit=limit,
                seed=seed,
                review_round=review_round,
            )
        except RuntimeError as error:
            print(error)
            return None
        editor=MaskEditor(
            queue,
            workspace,
            start_index=start_index,
            brush_radius=8,
            review_round=review_round,
            review_scope=scope,
        )
        context.update(editor=editor, status="manual_review")
        Pipeline.last_run=context
        return editor.show()

    @staticmethod
    def persistence_status(dataset_path=None, workspace_root=None, minimum_masks=None):
        """Validate every cache without starting matching, EfficientNet, or evaluation."""
        context=Pipeline._prepare_context(
            dataset_path=dataset_path, workspace_root=workspace_root,
            refresh_quality=False, minimum_masks=minimum_masks,
            require_manual=True, load_predictions=True,
            require_complete_predictions=True,
        )
        workspace=context["workspace"]
        rows=context["rows"]
        matching_status=PersistenceStage.matching_cache_status(rows, workspace)
        matching=matching_status["cached"] if matching_status["reusable"] else []

        if matching:
            feature_status=PersistenceStage.feature_bank_cache_status(
                rows, context["accepted"], matching, workspace
            )
        else:
            feature_status={
                "reusable":False, "reason":"matching cache is not reusable",
                "bank":None,
            }

        evaluation_status=(
            PersistenceStage.evaluation_cache_status(feature_status["bank"], workspace)
            if feature_status.get("reusable") and feature_status.get("bank") is not None
            else {"reusable":False,"reason":"feature bank is not reusable"}
        )
        report={
            "quality_audit_compatible": True,
            "manual_audit_compatible": True,
            "predictions_complete": context["persisted_predictions"].get("mask_files",0)==len(rows),
            "matching_reusable": matching_status["reusable"],
            "matching_reason": matching_status["reason"],
            "feature_bank_reusable": bool(feature_status.get("reusable",False)),
            "feature_bank_reason": str(feature_status.get("reason","")),
            "evaluation_reusable": evaluation_status.get("reusable",False),
            "evaluation_reason": evaluation_status.get("reason",""),
        }
        print(json.dumps(report, indent=2, ensure_ascii=False))
        return report

    @staticmethod
    def status(workspace_root=None):
        workspace=DataStage.create_workspace(workspace_root)
        checkpoints=[
            AttentionStage.checkpoint_path(workspace, fold)
            for fold in range(5)
        ]
        prediction_parts=[
            AttentionStage.prediction_part_paths(workspace, fold)[1]
            for fold in range(5)
        ]
        status={
            "workspace": str(workspace.root),
            "dataset_manifest": workspace.dataset_manifest.is_file(),
            "quality_audit": workspace.quality_audit.is_file(),
            "manual_audit": workspace.manual_audit.is_file(),
            "checkpoints_complete": all(path.is_file() for path in checkpoints),
            "checkpoint_folds": [fold for fold, path in enumerate(checkpoints) if path.is_file()],
            "prediction_parts_complete": all(path.is_file() for path in prediction_parts),
            "prediction_part_folds": [fold for fold, path in enumerate(prediction_parts) if path.is_file()],
            "prediction_audit": workspace.prediction_audit.is_file(),
            "predicted_mask_files": sum(1 for _ in workspace.predicted_masks.glob("*.png")),
            "matching_manifest": workspace.matching_manifest.is_file(),
            "feature_bank": workspace.feature_bank.is_file(),
            "evaluation_results": workspace.results_csv.is_file(),
            "last_stage": DataStage.read_json(workspace.stage_state, {}) or {},
        }
        print(json.dumps(status, indent=2, ensure_ascii=False, default=str))
        return status

    @staticmethod
    def clean_kaggle_working(
        workspace_root=None,
        dry_run=True,
        remove_other_working_items=True,
        keep_backup_zips=False,
    ):
        """Clean Kaggle working storage while preserving this pipeline's contract."""
        return DataStage.clean_kaggle_working(
            workspace_root=workspace_root,
            dry_run=dry_run,
            remove_other_working_items=remove_other_working_items,
            keep_backup_zips=keep_backup_zips,
        )

    @staticmethod
    def backup(name="cad_attention_workspace_backup", workspace_root=None):
        import shutil
        workspace=DataStage.create_workspace(workspace_root)
        archive_base=Path("/kaggle/working") / str(name) if Path("/kaggle/working").exists() else workspace.root.parent / str(name)
        archive=Path(shutil.make_archive(str(archive_base), "zip", workspace.root))
        print(f"[BACKUP] {archive}")
        return archive

    @staticmethod
    def run_kaggle_cpu_stage(
        dataset_path=None,
        workspace_root=None,
        refresh_quality=False,
        minimum_masks=None,
    ):
        return Pipeline.cpu_prepare(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            refresh_quality=refresh_quality,
            minimum_masks=minimum_masks,
        )

    @staticmethod
    def run_kaggle_gpu_stage(
        dataset_path=None,
        workspace_root=None,
        after_review=False,
        force_training=False,
        force_prediction=False,
        refresh_quality=False,
        minimum_masks=None,
    ):
        if after_review:
            print("[GPU] Post-review mode: fingerprints decide which folds must be retrained.")
        return Pipeline.gpu_attention(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            force_training=force_training,
            force_prediction=force_prediction,
            refresh_quality=refresh_quality,
            minimum_masks=minimum_masks,
            device="cuda",
        )

    @staticmethod
    def run_kaggle_cpu_final_stage(
        dataset_path=None,
        workspace_root=None,
        action="evaluate",
        force_matching=False,
        force_feature_bank=False,
        force_evaluation=False,
        refresh_quality=False,
        minimum_masks=None,
        review_scope="invalid",
        review_limit=300,
        review_start_index=0,
        review_round=1,
        review_seed=42,
    ):
        return Pipeline.cpu_finalize(
            dataset_path=dataset_path,
            workspace_root=workspace_root,
            action=action,
            force_matching=force_matching,
            force_feature_bank=force_feature_bank,
            force_evaluation=force_evaluation,
            refresh_quality=refresh_quality,
            minimum_masks=minimum_masks,
            review_scope=review_scope,
            review_limit=review_limit,
            review_start_index=review_start_index,
            review_round=review_round,
            review_seed=review_seed,
        )








# Backward-compatible call aliases; implementations live in Pipeline.
run_kaggle_cpu_stage = Pipeline.run_kaggle_cpu_stage
run_kaggle_gpu_stage = Pipeline.run_kaggle_gpu_stage
run_kaggle_cpu_final_stage = Pipeline.run_kaggle_cpu_final_stage

# Importing definitions does not train models. It only seeds CPU-side randomness.
DataStage.seed_everything(include_cuda=False)
print("[PIPELINE] Cleanup preview: Pipeline.clean_kaggle_working(dry_run=True)")
print("[PIPELINE] CPU initial: cpu_state = Pipeline.run_kaggle_cpu_stage()")
print("[PIPELINE] GPU Attention: gpu_state = Pipeline.run_kaggle_gpu_stage()")
print("[PIPELINE] CPU final: final_state = Pipeline.run_kaggle_cpu_final_stage(action='evaluate')")
print("[PIPELINE] Review: editor = Pipeline.review(scope='invalid')")
print("[PIPELINE] Cache validation: Pipeline.persistence_status()")
print("[PIPELINE] Status: Pipeline.status()")
print("[PIPELINE] Workspace: /kaggle/working/cad_attention_unet_workspace")


[PIPELINE] Cleanup preview: Pipeline.clean_kaggle_working(dry_run=True)
[PIPELINE] CPU initial: cpu_state = Pipeline.run_kaggle_cpu_stage()
[PIPELINE] GPU Attention: gpu_state = Pipeline.run_kaggle_gpu_stage()
[PIPELINE] CPU final: final_state = Pipeline.run_kaggle_cpu_final_stage(action='evaluate')
[PIPELINE] Review: editor = Pipeline.review(scope='invalid')
[PIPELINE] Cache validation: Pipeline.persistence_status()
[PIPELINE] Status: Pipeline.status()
[PIPELINE] Workspace: /kaggle/working/cad_attention_unet_workspace


## 2. One-time cleanup of `/kaggle/working`

The cleaner preserves the active `cad_attention_unet_workspace` contract (manual masks/labels, automatic masks, five checkpoints, fold prediction parts, matching, feature bank, evaluation tables, review state) and removes legacy/foreign artifacts. Hidden Kaggle system entries are never touched.

Run the preview first. Only after checking the printed list, set `APPLY_CLEANUP = True`. Cleanup never runs automatically.


In [ ]:
WORKSPACE_ROOT = None  # -> /kaggle/working/cad_attention_unet_workspace
APPLY_CLEANUP = False    # First run with False; change to True after reviewing the list.

cleanup_state = Pipeline.clean_kaggle_working(
    workspace_root=WORKSPACE_ROOT,
    dry_run=not APPLY_CLEANUP,
    remove_other_working_items=True,
    keep_backup_zips=False,
)


## 3. Initial CPU stage — manifest and audits

Set **Kaggle → Settings → Accelerator → None**. This stage is inexpensive after its first successful execution because the complete quality audit is read from the persistent workspace. It also validates the current manual masks and labels before GPU training.

In [ ]:
DATASET_PATH = None      # Automatic Kaggle dataset discovery, or provide a dataset directory.
WORKSPACE_ROOT = None     # Defaults to /kaggle/working/cad_attention_unet_workspace.
MINIMUM_MASKS = None      # None keeps the researched minimum of 800 positive manual masks.

cpu_state = Pipeline.run_kaggle_cpu_stage(
    dataset_path=DATASET_PATH,
    workspace_root=WORKSPACE_ROOT,
    refresh_quality=False,   # True only when image-quality measurements must be rebuilt.
    minimum_masks=MINIMUM_MASKS,
)


## 4. GPU stage — Attention U-Net training and OOF prediction

Enable a **Kaggle GPU**, allow the kernel to restart, rerun the definitions cell, and execute this block. The stage:

- reuses each compatible `attention_unet_fold_*.pt` checkpoint;
- retrains only folds invalidated by changed manual targets or settings;
- reuses each complete prediction part in `attention_prediction_parts/`;
- recomputes only the first missing or incompatible prediction fold.

A disconnect during a fold may require that one fold to restart, but earlier completed folds remain reusable.

In [ ]:
DATASET_PATH = None
WORKSPACE_ROOT = None
MINIMUM_MASKS = None

# Keep both force flags False for automatic fingerprint-based resume.
gpu_state = Pipeline.run_kaggle_gpu_stage(
    dataset_path=DATASET_PATH,
    workspace_root=WORKSPACE_ROOT,
    after_review=False,       # Set True after editing manual targets; fingerprints still decide what changes.
    force_training=False,     # True retrains all five Attention folds.
    force_prediction=False,   # True regenerates all five OOF prediction folds.
    refresh_quality=False,
    minimum_masks=MINIMUM_MASKS,
)


## 5. Final CPU stage — matching, feature bank, and evaluation

Disable the accelerator, rerun the definitions cell, and execute the block below. With all `force_*` flags set to `False`, the stage reuses the quality audit, complete OOF masks/metadata, matching manifest, 11-mode EfficientNet bank, and evaluation tables whenever their semantic inputs are unchanged.

Run `Pipeline.persistence_status()` first when you want a no-computation compatibility report.


In [27]:
DATASET_PATH = None
WORKSPACE_ROOT = None
MINIMUM_MASKS = None

# Optional: validates caches without starting matching, EfficientNet, or evaluation.
cache_report = Pipeline.persistence_status(
    dataset_path=DATASET_PATH,
    workspace_root=WORKSPACE_ROOT,
    minimum_masks=MINIMUM_MASKS,
)

final_state = Pipeline.run_kaggle_cpu_final_stage(
    dataset_path=DATASET_PATH,
    workspace_root=WORKSPACE_ROOT,
    action="evaluate",    # matching / features / evaluate / review / status
    force_matching=False,
    force_feature_bank=False,
    force_evaluation=False,
    refresh_quality=False,
    minimum_masks=MINIMUM_MASKS,
)

results = final_state.get("results")
comparisons = final_state.get("comparisons")



CARDIAC MRI CAD — KAGGLE CPU FINAL STAGE: EVALUATE
[DATA] Patients: 30
[DATA] Normal: 16
[DATA] Sick: 14
[DATA] Images: 63425
[DATA] Series proxies: 2859
[DATA] Scan time: 2m 06s
[QUALITY] Compatible persisted audit reused: 63425 images.
[MANUAL MASKS] Compatible persisted audit reused: accepted=3232, positive=2747.
[ATTENTION][PERSISTED] mask files=63425/63425, metadata=63425/63425, audit=/kaggle/working/cad_attention_unet_workspace/simple_attention_prediction_audit.csv
[MATCHING] Compatible persisted manifest reused (fingerprint): /kaggle/working/cad_attention_unet_workspace/simple_pipeline_outputs/cross_class_matching_manifest.csv
[FEATURE BANK][COHORT] attention_eligible_same_slices: modes=['B2_ATTENTION_ELIGIBLE_FULL_IMAGE', 'AU1_ATTENTION_ROI', 'C1_ATTENTION_COMPLEMENT'], patients=30, slices=53306, series=2600
[FEATURE BANK][COHORT] cross_class_matched_same_slices: modes=['B1_MATCHED_FULL_IMAGE', 'AU2_MATCHED_ATTENTION_ROI', 'C2_MATCHED_ATTENTION_COMPLEMENT'], patients=30, slice

,mode,experiment_group,description,cohort,patients,source_slices,series_proxies,auc,auc_ci_low,auc_ci_high,average_precision,brier,accuracy,sensitivity,specificity
0,AU1_ATTENTION_ROI,primary,Automatic Attention U-Net heart ROI on the sam...,attention_eligible_same_slices,30,53306,2600,0.982143,0.933035,1.000000,0.981548,0.059810,0.933333,0.928571,0.9375
1,M1_MANUAL_ROI,mask_validation,Manual heart ROI on the annotated subset,manual_positive_same_slices,30,2747,1153,0.977679,0.916597,1.000000,0.981203,0.052900,0.900000,0.785714,1.0000
2,AU2_MATCHED_ATTENTION_ROI,primary,Automatic heart ROI on the expanded balanced m...,cross_class_matched_same_slices,30,18634,2218,0.950893,0.856428,1.000000,0.940982,0.090525,0.900000,0.857143,0.9375
3,AU3_ATTENTION_ROI_MANUAL_SUBSET,mask_validation,Automatic ROI on the same manually annotated i...,manual_positive_same_slices,30,2747,1153,0.946429,0.821429,1.000000,0.967033,0.054334,0.966667,0.928571,1.0000
4,B2_ATTENTION_ELIGIBLE_FULL_IMAGE,primary,Full image on exactly the AU1/C1 attention-eli...,attention_eligible_same_slices,30,53306,2600,0.839286,0.662037,0.972851,0.866942,0.180710,0.800000,0.714286,0.8750
5,C3_MANUAL_COMPLEMENT,mask_validation,Complement of the manual ROI,manual_positive_same_slices,30,2747,1153,0.821429,0.640206,0.968753,0.838507,0.159930,0.866667,0.785714,0.9375
6,B0_FULL_IMAGE,primary,Full-image contextual baseline on every datase...,all_dataset_slices,30,63425,2859,0.803571,0.625000,0.955590,0.817033,0.165337,0.833333,0.714286,0.9375
7,C1_ATTENTION_COMPLEMENT,primary,Pixels outside the automatic heart ROI on the ...,attention_eligible_same_slices,30,53306,2600,0.803571,0.609972,0.966534,0.850152,0.169491,0.766667,0.571429,0.9375
8,B1_MATCHED_FULL_IMAGE,primary,Full-image baseline on the expanded balanced S...,cross_class_matched_same_slices,30,18634,2218,0.794643,0.606335,0.955357,0.774798,0.169069,0.833333,0.714286,0.9375
9,C4_ATTENTION_COMPLEMENT_MANUAL_SUBSET,mask_validation,Automatic complement on the same manually anno...,manual_positive_same_slices,30,2747,1153,0.790179,0.597198,0.955362,0.816329,0.183750,0.800000,0.642857,0.9375



[PAIRED AUC COMPARISONS]


,comparison,first_mode,second_mode,n_patients,auc_difference_first_minus_second,ci_low,ci_high
0,same_attention_eligible_slices_roi_vs_full_image,AU1_ATTENTION_ROI,B2_ATTENTION_ELIGIBLE_FULL_IMAGE,30,0.142857,0.031219,0.289603
1,same_attention_eligible_slices_full_image_vs_c...,B2_ATTENTION_ELIGIBLE_FULL_IMAGE,C1_ATTENTION_COMPLEMENT,30,0.035714,-0.111645,0.213357
2,same_attention_eligible_slices_roi_vs_complement,AU1_ATTENTION_ROI,C1_ATTENTION_COMPLEMENT,30,0.178571,0.032389,0.371041
3,attention_eligible_selection_effect_on_full_image,B2_ATTENTION_ELIGIBLE_FULL_IMAGE,B0_FULL_IMAGE,30,0.035714,-0.062500,0.149338
4,cross_class_matched_roi_vs_full_image,AU2_MATCHED_ATTENTION_ROI,B1_MATCHED_FULL_IMAGE,30,0.156250,0.004525,0.325794
5,cross_class_matched_roi_vs_complement,AU2_MATCHED_ATTENTION_ROI,C2_MATCHED_ATTENTION_COMPLEMENT,30,0.178571,0.026667,0.351852
6,cross_class_matched_full_image_vs_complement,B1_MATCHED_FULL_IMAGE,C2_MATCHED_ATTENTION_COMPLEMENT,30,0.022321,-0.122229,0.160714
7,all_slices_vs_cross_class_matched_full_image,B0_FULL_IMAGE,B1_MATCHED_FULL_IMAGE,30,0.008929,-0.095039,0.100513
8,attention_eligible_vs_cross_class_matched_roi,AU1_ATTENTION_ROI,AU2_MATCHED_ATTENTION_ROI,30,0.031250,0.000000,0.097222
9,attention_eligible_vs_cross_class_matched_comp...,C1_ATTENTION_COMPLEMENT,C2_MATCHED_ATTENTION_COMPLEMENT,30,0.031250,-0.095039,0.167422


## 6. Review, status, and backup

The HTML editor reopens the saved automatic masks after a restart. A later GPU run detects which manual targets changed and retrains only the affected model folds. `Pipeline.status()` shows which persistent stages are complete. A ZIP backup is useful before ending a Kaggle session or publishing a notebook version.

In [ ]:
# Inspect the persistent workspace without running the pipeline.
# status = Pipeline.status(workspace_root=WORKSPACE_ROOT)

# Open one persistent review queue.
# editor = Pipeline.review(
#     scope="invalid",       # invalid / uncertain / empty / novel / manual / all
#     limit=300,
#     start_index=0,
#     review_round=1,
#     dataset_path=DATASET_PATH,
#     workspace_root=WORKSPACE_ROOT,
# )

# Create /kaggle/working/cad_attention_workspace_backup.zip.
# backup_path = Pipeline.backup(
#     name="cad_attention_workspace_backup",
#     workspace_root=WORKSPACE_ROOT,
# )


## Recovery rules

1. **Kernel/browser disconnect:** reconnect, rerun the definitions cell, then rerun the `Pipeline.run_kaggle_*` method for the stage that was active. The pipeline validates fingerprints and file contents rather than trusting filenames alone.
2. **Interrupted Attention training:** completed fold checkpoints are reused; the unfinished or incompatible fold restarts.
3. **Interrupted Attention prediction:** completed fold CSV/JSON parts and mask PNGs are reused; the unfinished or incompatible fold restarts.
4. **Interrupted CPU final stage:** rerun `Pipeline.run_kaggle_cpu_final_stage(...)`. Matching, the feature bank, and evaluation each have their own fingerprinted cache.
5. **Manual-mask edits:** rerun `Pipeline.run_kaggle_gpu_stage(after_review=True)`. Checkpoint fingerprints include target masks, target types, weights, and source-image metadata.
6. **Intentional complete rebuild:** use the corresponding `force_*` flag. Avoid deleting the whole workspace unless all manual masks and labels have been backed up.

`/kaggle/working` is the active Kaggle workspace. Before deliberately ending or replacing the Kaggle session, save a notebook version/output or create the ZIP backup so the workspace can be restored in a future fresh session.
